# Dataset Validation Pipeline — MSA / Darija Parallel Benchmark

Automated checks for the C1 deliverable (Dialect-Aware Arabic RAG).

Run cells top to bottom.  Week 2.

### Install dependencies

In [1]:
# !pip install -q pandera sentence-transformers pandas

### Config + sample data

In [ ]:
CONFIG = {
    "alignment_similarity_threshold": 0.55,   # below this -> pairs may not mean the same thing
    "near_duplicate_threshold": 0.92,         # above this -> likely duplicate/near-duplicate
    "trivial_pair_threshold": 0.985,          # above this -> "darija" version is basically identical to MSA
    "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "darija_markers": [
        # original core list
        "ديال", "بزاف", "واخا", "دابا", "غادي", "كنبغي", "ماشي",
        "شنو", "علاش", "فين", "كيفاش", "بغيت", "نتا", "نتي", "حنا",
        "زعما", "خاصني", "كاين", "ماكاينش", "منين", "بأش", "واش",
        "بشحال", "شحال",
        # expanded: question words & particles found missing in real data (b002, b005, b006, b015, b019, b024, b028)
        "إمتى", "شكون", "فأش", "فوقاش", "علاه", "أشمن", "أش",
        # expanded: common Darija function words / demonstratives / discourse markers
        "ملي", "حيت", "راه", "بحال", "هادشي", "ديك", "هاد", "هادا", "هادي",
        "نتوما", "حتى", "غير", "باقي", "توما", "دغيا",
        # expanded: verb prefixes distinctive to Darija present tense (كيـ / غاديـ conjugation)
        "كيدير", "كتدير", "كيوفر", "كتترکز", "كيحدد", "كيساهم", "كيجيو", "كيزورو", "كيستافد",
    ],
    "known_chunk_ids": ["chunk_001", "chunk_002", "chunk_003", "chunk_004", "chunk_005", "chunk_006", "chunk_007", "chunk_008", "chunk_009", "chunk_010", "chunk_011", "chunk_012", "chunk_013", "chunk_014", "chunk_015", "chunk_016", "chunk_017", "chunk_018", "chunk_019", "chunk_020", "chunk_021", "chunk_022", "chunk_023", "chunk_024", "chunk_025", "chunk_026", "chunk_027", "chunk_028", "chunk_029", "chunk_030", "chunk_031", "chunk_032", "chunk_033", "chunk_034", "chunk_035", "chunk_036", "chunk_037", "chunk_038", "chunk_039", "chunk_040", "chunk_041", "chunk_042", "chunk_043", "chunk_044", "chunk_045", "chunk_046", "chunk_047", "chunk_048", "chunk_049", "chunk_050", "chunk_051", "chunk_052", "chunk_053", "chunk_054", "chunk_055", "chunk_056", "chunk_057", "chunk_058", "chunk_059", "chunk_060", "chunk_061", "chunk_062", "chunk_063", "chunk_064", "chunk_065", "chunk_066", "chunk_067", "chunk_068", "chunk_069", "chunk_070", "chunk_071", "chunk_072", "chunk_073", "chunk_074", "chunk_075", "chunk_076", "chunk_077", "chunk_078", "chunk_079", "chunk_080"],
}

# Real Week-2 batch: 34 items, 15 MSA corpus chunks (history/geo + social/economic)
SAMPLE_DATA = [
    {
        "id": "b001",
        "msa_query": "أين تقع الرباط؟",
        "darija_query": "فين كاينة الرباط؟",
        "gold_answer": "الرباط تقع على ساحل المحيط الأطلسي عند مصب نهر أبي رقراق.",
        "source_chunk_id": "chunk_001",
        "source_chunk_text": "الرباط هي العاصمة الإدارية للمملكة المغربية، وتقع على ساحل المحيط الأطلسي عند مصب نهر أبي رقراق. تأسست المدينة الحديثة في القرن الثاني عشر الميلادي، وتضم اليوم مقرات الحكومة والبرلمان والسفارات الأجنبية."
    },
    {
        "id": "b002",
        "msa_query": "متى تأسست مدينة الرباط الحديثة؟",
        "darija_query": "إمتى تأسست الرباط الجديدة؟",
        "gold_answer": "تأسست المدينة الحديثة في القرن الثاني عشر الميلادي.",
        "source_chunk_id": "chunk_001",
        "source_chunk_text": "الرباط هي العاصمة الإدارية للمملكة المغربية، وتقع على ساحل المحيط الأطلسي عند مصب نهر أبي رقراق. تأسست المدينة الحديثة في القرن الثاني عشر الميلادي، وتضم اليوم مقرات الحكومة والبرلمان والسفارات الأجنبية."
    },
    {
        "id": "b003",
        "msa_query": "كم عدد سكان المغرب؟",
        "darija_query": "شحال ديال الساكنة كاينة فالمغرب؟",
        "gold_answer": "يبلغ عدد سكان المغرب حوالي 37 مليون نسمة.",
        "source_chunk_id": "chunk_002",
        "source_chunk_text": "بلغ عدد سكان المملكة المغربية حوالي 37 مليون نسمة وفق آخر إحصاء رسمي أجرته المندوبية السامية للتخطيط. يتركز معظم السكان في المناطق الحضرية على طول الساحل الأطلسي."
    },
    {
        "id": "b004",
        "msa_query": "من أجرى آخر إحصاء رسمي للسكان في المغرب؟",
        "darija_query": "شكون لي دار آخر إحصاء ديال الساكنة فالمغرب؟",
        "gold_answer": "المندوبية السامية للتخطيط هي التي أجرت آخر إحصاء رسمي.",
        "source_chunk_id": "chunk_002",
        "source_chunk_text": "بلغ عدد سكان المملكة المغربية حوالي 37 مليون نسمة وفق آخر إحصاء رسمي أجرته المندوبية السامية للتخطيط. يتركز معظم السكان في المناطق الحضرية على طول الساحل الأطلسي."
    },
    {
        "id": "b005",
        "msa_query": "متى تأسست جامعة القرويين؟",
        "darija_query": "إمتى تأسست جامعة القرويين؟",
        "gold_answer": "تأسست جامعة القرويين سنة 859 ميلادية.",
        "source_chunk_id": "chunk_003",
        "source_chunk_text": "تأسست جامعة القرويين سنة 859 ميلادية في مدينة فاس على يد فاطمة الفهرية، وتعتبر من أقدم الجامعات في العالم التي ما زالت تمارس نشاطها التعليمي حتى اليوم."
    },
    {
        "id": "b006",
        "msa_query": "من أسس جامعة القرويين؟",
        "darija_query": "شكون لي أسس جامعة القرويين؟",
        "gold_answer": "أسستها فاطمة الفهرية.",
        "source_chunk_id": "chunk_003",
        "source_chunk_text": "تأسست جامعة القرويين سنة 859 ميلادية في مدينة فاس على يد فاطمة الفهرية، وتعتبر من أقدم الجامعات في العالم التي ما زالت تمارس نشاطها التعليمي حتى اليوم."
    },
    {
        "id": "b007",
        "msa_query": "كم سنة تستغرق المرحلة الابتدائية في المغرب؟",
        "darija_query": "شحال ديال السنين كتدوم المرحلة الابتدائية فالمغرب؟",
        "gold_answer": "تستغرق المرحلة الابتدائية ست سنوات.",
        "source_chunk_id": "chunk_004",
        "source_chunk_text": "يعتمد نظام التعليم في المغرب على مرحلة ابتدائية مدتها ست سنوات، تليها مرحلة إعدادية وثانوية. التعليم الابتدائي إلزامي ومجاني في المؤسسات العمومية."
    },
    {
        "id": "b008",
        "msa_query": "هل التعليم الابتدائي إلزامي في المغرب؟",
        "darija_query": "واش التعليم الابتدائي إجباري فالمغرب؟",
        "gold_answer": "نعم، التعليم الابتدائي إلزامي ومجاني في المؤسسات العمومية.",
        "source_chunk_id": "chunk_004",
        "source_chunk_text": "يعتمد نظام التعليم في المغرب على مرحلة ابتدائية مدتها ست سنوات، تليها مرحلة إعدادية وثانوية. التعليم الابتدائي إلزامي ومجاني في المؤسسات العمومية."
    },
    {
        "id": "b009",
        "msa_query": "ما هو نظام التغطية الصحية الذي أطلقته وزارة الصحة؟",
        "darija_query": "شنو هو نظام التغطية الصحية لي دارتو وزارة الصحة؟",
        "gold_answer": "أطلقت وزارة الصحة نظام التغطية الصحية الإجبارية الأساسية.",
        "source_chunk_id": "chunk_005",
        "source_chunk_text": "أطلقت وزارة الصحة المغربية نظام التغطية الصحية الإجبارية الأساسية لتوفير الرعاية الطبية لجميع المواطنين، بما في ذلك العمال غير الأجراء والفئات الهشة."
    },
    {
        "id": "b010",
        "msa_query": "لمن يوفر نظام التغطية الصحية الرعاية الطبية؟",
        "darija_query": "لمن كيوفر نظام التغطية الصحية الرعاية الطبية؟",
        "gold_answer": "يوفرها لجميع المواطنين، بما في ذلك العمال غير الأجراء والفئات الهشة.",
        "source_chunk_id": "chunk_005",
        "source_chunk_text": "أطلقت وزارة الصحة المغربية نظام التغطية الصحية الإجبارية الأساسية لتوفير الرعاية الطبية لجميع المواطنين، بما في ذلك العمال غير الأجراء والفئات الهشة."
    },
    {
        "id": "b011",
        "msa_query": "كم يبلغ طول الساحل المغربي؟",
        "darija_query": "شحال طول الساحل ديال المغرب؟",
        "gold_answer": "يمتد الساحل المغربي على مسافة تفوق 3500 كيلومتر.",
        "source_chunk_id": "chunk_006",
        "source_chunk_text": "يمتد الساحل المغربي على مسافة تفوق 3500 كيلومتر بين البحر الأبيض المتوسط شمالاً والمحيط الأطلسي غرباً وجنوباً، ما يمنح البلاد موقعاً استراتيجياً هاماً."
    },
    {
        "id": "b012",
        "msa_query": "بين أي بحرين يمتد الساحل المغربي؟",
        "darija_query": "بين شنو ديال البحور كيتمد الساحل ديال المغرب؟",
        "gold_answer": "يمتد بين البحر الأبيض المتوسط شمالاً والمحيط الأطلسي غرباً وجنوباً.",
        "source_chunk_id": "chunk_006",
        "source_chunk_text": "يمتد الساحل المغربي على مسافة تفوق 3500 كيلومتر بين البحر الأبيض المتوسط شمالاً والمحيط الأطلسي غرباً وجنوباً، ما يمنح البلاد موقعاً استراتيجياً هاماً."
    },
    {
        "id": "b013",
        "msa_query": "ما هو أهم قطاع اقتصادي في المغرب حسب النص؟",
        "darija_query": "شنو هو أهم قطاع اقتصادي فالمغرب حسب النص؟",
        "gold_answer": "تعتبر الفلاحة من أهم القطاعات الاقتصادية في المغرب.",
        "source_chunk_id": "chunk_007",
        "source_chunk_text": "تعتبر الفلاحة من أهم القطاعات الاقتصادية في المغرب، حيث تشغل نسبة كبيرة من اليد العاملة، وتساهم زراعة الحبوب والزيتون والحوامض بشكل أساسي في الإنتاج الوطني."
    },
    {
        "id": "b014",
        "msa_query": "ما هي أهم المحاصيل الزراعية المذكورة؟",
        "darija_query": "شنو هما أهم الغلات الفلاحية لي تهضرات عليهم؟",
        "gold_answer": "زراعة الحبوب والزيتون والحوامض.",
        "source_chunk_id": "chunk_007",
        "source_chunk_text": "تعتبر الفلاحة من أهم القطاعات الاقتصادية في المغرب، حيث تشغل نسبة كبيرة من اليد العاملة، وتساهم زراعة الحبوب والزيتون والحوامض بشكل أساسي في الإنتاج الوطني."
    },
    {
        "id": "b015",
        "msa_query": "متى أُدرجت مراكش القديمة في قائمة التراث العالمي؟",
        "darija_query": "إمتى تدرجات مراكش القديمة فلائحة التراث العالمي؟",
        "gold_answer": "أُدرجت سنة 1985.",
        "source_chunk_id": "chunk_008",
        "source_chunk_text": "أُدرجت مدينة مراكش القديمة ضمن قائمة التراث العالمي لمنظمة اليونسكو سنة 1985، وذلك تقديراً لعمارتها التاريخية وساحة جامع الفنا التي تشتهر بالحلقات الشعبية."
    },
    {
        "id": "b016",
        "msa_query": "بأي ساحة تشتهر مراكش القديمة؟",
        "darija_query": "بأش ساحة مشهورة مراكش القديمة؟",
        "gold_answer": "تشتهر بساحة جامع الفنا التي تشتهر بالحلقات الشعبية.",
        "source_chunk_id": "chunk_008",
        "source_chunk_text": "أُدرجت مدينة مراكش القديمة ضمن قائمة التراث العالمي لمنظمة اليونسكو سنة 1985، وذلك تقديراً لعمارتها التاريخية وساحة جامع الفنا التي تشتهر بالحلقات الشعبية."
    },
    {
        "id": "b017",
        "msa_query": "ما هي العاصمة الإدارية للمغرب؟",
        "darija_query": "شنو هي العاصمة الإدارية ديال المغرب؟",
        "gold_answer": "الرباط هي العاصمة الإدارية للمملكة المغربية.",
        "source_chunk_id": "chunk_001",
        "source_chunk_text": "الرباط هي العاصمة الإدارية للمملكة المغربية، وتقع على ساحل المحيط الأطلسي عند مصب نهر أبي رقراق. تأسست المدينة الحديثة في القرن الثاني عشر الميلادي، وتضم اليوم مقرات الحكومة والبرلمان والسفارات الأجنبية."
    },
    {
        "id": "b018",
        "msa_query": "هل نظام التعليم يشمل مرحلة إعدادية بعد الابتدائي؟",
        "darija_query": "واش نظام التعليم فيه مرحلة إعدادية من بعد الابتدائي؟",
        "gold_answer": "نعم، تليها مرحلة إعدادية وثانوية.",
        "source_chunk_id": "chunk_004",
        "source_chunk_text": "يعتمد نظام التعليم في المغرب على مرحلة ابتدائية مدتها ست سنوات، تليها مرحلة إعدادية وثانوية. التعليم الابتدائي إلزامي ومجاني في المؤسسات العمومية."
    },
    {
        "id": "b019",
        "msa_query": "في أي منظمة أُدرجت مراكش القديمة؟",
        "darija_query": "فأش منظمة تدرجات مراكش القديمة؟",
        "gold_answer": "أُدرجت في قائمة التراث العالمي لمنظمة اليونسكو.",
        "source_chunk_id": "chunk_008",
        "source_chunk_text": "أُدرجت مدينة مراكش القديمة ضمن قائمة التراث العالمي لمنظمة اليونسكو سنة 1985، وذلك تقديراً لعمارتها التاريخية وساحة جامع الفنا التي تشتهر بالحلقات الشعبية."
    },
    {
        "id": "b020",
        "msa_query": "أين تتركز غالبية سكان المغرب؟",
        "darija_query": "فين كتترکز غالبية الساكنة ديال المغرب؟",
        "gold_answer": "يتركز معظم السكان في المناطق الحضرية على طول الساحل الأطلسي.",
        "source_chunk_id": "chunk_002",
        "source_chunk_text": "بلغ عدد سكان المملكة المغربية حوالي 37 مليون نسمة وفق آخر إحصاء رسمي أجرته المندوبية السامية للتخطيط. يتركز معظم السكان في المناطق الحضرية على طول الساحل الأطلسي."
    },
    {
        "id": "b021",
        "msa_query": "بكم نسبة نما الاقتصاد المغربي خلال السنة الماضية؟",
        "darija_query": "بشحال دار النمو ديال الاقتصاد المغربي العام لي فات؟",
        "gold_answer": "نما الاقتصاد المغربي بنسبة تقارب 3.5 بالمئة.",
        "source_chunk_id": "chunk_009",
        "source_chunk_text": "سجل الاقتصاد المغربي نمواً بنسبة تقارب 3.5 بالمئة خلال السنة الماضية، مدفوعاً بانتعاش القطاع الفلاحي وتحسن أداء الصناعات التحويلية والسياحة."
    },
    {
        "id": "b022",
        "msa_query": "ما هي القطاعات التي دفعت النمو الاقتصادي حسب النص؟",
        "darija_query": "شنو هوما القطاعات لي دفعو النمو الاقتصادي حسب النص؟",
        "gold_answer": "القطاع الفلاحي والصناعات التحويلية والسياحة.",
        "source_chunk_id": "chunk_009",
        "source_chunk_text": "سجل الاقتصاد المغربي نمواً بنسبة تقارب 3.5 بالمئة خلال السنة الماضية، مدفوعاً بانتعاش القطاع الفلاحي وتحسن أداء الصناعات التحويلية والسياحة."
    },
    {
        "id": "b023",
        "msa_query": "كم يبلغ الحد الأدنى للأجور في القطاع الخاص بالمغرب؟",
        "darija_query": "شحال هو السميك ديال القطاع الخاص فالمغرب؟",
        "gold_answer": "يبلغ الحد الأدنى للأجور ما يقارب 3111 درهماً شهرياً.",
        "source_chunk_id": "chunk_010",
        "source_chunk_text": "يبلغ الحد الأدنى للأجور في القطاع الخاص بالمغرب، المعروف اختصاراً بالسميك، ما يقارب 3111 درهماً شهرياً، مع مراجعات دورية تحددها الحكومة بالتشاور مع النقابات وأرباب العمل."
    },
    {
        "id": "b024",
        "msa_query": "من يحدد مراجعة الحد الأدنى للأجور؟",
        "darija_query": "شكون لي كيحدد مراجعة السميك؟",
        "gold_answer": "تحددها الحكومة بالتشاور مع النقابات وأرباب العمل.",
        "source_chunk_id": "chunk_010",
        "source_chunk_text": "يبلغ الحد الأدنى للأجور في القطاع الخاص بالمغرب، المعروف اختصاراً بالسميك، ما يقارب 3111 درهماً شهرياً، مع مراجعات دورية تحددها الحكومة بالتشاور مع النقابات وأرباب العمل."
    },
    {
        "id": "b025",
        "msa_query": "كم يبلغ معدل البطالة الوطني في المغرب؟",
        "darija_query": "شحال ديال البطالة كاينة فالمغرب؟",
        "gold_answer": "يبلغ معدل البطالة حوالي 13 بالمئة على المستوى الوطني.",
        "source_chunk_id": "chunk_011",
        "source_chunk_text": "بلغ معدل البطالة في المغرب حوالي 13 بالمئة على المستوى الوطني، مع تفاوت كبير بين الوسط الحضري الذي يسجل نسبة أعلى والوسط القروي، ويتركز جزء كبير من العاطلين بين فئة الشباب."
    },
    {
        "id": "b026",
        "msa_query": "أي فئة تتركز فيها البطالة بشكل أكبر؟",
        "darija_query": "فأش فئة كتترکز البطالة بزاف؟",
        "gold_answer": "يتركز جزء كبير من العاطلين بين فئة الشباب.",
        "source_chunk_id": "chunk_011",
        "source_chunk_text": "بلغ معدل البطالة في المغرب حوالي 13 بالمئة على المستوى الوطني، مع تفاوت كبير بين الوسط الحضري الذي يسجل نسبة أعلى والوسط القروي، ويتركز جزء كبير من العاطلين بين فئة الشباب."
    },
    {
        "id": "b027",
        "msa_query": "ما هو الهدف من برنامج الدعم الاجتماعي المباشر؟",
        "darija_query": "شنو هو الهدف ديال برنامج الدعم الاجتماعي المباشر؟",
        "gold_answer": "تقديم مساعدات مالية شهرية للأسر الفقيرة والهشة التي لديها أطفال في سن التمدرس أو دون سن السادسة.",
        "source_chunk_id": "chunk_012",
        "source_chunk_text": "أطلقت الحكومة المغربية برنامج الدعم الاجتماعي المباشر لفائدة الأسر الفقيرة والهشة، ويقدم البرنامج مساعدات مالية شهرية للأسر التي لديها أطفال في سن التمدرس أو أطفال دون سن السادسة."
    },
    {
        "id": "b028",
        "msa_query": "من هي الفئة المستفيدة من برنامج الدعم الاجتماعي المباشر؟",
        "darija_query": "شكون لي كيستافد من برنامج الدعم الاجتماعي المباشر؟",
        "gold_answer": "الأسر الفقيرة والهشة التي لديها أطفال في سن التمدرس أو دون سن السادسة.",
        "source_chunk_id": "chunk_012",
        "source_chunk_text": "أطلقت الحكومة المغربية برنامج الدعم الاجتماعي المباشر لفائدة الأسر الفقيرة والهشة، ويقدم البرنامج مساعدات مالية شهرية للأسر التي لديها أطفال في سن التمدرس أو أطفال دون سن السادسة."
    },
    {
        "id": "b029",
        "msa_query": "كم تبلغ نسبة مشاركة المرأة في سوق الشغل بالمغرب؟",
        "darija_query": "شحال هي نسبة المرأة لي خدامة فسوق الشغل فالمغرب؟",
        "gold_answer": "لا تتجاوز نسبة مشاركة المرأة في سوق الشغل 20 بالمئة.",
        "source_chunk_id": "chunk_013",
        "source_chunk_text": "لا تتجاوز نسبة مشاركة المرأة في سوق الشغل بالمغرب 20 بالمئة، وهي من أدنى النسب في المنطقة، رغم أن النساء يشكلن ما يقارب نصف عدد السكان في سن العمل."
    },
    {
        "id": "b030",
        "msa_query": "بكم نسبة يساهم قطاع السياحة في الناتج الداخلي الخام؟",
        "darija_query": "بشحال كيساهم قطاع السياحة فالناتج الداخلي الخام؟",
        "gold_answer": "يساهم قطاع السياحة بنسبة تفوق 7 بالمئة من الناتج الداخلي الخام.",
        "source_chunk_id": "chunk_014",
        "source_chunk_text": "يساهم قطاع السياحة بنسبة تفوق 7 بالمئة من الناتج الداخلي الخام للمغرب، حيث استقبلت البلاد أكثر من 14 مليون سائح خلال السنة الماضية، أغلبهم قادمون من فرنسا وإسبانيا."
    },
    {
        "id": "b031",
        "msa_query": "كم عدد السياح الذين استقبلهم المغرب خلال السنة الماضية؟",
        "darija_query": "شحال ديال السياح لي جاو للمغرب العام لي فات؟",
        "gold_answer": "استقبل المغرب أكثر من 14 مليون سائح خلال السنة الماضية.",
        "source_chunk_id": "chunk_014",
        "source_chunk_text": "يساهم قطاع السياحة بنسبة تفوق 7 بالمئة من الناتج الداخلي الخام للمغرب، حيث استقبلت البلاد أكثر من 14 مليون سائح خلال السنة الماضية، أغلبهم قادمون من فرنسا وإسبانيا."
    },
    {
        "id": "b032",
        "msa_query": "من أين يأتي معظم السياح الذين يزورون المغرب؟",
        "darija_query": "منين كيجيو أغلب السياح لي كيزورو المغرب؟",
        "gold_answer": "أغلبهم قادمون من فرنسا وإسبانيا.",
        "source_chunk_id": "chunk_014",
        "source_chunk_text": "يساهم قطاع السياحة بنسبة تفوق 7 بالمئة من الناتج الداخلي الخام للمغرب، حيث استقبلت البلاد أكثر من 14 مليون سائح خلال السنة الماضية، أغلبهم قادمون من فرنسا وإسبانيا."
    },
    {
        "id": "b033",
        "msa_query": "كم بلغت تحويلات المغاربة المقيمين بالخارج خلال السنة الماضية؟",
        "darija_query": "شحال وصلات التحويلات ديال المغاربة لي ساكنين بره العام لي فات؟",
        "gold_answer": "تجاوزت هذه التحويلات 11 مليار دولار.",
        "source_chunk_id": "chunk_015",
        "source_chunk_text": "تشكل تحويلات المغاربة المقيمين بالخارج مصدراً أساسياً للعملة الصعبة، إذ تجاوزت هذه التحويلات 11 مليار دولار خلال السنة الماضية، وهي تفوق أحياناً عائدات السياحة والاستثمار الأجنبي المباشر مجتمعين."
    },
    {
        "id": "b034",
        "msa_query": "ماذا تمثل تحويلات المغاربة المقيمين بالخارج بالنسبة للاقتصاد؟",
        "darija_query": "أش كتمثل التحويلات ديال المغاربة لي ساكنين بره بالنسبة للاقتصاد؟",
        "gold_answer": "تمثل مصدراً أساسياً للعملة الصعبة، وتفوق أحياناً عائدات السياحة والاستثمار الأجنبي المباشر مجتمعين.",
        "source_chunk_id": "chunk_015",
        "source_chunk_text": "تشكل تحويلات المغاربة المقيمين بالخارج مصدراً أساسياً للعملة الصعبة، إذ تجاوزت هذه التحويلات 11 مليار دولار خلال السنة الماضية، وهي تفوق أحياناً عائدات السياحة والاستثمار الأجنبي المباشر مجتمعين."
    },
    {
        "id": "b035",
        "msa_query": "ما هي الحصة المستهدفة للطاقات المتجددة في مزيج الطاقة الكهربائية بالمغرب؟",
        "darija_query": "شحال هي الحصة لي غادي توصلها الطاقات المتجددة فالكهربا فالمغرب؟",
        "gold_answer": "تستهدف الاستراتيجية الوصول بحصة الطاقات النظيفة إلى أكثر من 52 بالمئة.",
        "source_chunk_id": "chunk_016",
        "source_chunk_text": "أطلق المغرب استراتيجية للطاقات المتجددة تهدف إلى الوصول بحصة الطاقات النظيفة إلى أكثر من 52 بالمئة من مزيج الطاقة الكهربائية، اعتماداً بشكل أساسي على الطاقة الشمسية والريحية، ومن أبرز مشاريعها محطة نور بورزازات."
    },
    {
        "id": "b036",
        "msa_query": "ما هو أبرز مشروع للطاقة الشمسية المذكور في النص؟",
        "darija_query": "شنو هو أبرز مشروع ديال الطاقة الشمسية لي تهضر عليه النص؟",
        "gold_answer": "من أبرز المشاريع محطة نور بورزازات.",
        "source_chunk_id": "chunk_016",
        "source_chunk_text": "أطلق المغرب استراتيجية للطاقات المتجددة تهدف إلى الوصول بحصة الطاقات النظيفة إلى أكثر من 52 بالمئة من مزيج الطاقة الكهربائية، اعتماداً بشكل أساسي على الطاقة الشمسية والريحية، ومن أبرز مشاريعها محطة نور بورزازات."
    },
    {
        "id": "b037",
        "msa_query": "لماذا تلجأ الحكومة إلى بناء محطات تحلية مياه البحر؟",
        "darija_query": "علاش الحكومة كتلجأ لبناء محطات تحلية مياه البحر؟",
        "gold_answer": "بسبب الضغط المتزايد على الموارد المائية نتيجة توالي سنوات الجفاف.",
        "source_chunk_id": "chunk_017",
        "source_chunk_text": "يعاني المغرب من ضغط متزايد على موارده المائية بسبب توالي سنوات الجفاف، ما دفع الحكومة إلى تسريع بناء محطات تحلية مياه البحر، خصوصاً في المدن الساحلية الكبرى مثل الدار البيضاء وأكادير."
    },
    {
        "id": "b038",
        "msa_query": "في أي مدن تتركز محطات تحلية المياه؟",
        "darija_query": "فأي مدن كتترکز محطات تحلية الما؟",
        "gold_answer": "خصوصاً في المدن الساحلية الكبرى مثل الدار البيضاء وأكادير.",
        "source_chunk_id": "chunk_017",
        "source_chunk_text": "يعاني المغرب من ضغط متزايد على موارده المائية بسبب توالي سنوات الجفاف، ما دفع الحكومة إلى تسريع بناء محطات تحلية مياه البحر، خصوصاً في المدن الساحلية الكبرى مثل الدار البيضاء وأكادير."
    },
    {
        "id": "b039",
        "msa_query": "ما هو هدف برنامج المغرب الرقمي؟",
        "darija_query": "شنو هو الهدف ديال برنامج المغرب الرقمي؟",
        "gold_answer": "تسريع التحول الرقمي للإدارة العمومية.",
        "source_chunk_id": "chunk_018",
        "source_chunk_text": "أطلقت الحكومة المغربية برنامج المغرب الرقمي بهدف تسريع التحول الرقمي للإدارة العمومية، ويشمل البرنامج رقمنة الخدمات الإدارية وتوسيع الولوج إلى الإنترنت عالي الصبيب في المناطق القروية."
    },
    {
        "id": "b040",
        "msa_query": "ماذا يشمل برنامج المغرب الرقمي؟",
        "darija_query": "أش كيشمل برنامج المغرب الرقمي؟",
        "gold_answer": "رقمنة الخدمات الإدارية وتوسيع الولوج إلى الإنترنت عالي الصبيب في المناطق القروية.",
        "source_chunk_id": "chunk_018",
        "source_chunk_text": "أطلقت الحكومة المغربية برنامج المغرب الرقمي بهدف تسريع التحول الرقمي للإدارة العمومية، ويشمل البرنامج رقمنة الخدمات الإدارية وتوسيع الولوج إلى الإنترنت عالي الصبيب في المناطق القروية."
    },
    {
        "id": "b041",
        "msa_query": "ما هو الإنجاز التاريخي للمنتخب المغربي في كأس العالم؟",
        "darija_query": "شنو هو الإنجاز التاريخي ديال المنتخب المغربي فكأس العالم؟",
        "gold_answer": "أصبح أول منتخب إفريقي وعربي يصل إلى نصف نهائي كأس العالم.",
        "source_chunk_id": "chunk_019",
        "source_chunk_text": "يعتبر المنتخب المغربي لكرة القدم أول منتخب إفريقي وعربي يصل إلى نصف نهائي كأس العالم، وذلك خلال نسخة قطر 2022، ما اعتبر إنجازاً تاريخياً على المستويين القاري والعربي."
    },
    {
        "id": "b042",
        "msa_query": "في أي نسخة من كأس العالم تحقق هذا الإنجاز؟",
        "darija_query": "فأي نسخة ديال كأس العالم تحقق هاد الإنجاز؟",
        "gold_answer": "تحقق ذلك خلال نسخة قطر 2022.",
        "source_chunk_id": "chunk_019",
        "source_chunk_text": "يعتبر المنتخب المغربي لكرة القدم أول منتخب إفريقي وعربي يصل إلى نصف نهائي كأس العالم، وذلك خلال نسخة قطر 2022، ما اعتبر إنجازاً تاريخياً على المستويين القاري والعربي."
    },
    {
        "id": "b043",
        "msa_query": "ما هي أعلى هيئة قضائية في النظام القضائي المغربي؟",
        "darija_query": "شنو هي أعلى هيئة قضائية فالنظام القضائي المغربي؟",
        "gold_answer": "محكمة النقض هي أعلى هيئة قضائية.",
        "source_chunk_id": "chunk_020",
        "source_chunk_text": "يتألف النظام القضائي المغربي من محاكم ابتدائية ومحاكم استئناف ومحكمة النقض كأعلى هيئة قضائية، وقد شهد القضاء إصلاحات لتعزيز استقلاليته بعد دستور 2011."
    },
    {
        "id": "b044",
        "msa_query": "متى شهد القضاء إصلاحات لتعزيز استقلاليته؟",
        "darija_query": "إمتى شهد القضاء إصلاحات باش يتعزز استقلاله؟",
        "gold_answer": "بعد دستور 2011.",
        "source_chunk_id": "chunk_020",
        "source_chunk_text": "يتألف النظام القضائي المغربي من محاكم ابتدائية ومحاكم استئناف ومحكمة النقض كأعلى هيئة قضائية، وقد شهد القضاء إصلاحات لتعزيز استقلاليته بعد دستور 2011."
    },
    {
        "id": "b045",
        "msa_query": "كم يبلغ الطول الإجمالي لشبكة الطرق السيارة بالمغرب؟",
        "darija_query": "شحال طول شبكة الطرق السيارة فالمغرب؟",
        "gold_answer": "يتجاوز طولها الإجمالي حالياً 1800 كيلومتر.",
        "source_chunk_id": "chunk_021",
        "source_chunk_text": "تشهد شبكة الطرق السيارة بالمغرب توسعاً مستمراً، إذ يتجاوز طولها الإجمالي حالياً 1800 كيلومتر، ما يربط أهم المدن الاقتصادية ويساهم في تسهيل حركة التجارة والنقل."
    },
    {
        "id": "b046",
        "msa_query": "ماذا يجمع مهرجان كناوة وموسيقى العالم؟",
        "darija_query": "أش كيجمع مهرجان كناوة وموسيقى العالم؟",
        "gold_answer": "يجمع بين الموسيقى الكناوية التقليدية وفنون موسيقية عالمية متنوعة.",
        "source_chunk_id": "chunk_022",
        "source_chunk_text": "يستقطب مهرجان كناوة وموسيقى العالم بمدينة الصويرة آلاف الزوار سنوياً، ويجمع بين الموسيقى الكناوية التقليدية وفنون موسيقية عالمية متنوعة في أجواء ساحلية مميزة."
    },
    {
        "id": "b047",
        "msa_query": "في أي مدينة يقام مهرجان كناوة؟",
        "darija_query": "فأي مدينة كيتقام مهرجان كناوة؟",
        "gold_answer": "يقام في مدينة الصويرة.",
        "source_chunk_id": "chunk_022",
        "source_chunk_text": "يستقطب مهرجان كناوة وموسيقى العالم بمدينة الصويرة آلاف الزوار سنوياً، ويجمع بين الموسيقى الكناوية التقليدية وفنون موسيقية عالمية متنوعة في أجواء ساحلية مميزة."
    },
    {
        "id": "b048",
        "msa_query": "ما هي التقنية التي يعتمدها قطاع الفلاحة لترشيد استهلاك المياه؟",
        "darija_query": "شنو هي التقنية لي كيعتمد عليها قطاع الفلاحة باش يرشد الما؟",
        "gold_answer": "يعتمد بشكل متزايد على تقنيات الري الموضعي (الري بالتنقيط).",
        "source_chunk_id": "chunk_023",
        "source_chunk_text": "يعتمد قطاع الفلاحة المغربي بشكل متزايد على تقنيات الري الموضعي لترشيد استهلاك المياه، وقد استفادت آلاف الهكتارات من برامج تحويل الري التقليدي إلى الري بالتنقيط."
    },
    {
        "id": "b049",
        "msa_query": "بأي طائر مهدد بالانقراض تشتهر محمية سوس ماسة؟",
        "darija_query": "بأش طائر مهدد بالانقراض مشهورة محمية سوس ماسة؟",
        "gold_answer": "تشتهر بطائر الحبارى المهدد بالانقراض.",
        "source_chunk_id": "chunk_024",
        "source_chunk_text": "أُدرجت محمية سوس ماسة الطبيعية ضمن المناطق المحمية ذات الأولوية بالمغرب، وتُعد موطناً لطائر الحبارى الذي يواجه خطر الانقراض، إضافة إلى أنواع نباتية وحيوانية نادرة أخرى."
    },
    {
        "id": "b050",
        "msa_query": "أي دستور يضمن حرية الصحافة في المغرب؟",
        "darija_query": "أي دستور كيضمن حرية الصحافة فالمغرب؟",
        "gold_answer": "دستور 2011 يضمن حرية الصحافة والنشر.",
        "source_chunk_id": "chunk_025",
        "source_chunk_text": "يضمن الدستور المغربي لسنة 2011 حرية الصحافة والنشر، مع وجود قانون خاص بالصحافة والنشر ينظم حقوق وواجبات الصحفيين ويحدد الأطر القانونية لممارسة المهنة."
    },
    {
        "id": "b051",
        "msa_query": "ما الذي ينظمه قانون الصحافة والنشر؟",
        "darija_query": "أش كينظم قانون الصحافة والنشر؟",
        "gold_answer": "ينظم حقوق وواجبات الصحفيين ويحدد الأطر القانونية لممارسة المهنة.",
        "source_chunk_id": "chunk_025",
        "source_chunk_text": "يضمن الدستور المغربي لسنة 2011 حرية الصحافة والنشر، مع وجود قانون خاص بالصحافة والنشر ينظم حقوق وواجبات الصحفيين ويحدد الأطر القانونية لممارسة المهنة."
    },
    {
        "id": "b052",
        "msa_query": "متى تُقام الأسواق الأسبوعية في المغرب؟",
        "darija_query": "إمتى كيكون السوق الأسبوعي فالمغرب؟",
        "gold_answer": "تُقام في يوم محدد من الأسبوع حسب كل مدينة أو قرية.",
        "source_chunk_id": "chunk_026",
        "source_chunk_text": "تُقام الأسواق الأسبوعية (السوق) في أغلب المدن والقرى المغربية في يوم محدد من الأسبوع، حيث يعرض الفلاحون والباعة منتجاتهم من خضر وفواكه ومواشي مباشرة للمستهلكين، وتبقى المساومة على الأثمنة عادة شائعة هناك."
    },
    {
        "id": "b053",
        "msa_query": "ماذا يعرض الفلاحون والباعة في السوق الأسبوعي؟",
        "darija_query": "أش كيبيعو الفلاحين والبياعة فالسوق؟",
        "gold_answer": "يعرضون منتجاتهم من خضر وفواكه ومواشي مباشرة للمستهلكين.",
        "source_chunk_id": "chunk_026",
        "source_chunk_text": "تُقام الأسواق الأسبوعية (السوق) في أغلب المدن والقرى المغربية في يوم محدد من الأسبوع، حيث يعرض الفلاحون والباعة منتجاتهم من خضر وفواكه ومواشي مباشرة للمستهلكين، وتبقى المساومة على الأثمنة عادة شائعة هناك."
    },
    {
        "id": "b054",
        "msa_query": "هل المساومة على الأثمنة شائعة في السوق الأسبوعي؟",
        "darija_query": "واش الساوماة على الثمن حاجة عادية فالسوق؟",
        "gold_answer": "نعم، تبقى المساومة على الأثمنة عادة شائعة هناك.",
        "source_chunk_id": "chunk_026",
        "source_chunk_text": "تُقام الأسواق الأسبوعية (السوق) في أغلب المدن والقرى المغربية في يوم محدد من الأسبوع، حيث يعرض الفلاحون والباعة منتجاتهم من خضر وفواكه ومواشي مباشرة للمستهلكين، وتبقى المساومة على الأثمنة عادة شائعة هناك."
    },
    {
        "id": "b055",
        "msa_query": "ما الفرق بين الطاكسي الكبير والطاكسي الصغير؟",
        "darija_query": "شنو الفرق بين الطاكسي الكبير والطاكسي الصغير؟",
        "gold_answer": "الطاكسي الكبير ينقل الركاب بين المدن، والطاكسي الصغير يعمل داخل المدينة.",
        "source_chunk_id": "chunk_027",
        "source_chunk_text": "تنقسم سيارات الأجرة في المغرب إلى الطاكسي الكبير الذي ينقل الركاب بين المدن بثمن يحدده كل راكب حسب المسافة، والطاكسي الصغير الذي يعمل داخل المدينة ويشتغل بالعداد أو بثمن متفق عليه مسبقاً."
    },
    {
        "id": "b056",
        "msa_query": "كيف يُحدد ثمن الطاكسي الصغير؟",
        "darija_query": "كيفاش كيتحدد الثمن ديال الطاكسي الصغير؟",
        "gold_answer": "يشتغل بالعداد أو بثمن متفق عليه مسبقاً.",
        "source_chunk_id": "chunk_027",
        "source_chunk_text": "تنقسم سيارات الأجرة في المغرب إلى الطاكسي الكبير الذي ينقل الركاب بين المدن بثمن يحدده كل راكب حسب المسافة، والطاكسي الصغير الذي يعمل داخل المدينة ويشتغل بالعداد أو بثمن متفق عليه مسبقاً."
    },
    {
        "id": "b057",
        "msa_query": "في أي ساعة يبدأ الطرامواي عمله وفي أي ساعة يتوقف؟",
        "darija_query": "فأي ساعة كيبدا الطرامواي وفأي ساعة كيوقف؟",
        "gold_answer": "يشتغل يومياً من الساعة السادسة صباحاً إلى منتصف الليل تقريباً.",
        "source_chunk_id": "chunk_028",
        "source_chunk_text": "يشتغل الطرامواي في الدار البيضاء والرباط يومياً من الساعة السادسة صباحاً إلى منتصف الليل تقريباً، وتتراوح تذكرة الرحلة الواحدة حوالي 7 دراهم، مع وجود اشتراكات شهرية بثمن مخفض للمستعملين المنتظمين."
    },
    {
        "id": "b058",
        "msa_query": "كم يبلغ ثمن تذكرة الطرامواي؟",
        "darija_query": "بشحال التذكرة ديال الطرامواي؟",
        "gold_answer": "تتراوح تذكرة الرحلة الواحدة حوالي 7 دراهم.",
        "source_chunk_id": "chunk_028",
        "source_chunk_text": "يشتغل الطرامواي في الدار البيضاء والرباط يومياً من الساعة السادسة صباحاً إلى منتصف الليل تقريباً، وتتراوح تذكرة الرحلة الواحدة حوالي 7 دراهم، مع وجود اشتراكات شهرية بثمن مخفض للمستعملين المنتظمين."
    },
    {
        "id": "b059",
        "msa_query": "هل توجد اشتراكات شهرية للطرامواي؟",
        "darija_query": "واش كاين اشتراك شهري ديال الطرامواي؟",
        "gold_answer": "نعم، توجد اشتراكات شهرية بثمن مخفض للمستعملين المنتظمين.",
        "source_chunk_id": "chunk_028",
        "source_chunk_text": "يشتغل الطرامواي في الدار البيضاء والرباط يومياً من الساعة السادسة صباحاً إلى منتصف الليل تقريباً، وتتراوح تذكرة الرحلة الواحدة حوالي 7 دراهم، مع وجود اشتراكات شهرية بثمن مخفض للمستعملين المنتظمين."
    },
    {
        "id": "b060",
        "msa_query": "ما هي شركات الاتصالات الثلاث الكبرى في المغرب؟",
        "darija_query": "شنو هوما الشركات الكبار ديال الاتصالات فالمغرب؟",
        "gold_answer": "اتصالات المغرب وأورونج وإنوي.",
        "source_chunk_id": "chunk_029",
        "source_chunk_text": "تقدم شركات الاتصالات الثلاث الكبرى في المغرب، اتصالات المغرب وأورونج وإنوي، عروضاً متنوعة لبطاقات الأنترنت المسبقة الأداء، تتراوح أثمنتها حسب حجم الرصيد ومدة الصلاحية، وتشهد المنافسة بينها عروضاً ترويجية متكررة."
    },
    {
        "id": "b061",
        "msa_query": "ما الذي يحدد ثمن بطاقات الأنترنت المسبقة الأداء؟",
        "darija_query": "أش لي كيحدد ثمن الكارطة ديال الأنترنت؟",
        "gold_answer": "تتراوح أثمنتها حسب حجم الرصيد ومدة الصلاحية.",
        "source_chunk_id": "chunk_029",
        "source_chunk_text": "تقدم شركات الاتصالات الثلاث الكبرى في المغرب، اتصالات المغرب وأورونج وإنوي، عروضاً متنوعة لبطاقات الأنترنت المسبقة الأداء، تتراوح أثمنتها حسب حجم الرصيد ومدة الصلاحية، وتشهد المنافسة بينها عروضاً ترويجية متكررة."
    },
    {
        "id": "b062",
        "msa_query": "كم يتراوح ثمن الخبز العادي في المخابز؟",
        "darija_query": "بشحال كيتبات الخبز العادي فالفرن؟",
        "gold_answer": "يتراوح ثمنه بين 1.2 و1.5 درهم للقطعة في أغلب المناطق.",
        "source_chunk_id": "chunk_030",
        "source_chunk_text": "يخضع ثمن الخبز التقليدي في المخابز لرقابة نسبية من طرف السلطات المحلية للحفاظ على ثمن معقول يتناسب مع القدرة الشرائية للمواطنين، ويتراوح ثمن الخبز العادي بين 1.2 و1.5 درهم للقطعة في أغلب المناطق."
    },
    {
        "id": "b063",
        "msa_query": "لماذا يخضع ثمن الخبز لرقابة السلطات المحلية؟",
        "darija_query": "علاش ثمن الخبز مراقب من طرف السلطات المحلية؟",
        "gold_answer": "للحفاظ على ثمن معقول يتناسب مع القدرة الشرائية للمواطنين.",
        "source_chunk_id": "chunk_030",
        "source_chunk_text": "يخضع ثمن الخبز التقليدي في المخابز لرقابة نسبية من طرف السلطات المحلية للحفاظ على ثمن معقول يتناسب مع القدرة الشرائية للمواطنين، ويتراوح ثمن الخبز العادي بين 1.2 و1.5 درهم للقطعة في أغلب المناطق."
    },
    {
        "id": "b064",
        "msa_query": "ماذا يشرب الأصدقاء عادة عند لقائهم في المقهى؟",
        "darija_query": "أش كيشربو الصحاب فالمقهى؟",
        "gold_answer": "يحتسون أتاي (الشاي بالنعناع) أو القهوة.",
        "source_chunk_id": "chunk_031",
        "source_chunk_text": "يعتبر المقهى فضاءً اجتماعياً أساسياً في الحياة اليومية المغربية، حيث يلتقي الأصدقاء لاحتساء أتاي (الشاي بالنعناع) أو القهوة ومتابعة أخبار الرياضة، وتنتشر المقاهي بكثرة في جميع الأحياء والمدن."
    },
    {
        "id": "b065",
        "msa_query": "لماذا يعتبر المقهى فضاءً اجتماعياً أساسياً؟",
        "darija_query": "علاش المقهى كيعتبر فضاء اجتماعي مهم؟",
        "gold_answer": "لأن الأصدقاء يلتقون فيه لاحتساء أتاي ومتابعة أخبار الرياضة.",
        "source_chunk_id": "chunk_031",
        "source_chunk_text": "يعتبر المقهى فضاءً اجتماعياً أساسياً في الحياة اليومية المغربية، حيث يلتقي الأصدقاء لاحتساء أتاي (الشاي بالنعناع) أو القهوة ومتابعة أخبار الرياضة، وتنتشر المقاهي بكثرة في جميع الأحياء والمدن."
    },
    {
        "id": "b066",
        "msa_query": "كيف تعمل الصيدليات خارج أوقات العمل العادية؟",
        "darija_query": "كيفاش كتخدم الصيدليات منين توقف أوقات الخدمة العادية؟",
        "gold_answer": "تعمل بنظام الحراسة بشكل تناوبي بين الصيدليات في كل مدينة.",
        "source_chunk_id": "chunk_032",
        "source_chunk_text": "تعمل بعض الصيدليات بنظام الحراسة (صيدلية الحراسة) خارج أوقات العمل العادية وأيام العطل، بشكل تناوبي بين الصيدليات في كل مدينة، لضمان توفر الأدوية للمرضى في كل الأوقات."
    },
    {
        "id": "b067",
        "msa_query": "ما هدف نظام صيدلية الحراسة؟",
        "darija_query": "شنو هو الهدف ديال صيدلية الحراسة؟",
        "gold_answer": "ضمان توفر الأدوية للمرضى في كل الأوقات.",
        "source_chunk_id": "chunk_032",
        "source_chunk_text": "تعمل بعض الصيدليات بنظام الحراسة (صيدلية الحراسة) خارج أوقات العمل العادية وأيام العطل، بشكل تناوبي بين الصيدليات في كل مدينة، لضمان توفر الأدوية للمرضى في كل الأوقات."
    },
    {
        "id": "b068",
        "msa_query": "ما هي أبرز العطل الدينية الرسمية في المغرب؟",
        "darija_query": "شنو هوما أبرز العطل الدينية فالمغرب؟",
        "gold_answer": "عيد الفطر وعيد الأضحى ورأس السنة الهجرية وعيد المولد النبوي.",
        "source_chunk_id": "chunk_033",
        "source_chunk_text": "تشمل العطل الرسمية بالمغرب عيد الفطر وعيد الأضحى ورأس السنة الهجرية وعيد المولد النبوي حسب التقويم الهجري، بالإضافة إلى عطل مدنية ثابتة مثل فاتح ماي وعيد العرش وذكرى المسيرة الخضراء."
    },
    {
        "id": "b069",
        "msa_query": "اذكر مثالاً عن عطلة مدنية ثابتة في المغرب؟",
        "darija_query": "عطيني مثال على عطلة مدنية ثابتة فالمغرب؟",
        "gold_answer": "فاتح ماي أو عيد العرش أو ذكرى المسيرة الخضراء.",
        "source_chunk_id": "chunk_033",
        "source_chunk_text": "تشمل العطل الرسمية بالمغرب عيد الفطر وعيد الأضحى ورأس السنة الهجرية وعيد المولد النبوي حسب التقويم الهجري، بالإضافة إلى عطل مدنية ثابتة مثل فاتح ماي وعيد العرش وذكرى المسيرة الخضراء."
    },
    {
        "id": "b070",
        "msa_query": "بكم يتراوح ثمن دخول الحمام العمومي؟",
        "darija_query": "بشحال كيتبات الدخول للحمام؟",
        "gold_answer": "يتراوح عادة بين 10 و20 درهماً حسب المدينة والخدمات الإضافية.",
        "source_chunk_id": "chunk_034",
        "source_chunk_text": "يبقى الحمام العمومي التقليدي جزءاً من العادات اليومية أو الأسبوعية لدى كثير من المغاربة، حيث يذهب الناس للاغتسال والتدليك التقليدي، ويتراوح ثمن الدخول عادة بين 10 و20 درهماً حسب المدينة والخدمات الإضافية."
    },
    {
        "id": "b071",
        "msa_query": "ماذا يفعل الناس عادة في الحمام العمومي؟",
        "darija_query": "أش كيديرو الناس فالحمام؟",
        "gold_answer": "يذهبون للاغتسال والتدليك التقليدي.",
        "source_chunk_id": "chunk_034",
        "source_chunk_text": "يبقى الحمام العمومي التقليدي جزءاً من العادات اليومية أو الأسبوعية لدى كثير من المغاربة، حيث يذهب الناس للاغتسال والتدليك التقليدي، ويتراوح ثمن الدخول عادة بين 10 و20 درهماً حسب المدينة والخدمات الإضافية."
    },
    {
        "id": "b072",
        "msa_query": "كم يطلب المكري عادة كوجيبة عند كراء شقة؟",
        "darija_query": "شحال كيطلب المكري ديال الوجيبة باش تكري شقة؟",
        "gold_answer": "يطلب عادة إيداع وجيبة تعادل شهراً أو شهرين من الكراء.",
        "source_chunk_id": "chunk_035",
        "source_chunk_text": "عند كراء شقة سكنية في المغرب، يطلب المكري عادة إيداع وجيبة تعادل شهراً أو شهرين من الكراء كضمانة، بالإضافة إلى توقيع عقد كراء يحدد مدة العقد وثمن الكراء الشهري وشروط الإخلاء."
    },
    {
        "id": "b073",
        "msa_query": "ماذا يحدد عقد الكراء؟",
        "darija_query": "أش كيحدد عقد الكراء؟",
        "gold_answer": "يحدد مدة العقد وثمن الكراء الشهري وشروط الإخلاء.",
        "source_chunk_id": "chunk_035",
        "source_chunk_text": "عند كراء شقة سكنية في المغرب، يطلب المكري عادة إيداع وجيبة تعادل شهراً أو شهرين من الكراء كضمانة، بالإضافة إلى توقيع عقد كراء يحدد مدة العقد وثمن الكراء الشهري وشروط الإخلاء."
    },
    {
        "id": "b074",
        "msa_query": "لماذا يفضل بعض الأسر التسوق في المتاجر الكبرى مثل مرجان وكارفور؟",
        "darija_query": "علاش بعض الناس كيفضلو يتسوقو فمرجان وكارفور؟",
        "gold_answer": "نظراً لتنوع المنتجات وثبات الأثمنة.",
        "source_chunk_id": "chunk_036",
        "source_chunk_text": "يتوجه عدد متزايد من الأسر المغربية للتسوق الأسبوعي في المتاجر الكبرى مثل مرجان وكارفور نظراً لتنوع المنتجات وثبات الأثمنة، في حين يفضل آخرون الحانوت الصغير القريب من المنزل للتسوق اليومي ولإمكانية الشراء بالدين."
    },
    {
        "id": "b075",
        "msa_query": "لماذا يفضل آخرون التسوق في الحانوت الصغير؟",
        "darija_query": "علاش بعض الناس كيفضلو الحانوت الصغير؟",
        "gold_answer": "لقربه من المنزل ولإمكانية الشراء بالدين.",
        "source_chunk_id": "chunk_036",
        "source_chunk_text": "يتوجه عدد متزايد من الأسر المغربية للتسوق الأسبوعي في المتاجر الكبرى مثل مرجان وكارفور نظراً لتنوع المنتجات وثبات الأثمنة، في حين يفضل آخرون الحانوت الصغير القريب من المنزل للتسوق اليومي ولإمكانية الشراء بالدين."
    },
    {
        "id": "b076",
        "msa_query": "ماذا يحدث لأثمنة الأغنام قبل عيد الأضحى؟",
        "darija_query": "أش كيوقع للثمن ديال الغنم قبل عيد الأضحى؟",
        "gold_answer": "ترتفع أثمنة الأغنام بشكل ملحوظ.",
        "source_chunk_id": "chunk_037",
        "source_chunk_text": "يشهد سوق الأضاحي إقبالاً كبيراً في الأسابيع التي تسبق عيد الأضحى، حيث ترتفع أثمنة الأغنام بشكل ملحوظ، وتحدد وزارة الفلاحة كل سنة توصيات لتنظيم أسواق بيع المواشي وضمان جودتها."
    },
    {
        "id": "b077",
        "msa_query": "من يحدد توصيات تنظيم أسواق بيع المواشي؟",
        "darija_query": "شكون لي كيحدد التوصيات ديال أسواق بيع المواشي؟",
        "gold_answer": "وزارة الفلاحة هي التي تحدد كل سنة توصيات لتنظيم هذه الأسواق.",
        "source_chunk_id": "chunk_037",
        "source_chunk_text": "يشهد سوق الأضاحي إقبالاً كبيراً في الأسابيع التي تسبق عيد الأضحى، حيث ترتفع أثمنة الأغنام بشكل ملحوظ، وتحدد وزارة الفلاحة كل سنة توصيات لتنظيم أسواق بيع المواشي وضمان جودتها."
    },
    {
        "id": "b078",
        "msa_query": "كيف يصل غاز البوطان إلى المنازل؟",
        "darija_query": "كيفاش كيوصل غاز البوطان للدور؟",
        "gold_answer": "يوزعه الباعة المتجولون بالعربات أو الدراجات في الأحياء يومياً.",
        "source_chunk_id": "chunk_038",
        "source_chunk_text": "يعتمد كثير من المنازل المغربية على قنينة غاز البوطان للطبخ والتسخين، ويوزعها الباعة المتجولون بالعربات أو الدراجات في الأحياء يومياً، ويبقى ثمنها مدعماً من طرف الدولة لضمان وصولها لجميع الفئات."
    },
    {
        "id": "b079",
        "msa_query": "لماذا يبقى ثمن قنينة الغاز في متناول الجميع؟",
        "darija_query": "علاش ثمن قنينة الغاز فمتناول الجميع؟",
        "gold_answer": "لأن ثمنها مدعوم من طرف الدولة.",
        "source_chunk_id": "chunk_038",
        "source_chunk_text": "يعتمد كثير من المنازل المغربية على قنينة غاز البوطان للطبخ والتسخين، ويوزعها الباعة المتجولون بالعربات أو الدراجات في الأحياء يومياً، ويبقى ثمنها مدعماً من طرف الدولة لضمان وصولها لجميع الفئات."
    },
    {
        "id": "b080",
        "msa_query": "ما الخدمات التي يوفرها بريد المغرب عبر بنك البريد؟",
        "darija_query": "أش هي الخدمات لي كيوفرها بنك البريد؟",
        "gold_answer": "خدمات التحويل المالي الداخلي والدولي، إلى جانب الادخار والحسابات البنكية.",
        "source_chunk_id": "chunk_039",
        "source_chunk_text": "يوفر بريد المغرب عبر بنك البريد خدمات التحويل المالي الداخلي والدولي، ويستعملها كثير من المغاربة المقيمين بالخارج لإرسال الأموال لعائلاتهم، إلى جانب خدمات الادخار والحسابات البنكية الأساسية."
    },
    {
        "id": "b081",
        "msa_query": "من يستعمل خدمة التحويل المالي عبر بنك البريد بكثرة؟",
        "darija_query": "شكون لي كيستعمل بزاف خدمة التحويل ديال بنك البريد؟",
        "gold_answer": "يستعملها كثير من المغاربة المقيمين بالخارج لإرسال الأموال لعائلاتهم.",
        "source_chunk_id": "chunk_039",
        "source_chunk_text": "يوفر بريد المغرب عبر بنك البريد خدمات التحويل المالي الداخلي والدولي، ويستعملها كثير من المغاربة المقيمين بالخارج لإرسال الأموال لعائلاتهم، إلى جانب خدمات الادخار والحسابات البنكية الأساسية."
    },
    {
        "id": "b082",
        "msa_query": "ماذا تميز عروض الأنترنت المنزلي عبر الألياف الضوئية؟",
        "darija_query": "أش كيميز عروض الأنترنت ديال الفيبر؟",
        "gold_answer": "أثمنة تنافسية وسرعات أعلى مقارنة بالتقنيات القديمة.",
        "source_chunk_id": "chunk_040",
        "source_chunk_text": "توسعت في السنوات الأخيرة عروض الأنترنت المنزلي عبر الألياف الضوئية في المدن المغربية الكبرى، بأثمنة تنافسية وسرعات أعلى مقارنة بالتقنيات القديمة، ما ساهم في تحسين جودة الاتصال داخل المنازل."
    },
    {
        "id": "b083",
        "msa_query": "ماذا حسّنت عروض الفيبر الضوئي في المنازل؟",
        "darija_query": "أش حسنات عروض الفيبر فالدور؟",
        "gold_answer": "ساهمت في تحسين جودة الاتصال داخل المنازل.",
        "source_chunk_id": "chunk_040",
        "source_chunk_text": "توسعت في السنوات الأخيرة عروض الأنترنت المنزلي عبر الألياف الضوئية في المدن المغربية الكبرى، بأثمنة تنافسية وسرعات أعلى مقارنة بالتقنيات القديمة، ما ساهم في تحسين جودة الاتصال داخل المنازل."
    },
    {
        "id": "b084",
        "msa_query": "هل يمكن الشراء بالدين في الحانوت الصغير؟",
        "darija_query": "واش ممكن نشري بالدين فالحانوت؟",
        "gold_answer": "نعم، الحانوت الصغير يتيح إمكانية الشراء بالدين.",
        "source_chunk_id": "chunk_036",
        "source_chunk_text": "يتوجه عدد متزايد من الأسر المغربية للتسوق الأسبوعي في المتاجر الكبرى مثل مرجان وكارفور نظراً لتنوع المنتجات وثبات الأثمنة، في حين يفضل آخرون الحانوت الصغير القريب من المنزل للتسوق اليومي ولإمكانية الشراء بالدين."
    },
    {
        "id": "b085",
        "msa_query": "هل يشتغل الطاكسي الصغير خارج المدينة؟",
        "darija_query": "واش الطاكسي الصغير كيخدم برا المدينة؟",
        "gold_answer": "لا، الطاكسي الصغير يعمل داخل المدينة فقط.",
        "source_chunk_id": "chunk_027",
        "source_chunk_text": "تنقسم سيارات الأجرة في المغرب إلى الطاكسي الكبير الذي ينقل الركاب بين المدن بثمن يحدده كل راكب حسب المسافة، والطاكسي الصغير الذي يعمل داخل المدينة ويشتغل بالعداد أو بثمن متفق عليه مسبقاً."
    },
    {
        "id": "b086",
        "msa_query": "هل تشهد العروض الترويجية لشركات الاتصالات تكراراً؟",
        "darija_query": "واش العروض ديال شركات الاتصالات كتتكرر بزاف؟",
        "gold_answer": "نعم، تشهد المنافسة بين الشركات عروضاً ترويجية متكررة.",
        "source_chunk_id": "chunk_029",
        "source_chunk_text": "تقدم شركات الاتصالات الثلاث الكبرى في المغرب، اتصالات المغرب وأورونج وإنوي، عروضاً متنوعة لبطاقات الأنترنت المسبقة الأداء، تتراوح أثمنتها حسب حجم الرصيد ومدة الصلاحية، وتشهد المنافسة بينها عروضاً ترويجية متكررة."
    },
    {
        "id": "b087",
        "msa_query": "هل تختلف أثمنة دخول الحمام حسب المدينة؟",
        "darija_query": "واش ثمن الحمام كيختلف من مدينة لأخرى؟",
        "gold_answer": "نعم، يختلف الثمن حسب المدينة والخدمات الإضافية.",
        "source_chunk_id": "chunk_034",
        "source_chunk_text": "يبقى الحمام العمومي التقليدي جزءاً من العادات اليومية أو الأسبوعية لدى كثير من المغاربة، حيث يذهب الناس للاغتسال والتدليك التقليدي، ويتراوح ثمن الدخول عادة بين 10 و20 درهماً حسب المدينة والخدمات الإضافية."
    },
    {
        "id": "b088",
        "msa_query": "هل الأنترنت عالي الصبيب متوفر في المناطق القروية؟",
        "darija_query": "واش الأنترنت الفيبر كاين فالبوادي؟",
        "gold_answer": "برنامج المغرب الرقمي يعمل على توسيع الولوج إلى الإنترنت عالي الصبيب في المناطق القروية.",
        "source_chunk_id": "chunk_018",
        "source_chunk_text": "أطلقت الحكومة المغربية برنامج المغرب الرقمي بهدف تسريع التحول الرقمي للإدارة العمومية، ويشمل البرنامج رقمنة الخدمات الإدارية وتوسيع الولوج إلى الإنترنت عالي الصبيب في المناطق القروية."
    },
    {
        "id": "b089",
        "msa_query": "هل يمكن السفر بين المدن بالطاكسي الكبير؟",
        "darija_query": "واش ممكن نسافر بين المدن بالطاكسي الكبير؟",
        "gold_answer": "نعم، الطاكسي الكبير ينقل الركاب بين المدن.",
        "source_chunk_id": "chunk_027",
        "source_chunk_text": "تنقسم سيارات الأجرة في المغرب إلى الطاكسي الكبير الذي ينقل الركاب بين المدن بثمن يحدده كل راكب حسب المسافة، والطاكسي الصغير الذي يعمل داخل المدينة ويشتغل بالعداد أو بثمن متفق عليه مسبقاً."
    },
    {
        "id": "b090",
        "msa_query": "ما هو أتاي؟",
        "darija_query": "شنو هو أتاي؟",
        "gold_answer": "أتاي هو الشاي بالنعناع الذي يُحتسى عادة في المقهى أو مع الأصدقاء.",
        "source_chunk_id": "chunk_031",
        "source_chunk_text": "يعتبر المقهى فضاءً اجتماعياً أساسياً في الحياة اليومية المغربية، حيث يلتقي الأصدقاء لاحتساء أتاي (الشاي بالنعناع) أو القهوة ومتابعة أخبار الرياضة، وتنتشر المقاهي بكثرة في جميع الأحياء والمدن."
    },
    {
        "id": "b091",
        "msa_query": "هل تحدد الدولة ثمن غاز البوطان بشكل حر؟",
        "darija_query": "واش ثمن الغاز حر ولا مدعوم؟",
        "gold_answer": "لا، يبقى ثمنه مدعماً من طرف الدولة.",
        "source_chunk_id": "chunk_038",
        "source_chunk_text": "يعتمد كثير من المنازل المغربية على قنينة غاز البوطان للطبخ والتسخين، ويوزعها الباعة المتجولون بالعربات أو الدراجات في الأحياء يومياً، ويبقى ثمنها مدعماً من طرف الدولة لضمان وصولها لجميع الفئات."
    },
    {
        "id": "b092",
        "msa_query": "ما هي مدة صلاحية بطاقات الأنترنت المسبقة الأداء؟",
        "darija_query": "شحال كتدوم صلاحية الكارطة ديال الأنترنت؟",
        "gold_answer": "تختلف مدة الصلاحية حسب العرض وحجم الرصيد المشترى.",
        "source_chunk_id": "chunk_029",
        "source_chunk_text": "تقدم شركات الاتصالات الثلاث الكبرى في المغرب، اتصالات المغرب وأورونج وإنوي، عروضاً متنوعة لبطاقات الأنترنت المسبقة الأداء، تتراوح أثمنتها حسب حجم الرصيد ومدة الصلاحية، وتشهد المنافسة بينها عروضاً ترويجية متكررة."
    },
    {
        "id": "b093",
        "msa_query": "كم عدد شركات الاتصالات الكبرى في المغرب؟",
        "darija_query": "شحال ديال الشركات الكبار ديال الاتصالات فالمغرب؟",
        "gold_answer": "توجد ثلاث شركات كبرى للاتصالات في المغرب.",
        "source_chunk_id": "chunk_029",
        "source_chunk_text": "تقدم شركات الاتصالات الثلاث الكبرى في المغرب، اتصالات المغرب وأورونج وإنوي، عروضاً متنوعة لبطاقات الأنترنت المسبقة الأداء، تتراوح أثمنتها حسب حجم الرصيد ومدة الصلاحية، وتشهد المنافسة بينها عروضاً ترويجية متكررة."
    },
    {
        "id": "b094",
        "msa_query": "أين يتم توقيع عقد الكراء عادة؟",
        "darija_query": "فين كيتوقع عقد الكراء؟",
        "gold_answer": "يوقَّع عقد الكراء بين المكري والمكتري لتحديد شروط السكن.",
        "source_chunk_id": "chunk_035",
        "source_chunk_text": "عند كراء شقة سكنية في المغرب، يطلب المكري عادة إيداع وجيبة تعادل شهراً أو شهرين من الكراء كضمانة، بالإضافة إلى توقيع عقد كراء يحدد مدة العقد وثمن الكراء الشهري وشروط الإخلاء."
    },
    {
        "id": "b095",
        "msa_query": "هل يعمل الطرامواي طيلة أيام الأسبوع؟",
        "darija_query": "واش الطرامواي كيخدم طول جيمع أيام السيمانة؟",
        "gold_answer": "نعم، يشتغل الطرامواي يومياً من الساعة السادسة صباحاً إلى منتصف الليل.",
        "source_chunk_id": "chunk_028",
        "source_chunk_text": "يشتغل الطرامواي في الدار البيضاء والرباط يومياً من الساعة السادسة صباحاً إلى منتصف الليل تقريباً، وتتراوح تذكرة الرحلة الواحدة حوالي 7 دراهم، مع وجود اشتراكات شهرية بثمن مخفض للمستعملين المنتظمين."
    },
    {
        "id": "b096",
        "msa_query": "ما نوع الخدمة البنكية الأساسية التي يوفرها بنك البريد؟",
        "darija_query": "شنو هي الخدمة البنكية الأساسية لي كيوفرها بنك البريد؟",
        "gold_answer": "يوفر خدمات الادخار والحسابات البنكية الأساسية.",
        "source_chunk_id": "chunk_039",
        "source_chunk_text": "يوفر بريد المغرب عبر بنك البريد خدمات التحويل المالي الداخلي والدولي، ويستعملها كثير من المغاربة المقيمين بالخارج لإرسال الأموال لعائلاتهم، إلى جانب خدمات الادخار والحسابات البنكية الأساسية."
    },
    {
        "id": "b097",
        "msa_query": "هل ترتفع أثمنة الأغنام في كل السنة أم في فترة محددة؟",
        "darija_query": "واش ثمن الغنم كيغلى فالعام كامل ولا فوقت معين؟",
        "gold_answer": "ترتفع بشكل خاص في الأسابيع التي تسبق عيد الأضحى.",
        "source_chunk_id": "chunk_037",
        "source_chunk_text": "يشهد سوق الأضاحي إقبالاً كبيراً في الأسابيع التي تسبق عيد الأضحى، حيث ترتفع أثمنة الأغنام بشكل ملحوظ، وتحدد وزارة الفلاحة كل سنة توصيات لتنظيم أسواق بيع المواشي وضمان جودتها."
    },
    {
        "id": "b098",
        "msa_query": "هل يشتري المغاربة الخبز يومياً من المخبزة؟",
        "darija_query": "واش المغاربة كيشريو الخبز كل نهار من الفرن؟",
        "gold_answer": "نعم، يخضع ثمن الخبز التقليدي للمخابز لرقابة نسبية لضمان توفره اليومي بثمن معقول.",
        "source_chunk_id": "chunk_030",
        "source_chunk_text": "يخضع ثمن الخبز التقليدي في المخابز لرقابة نسبية من طرف السلطات المحلية للحفاظ على ثمن معقول يتناسب مع القدرة الشرائية للمواطنين، ويتراوح ثمن الخبز العادي بين 1.2 و1.5 درهم للقطعة في أغلب المناطق."
    },
    {
        "id": "b099",
        "msa_query": "أين تُباع الخضر والفواكه الطازجة عادة في الأحياء الشعبية؟",
        "darija_query": "فين كيتباعو الخضرة والفواكه الطرية فالحومة؟",
        "gold_answer": "تُباع عادة في السوق الأسبوعي أو في الحانوت الصغير القريب من المنزل.",
        "source_chunk_id": "chunk_026",
        "source_chunk_text": "تُقام الأسواق الأسبوعية (السوق) في أغلب المدن والقرى المغربية في يوم محدد من الأسبوع، حيث يعرض الفلاحون والباعة منتجاتهم من خضر وفواكه ومواشي مباشرة للمستهلكين، وتبقى المساومة على الأثمنة عادة شائعة هناك."
    },
    {
        "id": "b100",
        "msa_query": "هل يفضل معظم الركاب الطاكسي الصغير أم الطرامواي داخل المدينة؟",
        "darija_query": "واش أغلب الناس كيفضلو الطاكسي الصغير ولا الطرامواي فالمدينة؟",
        "gold_answer": "يختلف حسب المسافة والثمن، فالطرامواي أرخص عادة على المسافات الطويلة داخل المدينة.",
        "source_chunk_id": "chunk_028",
        "source_chunk_text": "يشتغل الطرامواي في الدار البيضاء والرباط يومياً من الساعة السادسة صباحاً إلى منتصف الليل تقريباً، وتتراوح تذكرة الرحلة الواحدة حوالي 7 دراهم، مع وجود اشتراكات شهرية بثمن مخفض للمستعملين المنتظمين."
    },
    {
        "id": "b101",
        "msa_query": "عن ماذا يمتنع الصائمون خلال رمضان؟",
        "darija_query": "أش كيتحاشاو الصايمين فرمضان؟",
        "gold_answer": "يمتنعون عن الأكل والشرب من أذان الفجر إلى أذان المغرب.",
        "source_chunk_id": "chunk_041",
        "source_chunk_text": "يمتنع الصائمون خلال شهر رمضان عن الأكل والشرب من أذان الفجر إلى أذان المغرب، وتجتمع العائلات المغربية عادة على مائدة الإفطار حول أطباق تقليدية مثل الحريرة والشباكية والتمر، قبل التوجه لاحقاً لصلاة التراويح."
    },
    {
        "id": "b102",
        "msa_query": "ما هي الأطباق التقليدية على مائدة الإفطار؟",
        "darija_query": "شنو هوما الأكلات التقليدية ديال الفطور؟",
        "gold_answer": "الحريرة والشباكية والتمر من أبرز الأطباق التقليدية.",
        "source_chunk_id": "chunk_041",
        "source_chunk_text": "يمتنع الصائمون خلال شهر رمضان عن الأكل والشرب من أذان الفجر إلى أذان المغرب، وتجتمع العائلات المغربية عادة على مائدة الإفطار حول أطباق تقليدية مثل الحريرة والشباكية والتمر، قبل التوجه لاحقاً لصلاة التراويح."
    },
    {
        "id": "b103",
        "msa_query": "ماذا يفعل المصلون بعد الإفطار؟",
        "darija_query": "أش كيديرو الناس من بعد الفطور؟",
        "gold_answer": "يتوجهون لاحقاً لصلاة التراويح.",
        "source_chunk_id": "chunk_041",
        "source_chunk_text": "يمتنع الصائمون خلال شهر رمضان عن الأكل والشرب من أذان الفجر إلى أذان المغرب، وتجتمع العائلات المغربية عادة على مائدة الإفطار حول أطباق تقليدية مثل الحريرة والشباكية والتمر، قبل التوجه لاحقاً لصلاة التراويح."
    },
    {
        "id": "b104",
        "msa_query": "متى يبدأ الامتناع عن الأكل في رمضان؟",
        "darija_query": "إمتى كيبدا الصيام فرمضان؟",
        "gold_answer": "يبدأ من أذان الفجر.",
        "source_chunk_id": "chunk_041",
        "source_chunk_text": "يمتنع الصائمون خلال شهر رمضان عن الأكل والشرب من أذان الفجر إلى أذان المغرب، وتجتمع العائلات المغربية عادة على مائدة الإفطار حول أطباق تقليدية مثل الحريرة والشباكية والتمر، قبل التوجه لاحقاً لصلاة التراويح."
    },
    {
        "id": "b105",
        "msa_query": "هل تجتمع العائلات المغربية على الإفطار؟",
        "darija_query": "واش العائلات كتجتمع على الفطور؟",
        "gold_answer": "نعم، تجتمع العائلات المغربية عادة على مائدة الإفطار.",
        "source_chunk_id": "chunk_041",
        "source_chunk_text": "يمتنع الصائمون خلال شهر رمضان عن الأكل والشرب من أذان الفجر إلى أذان المغرب، وتجتمع العائلات المغربية عادة على مائدة الإفطار حول أطباق تقليدية مثل الحريرة والشباكية والتمر، قبل التوجه لاحقاً لصلاة التراويح."
    },
    {
        "id": "b106",
        "msa_query": "ما هي ليلة الحناء؟",
        "darija_query": "شنو هي ليلة الحنة؟",
        "gold_answer": "هي ليلة تُزين فيها العروس بالحناء ضمن مراسيم العرس التقليدي.",
        "source_chunk_id": "chunk_042",
        "source_chunk_text": "يتضمن العرس المغربي التقليدي عدة مراسيم تمتد أحياناً على أكثر من يوم، من أبرزها ليلة الحناء التي تُزين فيها العروس بالحناء، إضافة إلى تحضير جهاز العروس الذي يضم الأثاث والملابس التقليدية."
    },
    {
        "id": "b107",
        "msa_query": "ماذا يضم جهاز العروس عادة؟",
        "darija_query": "أش كيضم الجهاز ديال العروسة؟",
        "gold_answer": "يضم الأثاث والملابس التقليدية.",
        "source_chunk_id": "chunk_042",
        "source_chunk_text": "يتضمن العرس المغربي التقليدي عدة مراسيم تمتد أحياناً على أكثر من يوم، من أبرزها ليلة الحناء التي تُزين فيها العروس بالحناء، إضافة إلى تحضير جهاز العروس الذي يضم الأثاث والملابس التقليدية."
    },
    {
        "id": "b108",
        "msa_query": "هل يمتد العرس المغربي التقليدي ليوم واحد فقط؟",
        "darija_query": "واش العرس المغربي كيدوم غير نهار وحد؟",
        "gold_answer": "لا، يمتد أحياناً على أكثر من يوم.",
        "source_chunk_id": "chunk_042",
        "source_chunk_text": "يتضمن العرس المغربي التقليدي عدة مراسيم تمتد أحياناً على أكثر من يوم، من أبرزها ليلة الحناء التي تُزين فيها العروس بالحناء، إضافة إلى تحضير جهاز العروس الذي يضم الأثاث والملابس التقليدية."
    },
    {
        "id": "b109",
        "msa_query": "ماذا تتضمن مراسيم العرس المغربي التقليدي؟",
        "darija_query": "أش كتضم المراسيم ديال العرس المغربي؟",
        "gold_answer": "تتضمن عدة مراسيم من أبرزها ليلة الحناء وتحضير جهاز العروس.",
        "source_chunk_id": "chunk_042",
        "source_chunk_text": "يتضمن العرس المغربي التقليدي عدة مراسيم تمتد أحياناً على أكثر من يوم، من أبرزها ليلة الحناء التي تُزين فيها العروس بالحناء، إضافة إلى تحضير جهاز العروس الذي يضم الأثاث والملابس التقليدية."
    },
    {
        "id": "b110",
        "msa_query": "من تُزين بالحناء في ليلة الحناء؟",
        "darija_query": "شكون لي كتتزين بالحنة فليلة الحنة؟",
        "gold_answer": "العروس هي من تُزين بالحناء.",
        "source_chunk_id": "chunk_042",
        "source_chunk_text": "يتضمن العرس المغربي التقليدي عدة مراسيم تمتد أحياناً على أكثر من يوم، من أبرزها ليلة الحناء التي تُزين فيها العروس بالحناء، إضافة إلى تحضير جهاز العروس الذي يضم الأثاث والملابس التقليدية."
    },
    {
        "id": "b111",
        "msa_query": "ما هي خطوات الحصول على رخصة السياقة بالمغرب؟",
        "darija_query": "شنو هوما الخطوات باش ناخد رخصة السياقة؟",
        "gold_answer": "يجب اجتياز مباراة الكود النظرية ثم الامتحان التطبيقي.",
        "source_chunk_id": "chunk_043",
        "source_chunk_text": "للحصول على رخصة السياقة بالمغرب، يجب اجتياز مباراة الكود النظرية ثم الامتحان التطبيقي داخل مدرسة تعليم السياقة (أوطو إيكول)، وتختلف مدة التكوين حسب الصنف المطلوب من الرخصة."
    },
    {
        "id": "b112",
        "msa_query": "أين يتم التكوين على السياقة؟",
        "darija_query": "فين كيتم التكوين ديال السياقة؟",
        "gold_answer": "داخل مدرسة تعليم السياقة (أوطو إيكول).",
        "source_chunk_id": "chunk_043",
        "source_chunk_text": "للحصول على رخصة السياقة بالمغرب، يجب اجتياز مباراة الكود النظرية ثم الامتحان التطبيقي داخل مدرسة تعليم السياقة (أوطو إيكول)، وتختلف مدة التكوين حسب الصنف المطلوب من الرخصة."
    },
    {
        "id": "b113",
        "msa_query": "هل مدة التكوين على السياقة ثابتة لجميع الأصناف؟",
        "darija_query": "واش مدة التكوين كتبقى بحال بحال لجميع الأصناف؟",
        "gold_answer": "لا، تختلف مدة التكوين حسب الصنف المطلوب من الرخصة.",
        "source_chunk_id": "chunk_043",
        "source_chunk_text": "للحصول على رخصة السياقة بالمغرب، يجب اجتياز مباراة الكود النظرية ثم الامتحان التطبيقي داخل مدرسة تعليم السياقة (أوطو إيكول)، وتختلف مدة التكوين حسب الصنف المطلوب من الرخصة."
    },
    {
        "id": "b114",
        "msa_query": "ما هو الامتحان الذي يسبق الامتحان التطبيقي للسياقة؟",
        "darija_query": "شنو هو الامتحان لي كيسبق الامتحان التطبيقي؟",
        "gold_answer": "مباراة الكود النظرية.",
        "source_chunk_id": "chunk_043",
        "source_chunk_text": "للحصول على رخصة السياقة بالمغرب، يجب اجتياز مباراة الكود النظرية ثم الامتحان التطبيقي داخل مدرسة تعليم السياقة (أوطو إيكول)، وتختلف مدة التكوين حسب الصنف المطلوب من الرخصة."
    },
    {
        "id": "b115",
        "msa_query": "ماذا يختصر اسم مدرسة تعليم السياقة بالدارجة؟",
        "darija_query": "كيفاش كيسميو مدرسة تعليم السياقة بالدارجة؟",
        "gold_answer": "تُسمى أوطو إيكول.",
        "source_chunk_id": "chunk_043",
        "source_chunk_text": "للحصول على رخصة السياقة بالمغرب، يجب اجتياز مباراة الكود النظرية ثم الامتحان التطبيقي داخل مدرسة تعليم السياقة (أوطو إيكول)، وتختلف مدة التكوين حسب الصنف المطلوب من الرخصة."
    },
    {
        "id": "b116",
        "msa_query": "كل كم سنة يجب تجديد البطاقة الوطنية للتعريف؟",
        "darija_query": "شحال ديال السنين خاصك تجدد السيني؟",
        "gold_answer": "يجب تجديدها كل عشر سنوات.",
        "source_chunk_id": "chunk_044",
        "source_chunk_text": "تُعتبر البطاقة الوطنية للتعريف الإلكترونية (السيني) وثيقة إلزامية لكل مواطن مغربي بالغ سن الثامنة عشرة، ويجب تجديدها كل عشر سنوات عبر مصالح الأمن الوطني أو السلطة المحلية."
    },
    {
        "id": "b117",
        "msa_query": "من يجب عليه امتلاك بطاقة التعريف الوطنية؟",
        "darija_query": "شكون خاصو يكون عندو السيني؟",
        "gold_answer": "كل مواطن مغربي بالغ سن الثامنة عشرة.",
        "source_chunk_id": "chunk_044",
        "source_chunk_text": "تُعتبر البطاقة الوطنية للتعريف الإلكترونية (السيني) وثيقة إلزامية لكل مواطن مغربي بالغ سن الثامنة عشرة، ويجب تجديدها كل عشر سنوات عبر مصالح الأمن الوطني أو السلطة المحلية."
    },
    {
        "id": "b118",
        "msa_query": "أين يتم تجديد البطاقة الوطنية للتعريف؟",
        "darija_query": "فين كتجدد السيني؟",
        "gold_answer": "عبر مصالح الأمن الوطني أو السلطة المحلية.",
        "source_chunk_id": "chunk_044",
        "source_chunk_text": "تُعتبر البطاقة الوطنية للتعريف الإلكترونية (السيني) وثيقة إلزامية لكل مواطن مغربي بالغ سن الثامنة عشرة، ويجب تجديدها كل عشر سنوات عبر مصالح الأمن الوطني أو السلطة المحلية."
    },
    {
        "id": "b119",
        "msa_query": "هل البطاقة الوطنية للتعريف إلزامية؟",
        "darija_query": "واش السيني إجبارية؟",
        "gold_answer": "نعم، تُعتبر وثيقة إلزامية لكل مواطن بالغ.",
        "source_chunk_id": "chunk_044",
        "source_chunk_text": "تُعتبر البطاقة الوطنية للتعريف الإلكترونية (السيني) وثيقة إلزامية لكل مواطن مغربي بالغ سن الثامنة عشرة، ويجب تجديدها كل عشر سنوات عبر مصالح الأمن الوطني أو السلطة المحلية."
    },
    {
        "id": "b120",
        "msa_query": "بماذا تُعرف البطاقة الوطنية للتعريف الإلكترونية اختصاراً؟",
        "darija_query": "كيفاش كتسمى البطاقة الوطنية بالاختصار؟",
        "gold_answer": "تُعرف اختصاراً بالسيني.",
        "source_chunk_id": "chunk_044",
        "source_chunk_text": "تُعتبر البطاقة الوطنية للتعريف الإلكترونية (السيني) وثيقة إلزامية لكل مواطن مغربي بالغ سن الثامنة عشرة، ويجب تجديدها كل عشر سنوات عبر مصالح الأمن الوطني أو السلطة المحلية."
    },
    {
        "id": "b121",
        "msa_query": "ماذا يتضمن ملف طلب جواز السفر؟",
        "darija_query": "أش كيضم ملف طلب الباسبور؟",
        "gold_answer": "يتضمن البطاقة الوطنية وشهادة السكنى وصوراً شمسية حديثة.",
        "source_chunk_id": "chunk_045",
        "source_chunk_text": "يتقدم طالب جواز السفر المغربي بملف يتضمن البطاقة الوطنية وشهادة السكنى وصوراً شمسية حديثة إلى مصلحة الجوازات، وتستغرق عملية الاستخراج عادة بضعة أسابيع حسب الجهة."
    },
    {
        "id": "b122",
        "msa_query": "كم من الوقت تستغرق عملية استخراج جواز السفر؟",
        "darija_query": "شحال ديال الوقت كتاخد عملية استخراج الباسبور؟",
        "gold_answer": "تستغرق عادة بضعة أسابيع حسب الجهة.",
        "source_chunk_id": "chunk_045",
        "source_chunk_text": "يتقدم طالب جواز السفر المغربي بملف يتضمن البطاقة الوطنية وشهادة السكنى وصوراً شمسية حديثة إلى مصلحة الجوازات، وتستغرق عملية الاستخراج عادة بضعة أسابيع حسب الجهة."
    },
    {
        "id": "b123",
        "msa_query": "إلى أي مصلحة يتقدم طالب جواز السفر؟",
        "darija_query": "فين كيمشي طالب الباسبور باش يقدم الملف؟",
        "gold_answer": "يتقدم إلى مصلحة الجوازات.",
        "source_chunk_id": "chunk_045",
        "source_chunk_text": "يتقدم طالب جواز السفر المغربي بملف يتضمن البطاقة الوطنية وشهادة السكنى وصوراً شمسية حديثة إلى مصلحة الجوازات، وتستغرق عملية الاستخراج عادة بضعة أسابيع حسب الجهة."
    },
    {
        "id": "b124",
        "msa_query": "هل تختلف مدة استخراج جواز السفر باختلاف الجهة؟",
        "darija_query": "واش المدة ديال الباسبور كتختلف من جهة لجهة؟",
        "gold_answer": "نعم، تستغرق العملية بضعة أسابيع حسب الجهة.",
        "source_chunk_id": "chunk_045",
        "source_chunk_text": "يتقدم طالب جواز السفر المغربي بملف يتضمن البطاقة الوطنية وشهادة السكنى وصوراً شمسية حديثة إلى مصلحة الجوازات، وتستغرق عملية الاستخراج عادة بضعة أسابيع حسب الجهة."
    },
    {
        "id": "b125",
        "msa_query": "ما نوع الصور المطلوبة في ملف جواز السفر؟",
        "darija_query": "أش نوع التصاور لي مطلوبة فملف الباسبور؟",
        "gold_answer": "صور شمسية حديثة.",
        "source_chunk_id": "chunk_045",
        "source_chunk_text": "يتقدم طالب جواز السفر المغربي بملف يتضمن البطاقة الوطنية وشهادة السكنى وصوراً شمسية حديثة إلى مصلحة الجوازات، وتستغرق عملية الاستخراج عادة بضعة أسابيع حسب الجهة."
    },
    {
        "id": "b126",
        "msa_query": "ما هو الامتحان الوطني الموحد لتلاميذ الثانية باكالوريا؟",
        "darija_query": "شنو هو الامتحان الوطني ديال تلاميذ الثانية باك؟",
        "gold_answer": "يُعرف بالباكالوريا.",
        "source_chunk_id": "chunk_046",
        "source_chunk_text": "يجتاز تلاميذ الثانية باكالوريا في المغرب امتحاناً وطنياً موحداً يُعرف بالباكالوريا، يحدد نجاحهم فيه إمكانية الولوج للتعليم العالي، وتشمل المواد الأساسية الرياضيات والفيزياء واللغات حسب الشعبة المختارة."
    },
    {
        "id": "b127",
        "msa_query": "ماذا يحدد نجاح التلميذ في الباكالوريا؟",
        "darija_query": "أش كيحدد النجاح فالباكالوريا؟",
        "gold_answer": "يحدد إمكانية الولوج للتعليم العالي.",
        "source_chunk_id": "chunk_046",
        "source_chunk_text": "يجتاز تلاميذ الثانية باكالوريا في المغرب امتحاناً وطنياً موحداً يُعرف بالباكالوريا، يحدد نجاحهم فيه إمكانية الولوج للتعليم العالي، وتشمل المواد الأساسية الرياضيات والفيزياء واللغات حسب الشعبة المختارة."
    },
    {
        "id": "b128",
        "msa_query": "ما هي المواد الأساسية في امتحان الباكالوريا؟",
        "darija_query": "شنو هوما المواد الأساسية فامتحان الباكالوريا؟",
        "gold_answer": "الرياضيات والفيزياء واللغات حسب الشعبة المختارة.",
        "source_chunk_id": "chunk_046",
        "source_chunk_text": "يجتاز تلاميذ الثانية باكالوريا في المغرب امتحاناً وطنياً موحداً يُعرف بالباكالوريا، يحدد نجاحهم فيه إمكانية الولوج للتعليم العالي، وتشمل المواد الأساسية الرياضيات والفيزياء واللغات حسب الشعبة المختارة."
    },
    {
        "id": "b129",
        "msa_query": "هل تختلف مواد الباكالوريا حسب الشعبة؟",
        "darija_query": "واش المواد كتختلف حسب الشعبة؟",
        "gold_answer": "نعم، تشمل المواد الأساسية حسب الشعبة المختارة.",
        "source_chunk_id": "chunk_046",
        "source_chunk_text": "يجتاز تلاميذ الثانية باكالوريا في المغرب امتحاناً وطنياً موحداً يُعرف بالباكالوريا، يحدد نجاحهم فيه إمكانية الولوج للتعليم العالي، وتشمل المواد الأساسية الرياضيات والفيزياء واللغات حسب الشعبة المختارة."
    },
    {
        "id": "b130",
        "msa_query": "من يجتاز امتحان الباكالوريا؟",
        "darija_query": "شكون لي كيجتاز امتحان الباكالوريا؟",
        "gold_answer": "تلاميذ الثانية باكالوريا.",
        "source_chunk_id": "chunk_046",
        "source_chunk_text": "يجتاز تلاميذ الثانية باكالوريا في المغرب امتحاناً وطنياً موحداً يُعرف بالباكالوريا، يحدد نجاحهم فيه إمكانية الولوج للتعليم العالي، وتشمل المواد الأساسية الرياضيات والفيزياء واللغات حسب الشعبة المختارة."
    },
    {
        "id": "b131",
        "msa_query": "ما هي أبرز شبكات التواصل الاجتماعي المستعملة في المغرب؟",
        "darija_query": "شنو هوما أبرز الشبكات الاجتماعية المستعملة فالمغرب؟",
        "gold_answer": "فيسبوك وإنستغرام وتيك توك من أبرزها.",
        "source_chunk_id": "chunk_047",
        "source_chunk_text": "شهد استعمال شبكات التواصل الاجتماعي في المغرب، خصوصاً فيسبوك وإنستغرام وتيك توك، نمواً كبيراً في السنوات الأخيرة، حيث أصبحت هذه المنصات وسيلة أساسية للتواصل والترويج التجاري لدى الشباب."
    },
    {
        "id": "b132",
        "msa_query": "لماذا أصبحت شبكات التواصل الاجتماعي مهمة لدى الشباب؟",
        "darija_query": "علاش أصبحات الشبكات الاجتماعية مهمة عند الشباب؟",
        "gold_answer": "لأنها وسيلة أساسية للتواصل والترويج التجاري.",
        "source_chunk_id": "chunk_047",
        "source_chunk_text": "شهد استعمال شبكات التواصل الاجتماعي في المغرب، خصوصاً فيسبوك وإنستغرام وتيك توك، نمواً كبيراً في السنوات الأخيرة، حيث أصبحت هذه المنصات وسيلة أساسية للتواصل والترويج التجاري لدى الشباب."
    },
    {
        "id": "b133",
        "msa_query": "هل شهد استعمال شبكات التواصل نمواً في المغرب؟",
        "darija_query": "واش استعمال الشبكات الاجتماعية زاد فالمغرب؟",
        "gold_answer": "نعم، شهد نمواً كبيراً في السنوات الأخيرة.",
        "source_chunk_id": "chunk_047",
        "source_chunk_text": "شهد استعمال شبكات التواصل الاجتماعي في المغرب، خصوصاً فيسبوك وإنستغرام وتيك توك، نمواً كبيراً في السنوات الأخيرة، حيث أصبحت هذه المنصات وسيلة أساسية للتواصل والترويج التجاري لدى الشباب."
    },
    {
        "id": "b134",
        "msa_query": "لمن تُستعمل شبكات التواصل في الترويج التجاري؟",
        "darija_query": "لمن كتستعمل الشبكات الاجتماعية فالترويج التجاري؟",
        "gold_answer": "تُستعمل لدى الشباب بشكل خاص للترويج التجاري.",
        "source_chunk_id": "chunk_047",
        "source_chunk_text": "شهد استعمال شبكات التواصل الاجتماعي في المغرب، خصوصاً فيسبوك وإنستغرام وتيك توك، نمواً كبيراً في السنوات الأخيرة، حيث أصبحت هذه المنصات وسيلة أساسية للتواصل والترويج التجاري لدى الشباب."
    },
    {
        "id": "b135",
        "msa_query": "ما هي إحدى المنصات التي شهدت نمواً كبيراً بالمغرب؟",
        "darija_query": "شنو هي وحدة من المنصات لي زادات بزاف فالمغرب؟",
        "gold_answer": "تيك توك من المنصات التي شهدت نمواً كبيراً.",
        "source_chunk_id": "chunk_047",
        "source_chunk_text": "شهد استعمال شبكات التواصل الاجتماعي في المغرب، خصوصاً فيسبوك وإنستغرام وتيك توك، نمواً كبيراً في السنوات الأخيرة، حيث أصبحت هذه المنصات وسيلة أساسية للتواصل والترويج التجاري لدى الشباب."
    },
    {
        "id": "b136",
        "msa_query": "من ماذا تعاني المدن الكبرى بالمغرب في ساعات الذروة؟",
        "darija_query": "أش كتعاني منو المدن الكبار فساعات الذروة؟",
        "gold_answer": "تعاني من ازدحام مروري خانق.",
        "source_chunk_id": "chunk_048",
        "source_chunk_text": "تعاني المدن الكبرى بالمغرب، وخصوصاً الدار البيضاء، من ازدحام مروري خانق في ساعات الذروة، إضافة إلى نقص أماكن التوقف المنظمة، ما دفع بعض المجالس المحلية إلى إحداث مواقف مؤدى عنها لتنظيم الوضعية."
    },
    {
        "id": "b137",
        "msa_query": "ما الذي دفع بعض المجالس المحلية لإحداث مواقف مؤدى عنها؟",
        "darija_query": "أش لي دفع بعض المجالس المحلية يحدثو مواقف مؤدى عليها؟",
        "gold_answer": "نقص أماكن التوقف المنظمة.",
        "source_chunk_id": "chunk_048",
        "source_chunk_text": "تعاني المدن الكبرى بالمغرب، وخصوصاً الدار البيضاء، من ازدحام مروري خانق في ساعات الذروة، إضافة إلى نقص أماكن التوقف المنظمة، ما دفع بعض المجالس المحلية إلى إحداث مواقف مؤدى عنها لتنظيم الوضعية."
    },
    {
        "id": "b138",
        "msa_query": "أي مدينة مغربية تعاني بشكل خاص من الازدحام المروري؟",
        "darija_query": "شنو المدينة لي كتعاني بزاف من الزحمة؟",
        "gold_answer": "الدار البيضاء تعاني بشكل خاص من ازدحام مروري.",
        "source_chunk_id": "chunk_048",
        "source_chunk_text": "تعاني المدن الكبرى بالمغرب، وخصوصاً الدار البيضاء، من ازدحام مروري خانق في ساعات الذروة، إضافة إلى نقص أماكن التوقف المنظمة، ما دفع بعض المجالس المحلية إلى إحداث مواقف مؤدى عنها لتنظيم الوضعية."
    },
    {
        "id": "b139",
        "msa_query": "ماذا أحدثت بعض المجالس المحلية لتنظيم التوقف؟",
        "darija_query": "أش حدثو بعض المجالس المحلية باش ينظمو التوقف؟",
        "gold_answer": "مواقف مؤدى عنها.",
        "source_chunk_id": "chunk_048",
        "source_chunk_text": "تعاني المدن الكبرى بالمغرب، وخصوصاً الدار البيضاء، من ازدحام مروري خانق في ساعات الذروة، إضافة إلى نقص أماكن التوقف المنظمة، ما دفع بعض المجالس المحلية إلى إحداث مواقف مؤدى عنها لتنظيم الوضعية."
    },
    {
        "id": "b140",
        "msa_query": "متى يكون الازدحام المروري في أشده؟",
        "darija_query": "إمتى كتكون الزحمة فأوجها؟",
        "gold_answer": "في ساعات الذروة.",
        "source_chunk_id": "chunk_048",
        "source_chunk_text": "تعاني المدن الكبرى بالمغرب، وخصوصاً الدار البيضاء، من ازدحام مروري خانق في ساعات الذروة، إضافة إلى نقص أماكن التوقف المنظمة، ما دفع بعض المجالس المحلية إلى إحداث مواقف مؤدى عنها لتنظيم الوضعية."
    },
    {
        "id": "b141",
        "msa_query": "من أطلق حملات التوعية للحد من تلوث الهواء؟",
        "darija_query": "شكون لي أطلق الحملات ضد تلوث الهواء؟",
        "gold_answer": "وزارة البيئة أطلقت هذه الحملات.",
        "source_chunk_id": "chunk_049",
        "source_chunk_text": "أطلقت وزارة البيئة حملات توعوية للحد من تلوث الهواء في المدن الكبرى، من خلال تشجيع استعمال وسائل النقل الجماعي والدراجات، ومراقبة انبعاثات المصانع والمركبات القديمة."
    },
    {
        "id": "b142",
        "msa_query": "ماذا تشجع حملات وزارة البيئة؟",
        "darija_query": "أش كتشجع الحملات ديال وزارة البيئة؟",
        "gold_answer": "تشجع استعمال وسائل النقل الجماعي والدراجات.",
        "source_chunk_id": "chunk_049",
        "source_chunk_text": "أطلقت وزارة البيئة حملات توعوية للحد من تلوث الهواء في المدن الكبرى، من خلال تشجيع استعمال وسائل النقل الجماعي والدراجات، ومراقبة انبعاثات المصانع والمركبات القديمة."
    },
    {
        "id": "b143",
        "msa_query": "ماذا تراقب وزارة البيئة إضافة إلى تشجيع النقل الجماعي؟",
        "darija_query": "أش كتراقب وزارة البيئة زيادة على تشجيع النقل الجماعي؟",
        "gold_answer": "تراقب انبعاثات المصانع والمركبات القديمة.",
        "source_chunk_id": "chunk_049",
        "source_chunk_text": "أطلقت وزارة البيئة حملات توعوية للحد من تلوث الهواء في المدن الكبرى، من خلال تشجيع استعمال وسائل النقل الجماعي والدراجات، ومراقبة انبعاثات المصانع والمركبات القديمة."
    },
    {
        "id": "b144",
        "msa_query": "أين تتركز حملات التوعية ضد تلوث الهواء؟",
        "darija_query": "فين كتترکز الحملات ضد تلوث الهواء؟",
        "gold_answer": "تتركز في المدن الكبرى.",
        "source_chunk_id": "chunk_049",
        "source_chunk_text": "أطلقت وزارة البيئة حملات توعوية للحد من تلوث الهواء في المدن الكبرى، من خلال تشجيع استعمال وسائل النقل الجماعي والدراجات، ومراقبة انبعاثات المصانع والمركبات القديمة."
    },
    {
        "id": "b145",
        "msa_query": "ما هدف حملات الحد من تلوث الهواء؟",
        "darija_query": "شنو هو الهدف ديال الحملات ضد تلوث الهواء؟",
        "gold_answer": "الحد من تلوث الهواء في المدن الكبرى.",
        "source_chunk_id": "chunk_049",
        "source_chunk_text": "أطلقت وزارة البيئة حملات توعوية للحد من تلوث الهواء في المدن الكبرى، من خلال تشجيع استعمال وسائل النقل الجماعي والدراجات، ومراقبة انبعاثات المصانع والمركبات القديمة."
    },
    {
        "id": "b146",
        "msa_query": "متى تعرف القنوات التلفزية المغربية إقبالاً كبيراً؟",
        "darija_query": "إمتى كتعرف القنوات المغربية إقبال كبير؟",
        "gold_answer": "تعرف إقبالاً كبيراً خلال شهر رمضان.",
        "source_chunk_id": "chunk_050",
        "source_chunk_text": "تعرف القنوات التلفزية المغربية إقبالاً كبيراً خلال شهر رمضان بفضل بث المسلسلات والبرامج الرمضانية الخاصة، حيث تتنافس القنوات الوطنية على جذب أكبر عدد من المشاهدين بأعمال درامية جديدة كل سنة."
    },
    {
        "id": "b147",
        "msa_query": "على ماذا تتنافس القنوات الوطنية في رمضان؟",
        "darija_query": "على أش كتتنافس القنوات الوطنية فرمضان؟",
        "gold_answer": "تتنافس على جذب أكبر عدد من المشاهدين بأعمال درامية جديدة.",
        "source_chunk_id": "chunk_050",
        "source_chunk_text": "تعرف القنوات التلفزية المغربية إقبالاً كبيراً خلال شهر رمضان بفضل بث المسلسلات والبرامج الرمضانية الخاصة، حيث تتنافس القنوات الوطنية على جذب أكبر عدد من المشاهدين بأعمال درامية جديدة كل سنة."
    },
    {
        "id": "b148",
        "msa_query": "ماذا تبث القنوات المغربية بكثرة في رمضان؟",
        "darija_query": "أش كتبات القنوات المغربية بزاف فرمضان؟",
        "gold_answer": "تبث المسلسلات والبرامج الرمضانية الخاصة.",
        "source_chunk_id": "chunk_050",
        "source_chunk_text": "تعرف القنوات التلفزية المغربية إقبالاً كبيراً خلال شهر رمضان بفضل بث المسلسلات والبرامج الرمضانية الخاصة، حيث تتنافس القنوات الوطنية على جذب أكبر عدد من المشاهدين بأعمال درامية جديدة كل سنة."
    },
    {
        "id": "b149",
        "msa_query": "هل تقدم القنوات أعمالاً درامية جديدة كل سنة؟",
        "darija_query": "واش القنوات كتقدم أعمال جديدة كل عام؟",
        "gold_answer": "نعم، تتنافس على تقديم أعمال درامية جديدة كل سنة.",
        "source_chunk_id": "chunk_050",
        "source_chunk_text": "تعرف القنوات التلفزية المغربية إقبالاً كبيراً خلال شهر رمضان بفضل بث المسلسلات والبرامج الرمضانية الخاصة، حيث تتنافس القنوات الوطنية على جذب أكبر عدد من المشاهدين بأعمال درامية جديدة كل سنة."
    },
    {
        "id": "b150",
        "msa_query": "ما هو السبب الرئيسي لإقبال المشاهدين خلال رمضان؟",
        "darija_query": "شنو السبب الرئيسي لإقبال المشاهدين فرمضان؟",
        "gold_answer": "بث المسلسلات والبرامج الرمضانية الخاصة.",
        "source_chunk_id": "chunk_050",
        "source_chunk_text": "تعرف القنوات التلفزية المغربية إقبالاً كبيراً خلال شهر رمضان بفضل بث المسلسلات والبرامج الرمضانية الخاصة، حيث تتنافس القنوات الوطنية على جذب أكبر عدد من المشاهدين بأعمال درامية جديدة كل سنة."
    },
    {
        "id": "b151",
        "msa_query": "ما هما أشهر طبقين تقليديين مغربيين؟",
        "darija_query": "شنو هوما أشهر طبقين تقليديين مغاربة؟",
        "gold_answer": "الطاجين والكسكس من أشهر الأطباق المغربية.",
        "source_chunk_id": "chunk_051",
        "source_chunk_text": "يعتبر الطاجين والكسكس من أشهر الأطباق المغربية التقليدية، وترتبط عادة وجبة الكسكس بيوم الجمعة حيث تجتمع العائلة على مائدة واحدة بعد صلاة الجمعة."
    },
    {
        "id": "b152",
        "msa_query": "بأي يوم ترتبط وجبة الكسكس عادة؟",
        "darija_query": "بأش نهار كترتبط وجبة الكسكس؟",
        "gold_answer": "ترتبط عادة بيوم الجمعة.",
        "source_chunk_id": "chunk_051",
        "source_chunk_text": "يعتبر الطاجين والكسكس من أشهر الأطباق المغربية التقليدية، وترتبط عادة وجبة الكسكس بيوم الجمعة حيث تجتمع العائلة على مائدة واحدة بعد صلاة الجمعة."
    },
    {
        "id": "b153",
        "msa_query": "متى تجتمع العائلة على مائدة الكسكس؟",
        "darija_query": "إمتى كتجتمع العائلة على الكسكس؟",
        "gold_answer": "بعد صلاة الجمعة.",
        "source_chunk_id": "chunk_051",
        "source_chunk_text": "يعتبر الطاجين والكسكس من أشهر الأطباق المغربية التقليدية، وترتبط عادة وجبة الكسكس بيوم الجمعة حيث تجتمع العائلة على مائدة واحدة بعد صلاة الجمعة."
    },
    {
        "id": "b154",
        "msa_query": "هل يعتبر الطاجين من الأطباق المغربية التقليدية؟",
        "darija_query": "واش الطاجين من الأكلات المغربية التقليدية؟",
        "gold_answer": "نعم، يعتبر من أشهر الأطباق المغربية التقليدية.",
        "source_chunk_id": "chunk_051",
        "source_chunk_text": "يعتبر الطاجين والكسكس من أشهر الأطباق المغربية التقليدية، وترتبط عادة وجبة الكسكس بيوم الجمعة حيث تجتمع العائلة على مائدة واحدة بعد صلاة الجمعة."
    },
    {
        "id": "b155",
        "msa_query": "من يجتمع حول مائدة الكسكس يوم الجمعة؟",
        "darija_query": "شكون كيجتمع على الكسكس نهار الجمعة؟",
        "gold_answer": "العائلة تجتمع على مائدة واحدة.",
        "source_chunk_id": "chunk_051",
        "source_chunk_text": "يعتبر الطاجين والكسكس من أشهر الأطباق المغربية التقليدية، وترتبط عادة وجبة الكسكس بيوم الجمعة حيث تجتمع العائلة على مائدة واحدة بعد صلاة الجمعة."
    },
    {
        "id": "b156",
        "msa_query": "ماذا توفر البنوك المغربية لتشجيع اقتناء السكن؟",
        "darija_query": "أش كتوفر البنوك باش تشجع الناس يشريو دار؟",
        "gold_answer": "توفر قروضاً سكنية بأسعار فائدة تفاضلية.",
        "source_chunk_id": "chunk_052",
        "source_chunk_text": "توفر البنوك المغربية قروضاً سكنية بأسعار فائدة تفاضلية لتشجيع اقتناء السكن، خصوصاً في إطار برامج دعم السكن الاجتماعي، وتتطلب هذه القروض عادة تقديم ملف يثبت الدخل الشهري والضمانات المطلوبة."
    },
    {
        "id": "b157",
        "msa_query": "ماذا يتطلب الحصول على قرض سكني؟",
        "darija_query": "أش خاصك باش تاخد قرض سكني؟",
        "gold_answer": "يتطلب تقديم ملف يثبت الدخل الشهري والضمانات المطلوبة.",
        "source_chunk_id": "chunk_052",
        "source_chunk_text": "توفر البنوك المغربية قروضاً سكنية بأسعار فائدة تفاضلية لتشجيع اقتناء السكن، خصوصاً في إطار برامج دعم السكن الاجتماعي، وتتطلب هذه القروض عادة تقديم ملف يثبت الدخل الشهري والضمانات المطلوبة."
    },
    {
        "id": "b158",
        "msa_query": "في إطار أي برامج تُمنح القروض السكنية التفاضلية؟",
        "darija_query": "فأي برامج كتعطى القروض السكنية التفاضلية؟",
        "gold_answer": "في إطار برامج دعم السكن الاجتماعي.",
        "source_chunk_id": "chunk_052",
        "source_chunk_text": "توفر البنوك المغربية قروضاً سكنية بأسعار فائدة تفاضلية لتشجيع اقتناء السكن، خصوصاً في إطار برامج دعم السكن الاجتماعي، وتتطلب هذه القروض عادة تقديم ملف يثبت الدخل الشهري والضمانات المطلوبة."
    },
    {
        "id": "b159",
        "msa_query": "هل جميع القروض السكنية بنفس سعر الفائدة؟",
        "darija_query": "واش القروض السكنية كامل بنفس سعر الفائدة؟",
        "gold_answer": "لا، توفر البنوك أسعار فائدة تفاضلية لتشجيع السكن.",
        "source_chunk_id": "chunk_052",
        "source_chunk_text": "توفر البنوك المغربية قروضاً سكنية بأسعار فائدة تفاضلية لتشجيع اقتناء السكن، خصوصاً في إطار برامج دعم السكن الاجتماعي، وتتطلب هذه القروض عادة تقديم ملف يثبت الدخل الشهري والضمانات المطلوبة."
    },
    {
        "id": "b160",
        "msa_query": "ما الذي يثبته ملف طلب القرض السكني؟",
        "darija_query": "أش لي كيثبتو ملف طلب القرض السكني؟",
        "gold_answer": "يثبت الدخل الشهري والضمانات المطلوبة.",
        "source_chunk_id": "chunk_052",
        "source_chunk_text": "توفر البنوك المغربية قروضاً سكنية بأسعار فائدة تفاضلية لتشجيع اقتناء السكن، خصوصاً في إطار برامج دعم السكن الاجتماعي، وتتطلب هذه القروض عادة تقديم ملف يثبت الدخل الشهري والضمانات المطلوبة."
    },
    {
        "id": "b161",
        "msa_query": "ماذا يجب فعله عند شراء سيارة بالمغرب؟",
        "darija_query": "أش خاصك تدير منين تشري طوموبيل؟",
        "gold_answer": "يجب تسجيلها باسم المالك الجديد عبر مصلحة البطاقة الرمادية.",
        "source_chunk_id": "chunk_053",
        "source_chunk_text": "عند شراء سيارة بالمغرب، يجب تسجيلها باسم المالك الجديد عبر مصلحة البطاقة الرمادية، وهي الوثيقة الرسمية التي تثبت ملكية السيارة وتحمل معلومات صاحبها ومواصفات المركبة."
    },
    {
        "id": "b162",
        "msa_query": "ما هي الوثيقة التي تثبت ملكية السيارة؟",
        "darija_query": "شنو هي الوثيقة لي كتثبت الملكية ديال الطوموبيل؟",
        "gold_answer": "البطاقة الرمادية.",
        "source_chunk_id": "chunk_053",
        "source_chunk_text": "عند شراء سيارة بالمغرب، يجب تسجيلها باسم المالك الجديد عبر مصلحة البطاقة الرمادية، وهي الوثيقة الرسمية التي تثبت ملكية السيارة وتحمل معلومات صاحبها ومواصفات المركبة."
    },
    {
        "id": "b163",
        "msa_query": "ماذا تحمل البطاقة الرمادية من معلومات؟",
        "darija_query": "أش كتحمل البطاقة الرمادية من المعلومات؟",
        "gold_answer": "تحمل معلومات صاحب السيارة ومواصفات المركبة.",
        "source_chunk_id": "chunk_053",
        "source_chunk_text": "عند شراء سيارة بالمغرب، يجب تسجيلها باسم المالك الجديد عبر مصلحة البطاقة الرمادية، وهي الوثيقة الرسمية التي تثبت ملكية السيارة وتحمل معلومات صاحبها ومواصفات المركبة."
    },
    {
        "id": "b164",
        "msa_query": "عبر أي مصلحة تُسجل السيارة باسم المالك الجديد؟",
        "darija_query": "من أي مصلحة كتسجل الطوموبيل بسمية المالك الجديد؟",
        "gold_answer": "عبر مصلحة البطاقة الرمادية.",
        "source_chunk_id": "chunk_053",
        "source_chunk_text": "عند شراء سيارة بالمغرب، يجب تسجيلها باسم المالك الجديد عبر مصلحة البطاقة الرمادية، وهي الوثيقة الرسمية التي تثبت ملكية السيارة وتحمل معلومات صاحبها ومواصفات المركبة."
    },
    {
        "id": "b165",
        "msa_query": "هل البطاقة الرمادية وثيقة رسمية؟",
        "darija_query": "واش البطاقة الرمادية وثيقة رسمية؟",
        "gold_answer": "نعم، هي الوثيقة الرسمية التي تثبت ملكية السيارة.",
        "source_chunk_id": "chunk_053",
        "source_chunk_text": "عند شراء سيارة بالمغرب، يجب تسجيلها باسم المالك الجديد عبر مصلحة البطاقة الرمادية، وهي الوثيقة الرسمية التي تثبت ملكية السيارة وتحمل معلومات صاحبها ومواصفات المركبة."
    },
    {
        "id": "b166",
        "msa_query": "ماذا تقدم أنابيك للباحثين عن الشغل؟",
        "darija_query": "أش كتقدم أنابيك للباحثين على الخدمة؟",
        "gold_answer": "تقدم عروض التوظيف والتكوينات المهنية.",
        "source_chunk_id": "chunk_054",
        "source_chunk_text": "تساعد الوكالة الوطنية لإنعاش التشغيل والكفاءات (أنابيك) الباحثين عن الشغل من خلال عروض التوظيف والتكوينات المهنية، وتنظم بشكل دوري منتديات وملتقيات للتشغيل تجمع الباحثين عن العمل بالمشغلين."
    },
    {
        "id": "b167",
        "msa_query": "ماذا تنظم أنابيك بشكل دوري؟",
        "darija_query": "أش كتنظم أنابيك بشكل دوري؟",
        "gold_answer": "تنظم منتديات وملتقيات للتشغيل.",
        "source_chunk_id": "chunk_054",
        "source_chunk_text": "تساعد الوكالة الوطنية لإنعاش التشغيل والكفاءات (أنابيك) الباحثين عن الشغل من خلال عروض التوظيف والتكوينات المهنية، وتنظم بشكل دوري منتديات وملتقيات للتشغيل تجمع الباحثين عن العمل بالمشغلين."
    },
    {
        "id": "b168",
        "msa_query": "من تجمع ملتقيات التشغيل التي تنظمها أنابيك؟",
        "darija_query": "شكون كتجمع الملتقيات ديال التشغيل؟",
        "gold_answer": "تجمع الباحثين عن العمل بالمشغلين.",
        "source_chunk_id": "chunk_054",
        "source_chunk_text": "تساعد الوكالة الوطنية لإنعاش التشغيل والكفاءات (أنابيك) الباحثين عن الشغل من خلال عروض التوظيف والتكوينات المهنية، وتنظم بشكل دوري منتديات وملتقيات للتشغيل تجمع الباحثين عن العمل بالمشغلين."
    },
    {
        "id": "b169",
        "msa_query": "ما هو الاسم الكامل لأنابيك؟",
        "darija_query": "شنو هو الاسم الكامل ديال أنابيك؟",
        "gold_answer": "الوكالة الوطنية لإنعاش التشغيل والكفاءات.",
        "source_chunk_id": "chunk_054",
        "source_chunk_text": "تساعد الوكالة الوطنية لإنعاش التشغيل والكفاءات (أنابيك) الباحثين عن الشغل من خلال عروض التوظيف والتكوينات المهنية، وتنظم بشكل دوري منتديات وملتقيات للتشغيل تجمع الباحثين عن العمل بالمشغلين."
    },
    {
        "id": "b170",
        "msa_query": "لمن تساعد أنابيك بشكل أساسي؟",
        "darija_query": "لمن كتعاون أنابيك بشكل أساسي؟",
        "gold_answer": "تساعد الباحثين عن الشغل.",
        "source_chunk_id": "chunk_054",
        "source_chunk_text": "تساعد الوكالة الوطنية لإنعاش التشغيل والكفاءات (أنابيك) الباحثين عن الشغل من خلال عروض التوظيف والتكوينات المهنية، وتنظم بشكل دوري منتديات وملتقيات للتشغيل تجمع الباحثين عن العمل بالمشغلين."
    },
    {
        "id": "b171",
        "msa_query": "ما هدف برامج فرز النفايات التي أطلقتها المدن المغربية؟",
        "darija_query": "شنو هو الهدف ديال برامج فرز النفايات؟",
        "gold_answer": "الحد من التلوث وتثمين النفايات القابلة لإعادة الاستعمال.",
        "source_chunk_id": "chunk_055",
        "source_chunk_text": "أطلقت عدة مدن مغربية برامج لفرز النفايات وإعادة التدوير، بهدف الحد من التلوث وتثمين النفايات القابلة لإعادة الاستعمال مثل البلاستيك والورق والزجاج."
    },
    {
        "id": "b172",
        "msa_query": "ما هي أنواع النفايات القابلة لإعادة التدوير المذكورة؟",
        "darija_query": "شنو هوما أنواع النفايات لي كتتصاوب من جديد؟",
        "gold_answer": "البلاستيك والورق والزجاج.",
        "source_chunk_id": "chunk_055",
        "source_chunk_text": "أطلقت عدة مدن مغربية برامج لفرز النفايات وإعادة التدوير، بهدف الحد من التلوث وتثمين النفايات القابلة لإعادة الاستعمال مثل البلاستيك والورق والزجاج."
    },
    {
        "id": "b173",
        "msa_query": "هل أطلقت جميع المدن المغربية برامج إعادة التدوير؟",
        "darija_query": "واش جميع المدن المغربية عندها برامج إعادة التدوير؟",
        "gold_answer": "أطلقت عدة مدن مغربية هذه البرامج، وليس جميعها.",
        "source_chunk_id": "chunk_055",
        "source_chunk_text": "أطلقت عدة مدن مغربية برامج لفرز النفايات وإعادة التدوير، بهدف الحد من التلوث وتثمين النفايات القابلة لإعادة الاستعمال مثل البلاستيك والورق والزجاج."
    },
    {
        "id": "b174",
        "msa_query": "ماذا يقصد بتثمين النفايات؟",
        "darija_query": "أش المقصود بتثمين النفايات؟",
        "gold_answer": "إعادة استعمال النفايات القابلة للتدوير مثل البلاستيك والورق والزجاج.",
        "source_chunk_id": "chunk_055",
        "source_chunk_text": "أطلقت عدة مدن مغربية برامج لفرز النفايات وإعادة التدوير، بهدف الحد من التلوث وتثمين النفايات القابلة لإعادة الاستعمال مثل البلاستيك والورق والزجاج."
    },
    {
        "id": "b175",
        "msa_query": "لماذا أطلقت المدن برامج فرز النفايات؟",
        "darija_query": "علاش المدن أطلقات برامج فرز النفايات؟",
        "gold_answer": "بهدف الحد من التلوث.",
        "source_chunk_id": "chunk_055",
        "source_chunk_text": "أطلقت عدة مدن مغربية برامج لفرز النفايات وإعادة التدوير، بهدف الحد من التلوث وتثمين النفايات القابلة لإعادة الاستعمال مثل البلاستيك والورق والزجاج."
    },
    {
        "id": "b176",
        "msa_query": "متى تشكل الحدائق العمومية وجهة مفضلة للعائلات؟",
        "darija_query": "إمتى كتكون الحدائق العمومية وجهة مفضلة للعائلات؟",
        "gold_answer": "خلال عطلة نهاية الأسبوع.",
        "source_chunk_id": "chunk_056",
        "source_chunk_text": "تشكل الحدائق العمومية وجهة مفضلة للعائلات المغربية خلال عطلة نهاية الأسبوع، حيث يقضي الأطفال أوقاتاً في اللعب بينما يجلس الكبار للراحة أو تناول وجبة خفيفة في الهواء الطلق."
    },
    {
        "id": "b177",
        "msa_query": "ماذا يفعل الأطفال في الحدائق العمومية؟",
        "darija_query": "أش كيديرو الدراري فالحديقة؟",
        "gold_answer": "يقضون أوقاتاً في اللعب.",
        "source_chunk_id": "chunk_056",
        "source_chunk_text": "تشكل الحدائق العمومية وجهة مفضلة للعائلات المغربية خلال عطلة نهاية الأسبوع، حيث يقضي الأطفال أوقاتاً في اللعب بينما يجلس الكبار للراحة أو تناول وجبة خفيفة في الهواء الطلق."
    },
    {
        "id": "b178",
        "msa_query": "ماذا يفعل الكبار في الحدائق العمومية؟",
        "darija_query": "أش كيديرو الكبار فالحديقة؟",
        "gold_answer": "يجلسون للراحة أو تناول وجبة خفيفة في الهواء الطلق.",
        "source_chunk_id": "chunk_056",
        "source_chunk_text": "تشكل الحدائق العمومية وجهة مفضلة للعائلات المغربية خلال عطلة نهاية الأسبوع، حيث يقضي الأطفال أوقاتاً في اللعب بينما يجلس الكبار للراحة أو تناول وجبة خفيفة في الهواء الطلق."
    },
    {
        "id": "b179",
        "msa_query": "هل تعتبر الحدائق العمومية وجهة مفضلة فقط للأطفال؟",
        "darija_query": "واش الحديقة وجهة غير للدراري؟",
        "gold_answer": "لا، هي وجهة مفضلة للعائلة بأكملها كباراً وصغاراً.",
        "source_chunk_id": "chunk_056",
        "source_chunk_text": "تشكل الحدائق العمومية وجهة مفضلة للعائلات المغربية خلال عطلة نهاية الأسبوع، حيث يقضي الأطفال أوقاتاً في اللعب بينما يجلس الكبار للراحة أو تناول وجبة خفيفة في الهواء الطلق."
    },
    {
        "id": "b180",
        "msa_query": "في أي فترة يزور المغاربة الحدائق العمومية بكثرة؟",
        "darija_query": "فأي وقت كيزورو المغاربة الحدائق بزاف؟",
        "gold_answer": "خلال عطلة نهاية الأسبوع.",
        "source_chunk_id": "chunk_056",
        "source_chunk_text": "تشكل الحدائق العمومية وجهة مفضلة للعائلات المغربية خلال عطلة نهاية الأسبوع، حيث يقضي الأطفال أوقاتاً في اللعب بينما يجلس الكبار للراحة أو تناول وجبة خفيفة في الهواء الطلق."
    },
    {
        "id": "b181",
        "msa_query": "ما هي المدن التي تستقطب السياح الداخليين المغاربة؟",
        "darija_query": "شنو هوما المدن لي كتجبد السياح المغاربة؟",
        "gold_answer": "شفشاون والصويرة وإفران من أبرزها.",
        "source_chunk_id": "chunk_057",
        "source_chunk_text": "تستقطب مدن مثل شفشاون والصويرة وإفران عدداً متزايداً من السياح الداخليين المغاربة، بفضل طبيعتها المميزة وأجوائها الهادئة البعيدة عن ازدحام المدن الكبرى."
    },
    {
        "id": "b182",
        "msa_query": "لماذا تستقطب هذه المدن السياح الداخليين؟",
        "darija_query": "علاش هاد المدن كتجبد السياح المغاربة؟",
        "gold_answer": "بفضل طبيعتها المميزة وأجوائها الهادئة البعيدة عن ازدحام المدن الكبرى.",
        "source_chunk_id": "chunk_057",
        "source_chunk_text": "تستقطب مدن مثل شفشاون والصويرة وإفران عدداً متزايداً من السياح الداخليين المغاربة، بفضل طبيعتها المميزة وأجوائها الهادئة البعيدة عن ازدحام المدن الكبرى."
    },
    {
        "id": "b183",
        "msa_query": "هل عدد السياح الداخليين في تزايد؟",
        "darija_query": "واش السياح المغاربة فتزايد؟",
        "gold_answer": "نعم، تستقطب هذه المدن عدداً متزايداً من السياح الداخليين.",
        "source_chunk_id": "chunk_057",
        "source_chunk_text": "تستقطب مدن مثل شفشاون والصويرة وإفران عدداً متزايداً من السياح الداخليين المغاربة، بفضل طبيعتها المميزة وأجوائها الهادئة البعيدة عن ازدحام المدن الكبرى."
    },
    {
        "id": "b184",
        "msa_query": "ما هي مدينة إفران من حيث الطبيعة؟",
        "darija_query": "شنو خاصية إفران من حيث الطبيعة؟",
        "gold_answer": "تتميز بطبيعتها المميزة وأجوائها الهادئة.",
        "source_chunk_id": "chunk_057",
        "source_chunk_text": "تستقطب مدن مثل شفشاون والصويرة وإفران عدداً متزايداً من السياح الداخليين المغاربة، بفضل طبيعتها المميزة وأجوائها الهادئة البعيدة عن ازدحام المدن الكبرى."
    },
    {
        "id": "b185",
        "msa_query": "ماذا يبحث عنه السياح الداخليون في هذه المدن؟",
        "darija_query": "أش كيقلبو عليه السياح المغاربة فهاد المدن؟",
        "gold_answer": "يبحثون عن الهدوء بعيداً عن ازدحام المدن الكبرى.",
        "source_chunk_id": "chunk_057",
        "source_chunk_text": "تستقطب مدن مثل شفشاون والصويرة وإفران عدداً متزايداً من السياح الداخليين المغاربة، بفضل طبيعتها المميزة وأجوائها الهادئة البعيدة عن ازدحام المدن الكبرى."
    },
    {
        "id": "b186",
        "msa_query": "أين تتساقط الثلوج شتاءً في المغرب؟",
        "darija_query": "فين كتساقط الثلج فالشتا فالمغرب؟",
        "gold_answer": "تتساقط الثلوج على قمم جبال الأطلس.",
        "source_chunk_id": "chunk_058",
        "source_chunk_text": "يشهد المغرب تبايناً كبيراً في المناخ حسب الفصول والمناطق، حيث تتساقط الثلوج شتاءً على قمم جبال الأطلس، بينما تكون درجات الحرارة صيفاً مرتفعة جداً في المناطق الداخلية والجنوبية."
    },
    {
        "id": "b187",
        "msa_query": "كيف تكون درجات الحرارة صيفاً في المناطق الداخلية؟",
        "darija_query": "كيفاش كتكون درجة الحرارة فالصيف فالمناطق الداخلية؟",
        "gold_answer": "تكون مرتفعة جداً صيفاً.",
        "source_chunk_id": "chunk_058",
        "source_chunk_text": "يشهد المغرب تبايناً كبيراً في المناخ حسب الفصول والمناطق، حيث تتساقط الثلوج شتاءً على قمم جبال الأطلس، بينما تكون درجات الحرارة صيفاً مرتفعة جداً في المناطق الداخلية والجنوبية."
    },
    {
        "id": "b188",
        "msa_query": "هل يتباين المناخ في المغرب حسب المنطقة؟",
        "darija_query": "واش المناخ كيختلف من منطقة لمنطقة فالمغرب؟",
        "gold_answer": "نعم، يشهد المغرب تبايناً كبيراً في المناخ حسب الفصول والمناطق.",
        "source_chunk_id": "chunk_058",
        "source_chunk_text": "يشهد المغرب تبايناً كبيراً في المناخ حسب الفصول والمناطق، حيث تتساقط الثلوج شتاءً على قمم جبال الأطلس، بينما تكون درجات الحرارة صيفاً مرتفعة جداً في المناطق الداخلية والجنوبية."
    },
    {
        "id": "b189",
        "msa_query": "في أي فصل تتساقط الثلوج على جبال الأطلس؟",
        "darija_query": "فأي فصل كتساقط الثلج على الأطلس؟",
        "gold_answer": "تتساقط شتاءً.",
        "source_chunk_id": "chunk_058",
        "source_chunk_text": "يشهد المغرب تبايناً كبيراً في المناخ حسب الفصول والمناطق، حيث تتساقط الثلوج شتاءً على قمم جبال الأطلس، بينما تكون درجات الحرارة صيفاً مرتفعة جداً في المناطق الداخلية والجنوبية."
    },
    {
        "id": "b190",
        "msa_query": "أين تكون درجات الحرارة مرتفعة جداً صيفاً؟",
        "darija_query": "فين كتكون الحرارة عالية بزاف فالصيف؟",
        "gold_answer": "في المناطق الداخلية والجنوبية.",
        "source_chunk_id": "chunk_058",
        "source_chunk_text": "يشهد المغرب تبايناً كبيراً في المناخ حسب الفصول والمناطق، حيث تتساقط الثلوج شتاءً على قمم جبال الأطلس، بينما تكون درجات الحرارة صيفاً مرتفعة جداً في المناطق الداخلية والجنوبية."
    },
    {
        "id": "b191",
        "msa_query": "من تلعب دوراً محورياً في الحفاظ على تقاليد الطبخ العائلي؟",
        "darija_query": "شكون كتلعب دور مهم فالحفاظ على تقاليد الطبخ؟",
        "gold_answer": "الجدة أو الأم الكبيرة.",
        "source_chunk_id": "chunk_059",
        "source_chunk_text": "يحرص كثير من العائلات المغربية على التجمع أيام الجمعة أو المناسبات الخاصة حول مائدة الكسكس، حيث تلعب الجدة أو الأم الكبيرة دوراً محورياً في الحفاظ على تقاليد الطبخ العائلي."
    },
    {
        "id": "b192",
        "msa_query": "متى تجتمع العائلات المغربية حول مائدة الكسكس؟",
        "darija_query": "إمتى كتجتمع العائلات على الكسكس؟",
        "gold_answer": "أيام الجمعة أو في المناسبات الخاصة.",
        "source_chunk_id": "chunk_059",
        "source_chunk_text": "يحرص كثير من العائلات المغربية على التجمع أيام الجمعة أو المناسبات الخاصة حول مائدة الكسكس، حيث تلعب الجدة أو الأم الكبيرة دوراً محورياً في الحفاظ على تقاليد الطبخ العائلي."
    },
    {
        "id": "b193",
        "msa_query": "ماذا يحرص كثير من العائلات على فعله يوم الجمعة؟",
        "darija_query": "أش كتحرص عليه بزاف من العائلات نهار الجمعة؟",
        "gold_answer": "التجمع حول مائدة الكسكس.",
        "source_chunk_id": "chunk_059",
        "source_chunk_text": "يحرص كثير من العائلات المغربية على التجمع أيام الجمعة أو المناسبات الخاصة حول مائدة الكسكس، حيث تلعب الجدة أو الأم الكبيرة دوراً محورياً في الحفاظ على تقاليد الطبخ العائلي."
    },
    {
        "id": "b194",
        "msa_query": "ما هو دور الجدة في العائلة المغربية عادة؟",
        "darija_query": "شنو هو الدور ديال الجدة فالعائلة؟",
        "gold_answer": "تلعب دوراً محورياً في الحفاظ على تقاليد الطبخ العائلي.",
        "source_chunk_id": "chunk_059",
        "source_chunk_text": "يحرص كثير من العائلات المغربية على التجمع أيام الجمعة أو المناسبات الخاصة حول مائدة الكسكس، حيث تلعب الجدة أو الأم الكبيرة دوراً محورياً في الحفاظ على تقاليد الطبخ العائلي."
    },
    {
        "id": "b195",
        "msa_query": "في أي مناسبات تجتمع العائلة حول الكسكس غير الجمعة؟",
        "darija_query": "فأي مناسبات كتجتمع العائلة على الكسكس غير الجمعة؟",
        "gold_answer": "في المناسبات الخاصة.",
        "source_chunk_id": "chunk_059",
        "source_chunk_text": "يحرص كثير من العائلات المغربية على التجمع أيام الجمعة أو المناسبات الخاصة حول مائدة الكسكس، حيث تلعب الجدة أو الأم الكبيرة دوراً محورياً في الحفاظ على تقاليد الطبخ العائلي."
    },
    {
        "id": "b196",
        "msa_query": "متى تُقام صلاة الجمعة في المساجد المغربية؟",
        "darija_query": "إمتى كتصلى صلاة الجمعة فالمساجد؟",
        "gold_answer": "تُقام ظهراً.",
        "source_chunk_id": "chunk_060",
        "source_chunk_text": "تُقام صلاة الجمعة في المساجد المغربية ظهراً وتتضمن خطبة دينية قبل الصلاة، وتُعتبر من أهم المناسبات الأسبوعية التي يحرص عليها المصلون، خصوصاً الرجال الذين يتوجهون جماعياً للمسجد."
    },
    {
        "id": "b197",
        "msa_query": "ماذا تتضمن صلاة الجمعة قبل الصلاة نفسها؟",
        "darija_query": "أش كتضم صلاة الجمعة قبل الصلاة؟",
        "gold_answer": "تتضمن خطبة دينية.",
        "source_chunk_id": "chunk_060",
        "source_chunk_text": "تُقام صلاة الجمعة في المساجد المغربية ظهراً وتتضمن خطبة دينية قبل الصلاة، وتُعتبر من أهم المناسبات الأسبوعية التي يحرص عليها المصلون، خصوصاً الرجال الذين يتوجهون جماعياً للمسجد."
    },
    {
        "id": "b198",
        "msa_query": "من يتوجه جماعياً للمسجد يوم الجمعة؟",
        "darija_query": "شكون لي كيمشي جماعي للجامع نهار الجمعة؟",
        "gold_answer": "الرجال يتوجهون جماعياً للمسجد.",
        "source_chunk_id": "chunk_060",
        "source_chunk_text": "تُقام صلاة الجمعة في المساجد المغربية ظهراً وتتضمن خطبة دينية قبل الصلاة، وتُعتبر من أهم المناسبات الأسبوعية التي يحرص عليها المصلون، خصوصاً الرجال الذين يتوجهون جماعياً للمسجد."
    },
    {
        "id": "b199",
        "msa_query": "هل تعتبر صلاة الجمعة من المناسبات الأسبوعية المهمة؟",
        "darija_query": "واش صلاة الجمعة من المناسبات الأسبوعية المهمة؟",
        "gold_answer": "نعم، تُعتبر من أهم المناسبات الأسبوعية.",
        "source_chunk_id": "chunk_060",
        "source_chunk_text": "تُقام صلاة الجمعة في المساجد المغربية ظهراً وتتضمن خطبة دينية قبل الصلاة، وتُعتبر من أهم المناسبات الأسبوعية التي يحرص عليها المصلون، خصوصاً الرجال الذين يتوجهون جماعياً للمسجد."
    },
    {
        "id": "b200",
        "msa_query": "ماذا يسبق صلاة الجمعة في المساجد؟",
        "darija_query": "أش كيسبق صلاة الجمعة فالمساجد؟",
        "gold_answer": "تسبقها خطبة دينية.",
        "source_chunk_id": "chunk_060",
        "source_chunk_text": "تُقام صلاة الجمعة في المساجد المغربية ظهراً وتتضمن خطبة دينية قبل الصلاة، وتُعتبر من أهم المناسبات الأسبوعية التي يحرص عليها المصلون، خصوصاً الرجال الذين يتوجهون جماعياً للمسجد."
    },
    {
        "id": "b201",
        "msa_query": "كيف يمكن للمواطن أخذ موعد في المراكز الصحية العمومية؟",
        "darija_query": "كيفاش المواطن يقدر ياخد موعد فالمركز الصحي؟",
        "gold_answer": "يمكنه أخذ موعد للاستشارة الطبية العامة مجاناً أو بثمن رمزي.",
        "source_chunk_id": "chunk_061",
        "source_chunk_text": "يمكن لأي مواطن أخذ موعد في المراكز الصحية العمومية بالمغرب للاستشارة الطبية العامة مجاناً أو بثمن رمزي، في حين تبقى الاستشارات في العيادات الخاصة أغلى ثمناً لكنها غالباً أسرع من حيث موعد الحصول على الموعد."
    },
    {
        "id": "b202",
        "msa_query": "لماذا يفضل البعض العيادات الخاصة رغم غلاء ثمنها؟",
        "darija_query": "علاش بعض الناس كيفضلو العيادات الخاصة؟",
        "gold_answer": "لأنها غالباً أسرع من حيث موعد الحصول على الاستشارة.",
        "source_chunk_id": "chunk_061",
        "source_chunk_text": "يمكن لأي مواطن أخذ موعد في المراكز الصحية العمومية بالمغرب للاستشارة الطبية العامة مجاناً أو بثمن رمزي، في حين تبقى الاستشارات في العيادات الخاصة أغلى ثمناً لكنها غالباً أسرع من حيث موعد الحصول على الموعد."
    },
    {
        "id": "b203",
        "msa_query": "هل الاستشارة في المراكز الصحية العمومية مجانية دائماً؟",
        "darija_query": "واش الاستشارة فالمركز الصحي مجانية ديما؟",
        "gold_answer": "تكون مجانية أو بثمن رمزي حسب الحالة.",
        "source_chunk_id": "chunk_061",
        "source_chunk_text": "يمكن لأي مواطن أخذ موعد في المراكز الصحية العمومية بالمغرب للاستشارة الطبية العامة مجاناً أو بثمن رمزي، في حين تبقى الاستشارات في العيادات الخاصة أغلى ثمناً لكنها غالباً أسرع من حيث موعد الحصول على الموعد."
    },
    {
        "id": "b204",
        "msa_query": "ما الفرق الأساسي بين العيادة الخاصة والمركز الصحي العمومي؟",
        "darija_query": "شنو الفرق بين العيادة الخاصة والمركز الصحي؟",
        "gold_answer": "العيادة الخاصة أغلى لكن أسرع في الموعد.",
        "source_chunk_id": "chunk_061",
        "source_chunk_text": "يمكن لأي مواطن أخذ موعد في المراكز الصحية العمومية بالمغرب للاستشارة الطبية العامة مجاناً أو بثمن رمزي، في حين تبقى الاستشارات في العيادات الخاصة أغلى ثمناً لكنها غالباً أسرع من حيث موعد الحصول على الموعد."
    },
    {
        "id": "b205",
        "msa_query": "من يستطيع أخذ موعد في المراكز الصحية العمومية؟",
        "darija_query": "شكون لي يقدر ياخد موعد فالمركز الصحي؟",
        "gold_answer": "أي مواطن يمكنه أخذ موعد.",
        "source_chunk_id": "chunk_061",
        "source_chunk_text": "يمكن لأي مواطن أخذ موعد في المراكز الصحية العمومية بالمغرب للاستشارة الطبية العامة مجاناً أو بثمن رمزي، في حين تبقى الاستشارات في العيادات الخاصة أغلى ثمناً لكنها غالباً أسرع من حيث موعد الحصول على الموعد."
    },
    {
        "id": "b206",
        "msa_query": "ما نوع الأدوية التي تُباع دون وصفة طبية؟",
        "darija_query": "شنو نوع الدوا لي كيتباع بلا وصفة؟",
        "gold_answer": "المسكنات البسيطة وأدوية الزكام تُباع دون وصفة طبية.",
        "source_chunk_id": "chunk_062",
        "source_chunk_text": "تبيع بعض الصيدليات المغربية أدوية معينة دون وصفة طبية، خصوصاً المسكنات البسيطة وأدوية الزكام، في حين تتطلب الأدوية القوية أو المضادات الحيوية وصفة من طبيب مختص."
    },
    {
        "id": "b207",
        "msa_query": "هل تحتاج المضادات الحيوية وصفة طبية؟",
        "darija_query": "واش الأنتيبيوتيك محتاجة وصفة؟",
        "gold_answer": "نعم، تتطلب المضادات الحيوية وصفة من طبيب مختص.",
        "source_chunk_id": "chunk_062",
        "source_chunk_text": "تبيع بعض الصيدليات المغربية أدوية معينة دون وصفة طبية، خصوصاً المسكنات البسيطة وأدوية الزكام، في حين تتطلب الأدوية القوية أو المضادات الحيوية وصفة من طبيب مختص."
    },
    {
        "id": "b208",
        "msa_query": "ما هي الأدوية التي تتطلب وصفة طبية إلزامية؟",
        "darija_query": "شنو هوما الأدوية لي خاصها وصفة؟",
        "gold_answer": "الأدوية القوية أو المضادات الحيوية.",
        "source_chunk_id": "chunk_062",
        "source_chunk_text": "تبيع بعض الصيدليات المغربية أدوية معينة دون وصفة طبية، خصوصاً المسكنات البسيطة وأدوية الزكام، في حين تتطلب الأدوية القوية أو المضادات الحيوية وصفة من طبيب مختص."
    },
    {
        "id": "b209",
        "msa_query": "هل جميع الصيدليات تبيع الأدوية دون وصفة؟",
        "darija_query": "واش جميع الصيدليات كتبيع الدوا بلا وصفة؟",
        "gold_answer": "بعض الصيدليات تبيع أدوية معينة دون وصفة طبية.",
        "source_chunk_id": "chunk_062",
        "source_chunk_text": "تبيع بعض الصيدليات المغربية أدوية معينة دون وصفة طبية، خصوصاً المسكنات البسيطة وأدوية الزكام، في حين تتطلب الأدوية القوية أو المضادات الحيوية وصفة من طبيب مختص."
    },
    {
        "id": "b210",
        "msa_query": "اذكر مثالاً عن دواء لا يحتاج وصفة طبية؟",
        "darija_query": "عطيني مثال على دوا ما محتاجش وصفة؟",
        "gold_answer": "أدوية الزكام مثال على دواء لا يحتاج وصفة.",
        "source_chunk_id": "chunk_062",
        "source_chunk_text": "تبيع بعض الصيدليات المغربية أدوية معينة دون وصفة طبية، خصوصاً المسكنات البسيطة وأدوية الزكام، في حين تتطلب الأدوية القوية أو المضادات الحيوية وصفة من طبيب مختص."
    },
    {
        "id": "b211",
        "msa_query": "ماذا تناقش لقاءات الأساتذة وأولياء التلاميذ؟",
        "darija_query": "أش كتناقش اللقاءات بين الأساتذة والأولياء؟",
        "gold_answer": "تناقش نتائج ومستوى التلميذ الدراسي.",
        "source_chunk_id": "chunk_063",
        "source_chunk_text": "تنظم المؤسسات التعليمية المغربية لقاءات دورية بين الأساتذة وأولياء التلاميذ لمناقشة نتائج ومستوى التلميذ الدراسي، وتُسلَّم بطاقة النقط لكل تلميذ في نهاية كل دورة دراسية."
    },
    {
        "id": "b212",
        "msa_query": "متى تُسلَّم بطاقة النقط للتلميذ؟",
        "darija_query": "إمتى كتسلم بطاقة النقط للتلميذ؟",
        "gold_answer": "تُسلَّم في نهاية كل دورة دراسية.",
        "source_chunk_id": "chunk_063",
        "source_chunk_text": "تنظم المؤسسات التعليمية المغربية لقاءات دورية بين الأساتذة وأولياء التلاميذ لمناقشة نتائج ومستوى التلميذ الدراسي، وتُسلَّم بطاقة النقط لكل تلميذ في نهاية كل دورة دراسية."
    },
    {
        "id": "b213",
        "msa_query": "من ينظم لقاءات مناقشة نتائج التلاميذ؟",
        "darija_query": "شكون لي كينظم اللقاءات ديال النتائج؟",
        "gold_answer": "تنظمها المؤسسات التعليمية.",
        "source_chunk_id": "chunk_063",
        "source_chunk_text": "تنظم المؤسسات التعليمية المغربية لقاءات دورية بين الأساتذة وأولياء التلاميذ لمناقشة نتائج ومستوى التلميذ الدراسي، وتُسلَّم بطاقة النقط لكل تلميذ في نهاية كل دورة دراسية."
    },
    {
        "id": "b214",
        "msa_query": "بين من تُعقد اللقاءات الدورية في المدرسة؟",
        "darija_query": "بين شكون كتنعقد اللقاءات فالمدرسة؟",
        "gold_answer": "بين الأساتذة وأولياء التلاميذ.",
        "source_chunk_id": "chunk_063",
        "source_chunk_text": "تنظم المؤسسات التعليمية المغربية لقاءات دورية بين الأساتذة وأولياء التلاميذ لمناقشة نتائج ومستوى التلميذ الدراسي، وتُسلَّم بطاقة النقط لكل تلميذ في نهاية كل دورة دراسية."
    },
    {
        "id": "b215",
        "msa_query": "ماذا يحصل عليه كل تلميذ في نهاية الدورة؟",
        "darija_query": "أش كياخد كل تلميذ فآخر الدورة؟",
        "gold_answer": "يحصل على بطاقة النقط.",
        "source_chunk_id": "chunk_063",
        "source_chunk_text": "تنظم المؤسسات التعليمية المغربية لقاءات دورية بين الأساتذة وأولياء التلاميذ لمناقشة نتائج ومستوى التلميذ الدراسي، وتُسلَّم بطاقة النقط لكل تلميذ في نهاية كل دورة دراسية."
    },
    {
        "id": "b216",
        "msa_query": "لماذا يلجأ التلاميذ إلى الدروس الخصوصية؟",
        "darija_query": "علاش التلاميذ كيلجاو للدروس الخصوصية؟",
        "gold_answer": "لتقوية مستواهم في بعض المواد الأساسية.",
        "source_chunk_id": "chunk_064",
        "source_chunk_text": "يلجأ كثير من التلاميذ المغاربة إلى الدروس الخصوصية (الدوزينج) لتقوية مستواهم في بعض المواد الأساسية، خصوصاً في سنوات الإعدادي والثانوي استعداداً للامتحانات الإشهادية."
    },
    {
        "id": "b217",
        "msa_query": "في أي سنوات دراسية ينتشر الدوزينج أكثر؟",
        "darija_query": "فأي سنوات كيكثر الدوزينج؟",
        "gold_answer": "في سنوات الإعدادي والثانوي.",
        "source_chunk_id": "chunk_064",
        "source_chunk_text": "يلجأ كثير من التلاميذ المغاربة إلى الدروس الخصوصية (الدوزينج) لتقوية مستواهم في بعض المواد الأساسية، خصوصاً في سنوات الإعدادي والثانوي استعداداً للامتحانات الإشهادية."
    },
    {
        "id": "b218",
        "msa_query": "استعداداً لماذا يلجأ التلاميذ للدروس الخصوصية؟",
        "darija_query": "استعداد لأش كيلجاو التلاميذ للدوزينج؟",
        "gold_answer": "استعداداً للامتحانات الإشهادية.",
        "source_chunk_id": "chunk_064",
        "source_chunk_text": "يلجأ كثير من التلاميذ المغاربة إلى الدروس الخصوصية (الدوزينج) لتقوية مستواهم في بعض المواد الأساسية، خصوصاً في سنوات الإعدادي والثانوي استعداداً للامتحانات الإشهادية."
    },
    {
        "id": "b219",
        "msa_query": "ماذا يسمي المغاربة الدروس الخصوصية بالدارجة؟",
        "darija_query": "كيفاش كيسميو المغاربة الدروس الخصوصية؟",
        "gold_answer": "يسمونها الدوزينج.",
        "source_chunk_id": "chunk_064",
        "source_chunk_text": "يلجأ كثير من التلاميذ المغاربة إلى الدروس الخصوصية (الدوزينج) لتقوية مستواهم في بعض المواد الأساسية، خصوصاً في سنوات الإعدادي والثانوي استعداداً للامتحانات الإشهادية."
    },
    {
        "id": "b220",
        "msa_query": "هل الدروس الخصوصية منتشرة بين كثير من التلاميذ؟",
        "darija_query": "واش الدوزينج منتشر بزاف؟",
        "gold_answer": "نعم، يلجأ إليها كثير من التلاميذ المغاربة.",
        "source_chunk_id": "chunk_064",
        "source_chunk_text": "يلجأ كثير من التلاميذ المغاربة إلى الدروس الخصوصية (الدوزينج) لتقوية مستواهم في بعض المواد الأساسية، خصوصاً في سنوات الإعدادي والثانوي استعداداً للامتحانات الإشهادية."
    },
    {
        "id": "b221",
        "msa_query": "ماذا يسأل المغاربة عادة في بداية المكالمة الهاتفية؟",
        "darija_query": "أش كيسولو المغاربة فبداية التيليفون؟",
        "gold_answer": "يسألون عن الأحوال عبر عبارة 'لاباس' أو 'بخير'.",
        "source_chunk_id": "chunk_065",
        "source_chunk_text": "من العادات الاجتماعية الشائعة في المغرب السؤال عن الأحوال عبر عبارة 'لاباس' أو 'بخير' عند بداية أي مكالمة هاتفية أو لقاء، قبل الدخول في موضوع المكالمة أو الحديث الأساسي."
    },
    {
        "id": "b222",
        "msa_query": "متى تُقال عبارة السؤال عن الأحوال؟",
        "darija_query": "إمتى كتقال عبارة السؤال على الأحوال؟",
        "gold_answer": "تُقال قبل الدخول في موضوع المكالمة أو الحديث الأساسي.",
        "source_chunk_id": "chunk_065",
        "source_chunk_text": "من العادات الاجتماعية الشائعة في المغرب السؤال عن الأحوال عبر عبارة 'لاباس' أو 'بخير' عند بداية أي مكالمة هاتفية أو لقاء، قبل الدخول في موضوع المكالمة أو الحديث الأساسي."
    },
    {
        "id": "b223",
        "msa_query": "هل يقتصر السؤال عن الأحوال على المكالمات الهاتفية؟",
        "darija_query": "واش السؤال على الأحوال غير فالتيليفون؟",
        "gold_answer": "لا، يقال أيضاً عند بداية أي لقاء.",
        "source_chunk_id": "chunk_065",
        "source_chunk_text": "من العادات الاجتماعية الشائعة في المغرب السؤال عن الأحوال عبر عبارة 'لاباس' أو 'بخير' عند بداية أي مكالمة هاتفية أو لقاء، قبل الدخول في موضوع المكالمة أو الحديث الأساسي."
    },
    {
        "id": "b224",
        "msa_query": "ما هي إحدى العبارات الشائعة للسؤال عن الأحوال؟",
        "darija_query": "شنو وحدة من العبارات المشهورة للسؤال على الأحوال؟",
        "gold_answer": "عبارة 'لاباس'.",
        "source_chunk_id": "chunk_065",
        "source_chunk_text": "من العادات الاجتماعية الشائعة في المغرب السؤال عن الأحوال عبر عبارة 'لاباس' أو 'بخير' عند بداية أي مكالمة هاتفية أو لقاء، قبل الدخول في موضوع المكالمة أو الحديث الأساسي."
    },
    {
        "id": "b225",
        "msa_query": "هل تعتبر هذه العادة اجتماعية شائعة في المغرب؟",
        "darija_query": "واش هاد العادة منتشرة فالمغرب؟",
        "gold_answer": "نعم، من العادات الاجتماعية الشائعة في المغرب.",
        "source_chunk_id": "chunk_065",
        "source_chunk_text": "من العادات الاجتماعية الشائعة في المغرب السؤال عن الأحوال عبر عبارة 'لاباس' أو 'بخير' عند بداية أي مكالمة هاتفية أو لقاء، قبل الدخول في موضوع المكالمة أو الحديث الأساسي."
    },
    {
        "id": "b226",
        "msa_query": "في أي مناسبات يتعاون الجيران فيما بينهم؟",
        "darija_query": "فأي مناسبات كيتعاونو الجيران؟",
        "gold_answer": "في المناسبات السعيدة كالأعراس أو الحزينة كالوفيات.",
        "source_chunk_id": "chunk_066",
        "source_chunk_text": "يحرص الجيران في الأحياء المغربية التقليدية على التعاون والتضامن فيما بينهم، سواء في المناسبات السعيدة كالأعراس أو الحزينة كالوفيات، ويُعتبر الجار من أقرب الناس بحكم القرب اليومي."
    },
    {
        "id": "b227",
        "msa_query": "لماذا يُعتبر الجار من أقرب الناس؟",
        "darija_query": "علاش الجار من أقرب الناس؟",
        "gold_answer": "بحكم القرب اليومي.",
        "source_chunk_id": "chunk_066",
        "source_chunk_text": "يحرص الجيران في الأحياء المغربية التقليدية على التعاون والتضامن فيما بينهم، سواء في المناسبات السعيدة كالأعراس أو الحزينة كالوفيات، ويُعتبر الجار من أقرب الناس بحكم القرب اليومي."
    },
    {
        "id": "b228",
        "msa_query": "أين يحرص الجيران على التعاون والتضامن؟",
        "darija_query": "فين كيحرص الجيران على التعاون؟",
        "gold_answer": "في الأحياء المغربية التقليدية.",
        "source_chunk_id": "chunk_066",
        "source_chunk_text": "يحرص الجيران في الأحياء المغربية التقليدية على التعاون والتضامن فيما بينهم، سواء في المناسبات السعيدة كالأعراس أو الحزينة كالوفيات، ويُعتبر الجار من أقرب الناس بحكم القرب اليومي."
    },
    {
        "id": "b229",
        "msa_query": "اذكر مثالاً عن مناسبة حزينة يتضامن فيها الجيران؟",
        "darija_query": "عطيني مثال على مناسبة حزينة كيتضامنو فيها الجيران؟",
        "gold_answer": "الوفيات مثال على مناسبة حزينة.",
        "source_chunk_id": "chunk_066",
        "source_chunk_text": "يحرص الجيران في الأحياء المغربية التقليدية على التعاون والتضامن فيما بينهم، سواء في المناسبات السعيدة كالأعراس أو الحزينة كالوفيات، ويُعتبر الجار من أقرب الناس بحكم القرب اليومي."
    },
    {
        "id": "b230",
        "msa_query": "هل التضامن بين الجيران عادة تقليدية؟",
        "darija_query": "واش التضامن بين الجيران عادة قديمة؟",
        "gold_answer": "نعم، يحرص الجيران في الأحياء التقليدية على التعاون والتضامن.",
        "source_chunk_id": "chunk_066",
        "source_chunk_text": "يحرص الجيران في الأحياء المغربية التقليدية على التعاون والتضامن فيما بينهم، سواء في المناسبات السعيدة كالأعراس أو الحزينة كالوفيات، ويُعتبر الجار من أقرب الناس بحكم القرب اليومي."
    },
    {
        "id": "b231",
        "msa_query": "لماذا يترقب الفلاحون أمطار الخريف والشتاء؟",
        "darija_query": "علاش الفلاحين كيترقبو شتا الخريف والشتا؟",
        "gold_answer": "لأنها أساسية لنجاح الموسم الفلاحي.",
        "source_chunk_id": "chunk_067",
        "source_chunk_text": "تتفاوت كمية التساقطات المطرية في المغرب بشكل كبير من سنة لأخرى، ويترقب الفلاحون بشكل خاص أمطار الخريف والشتاء لأنها أساسية لنجاح الموسم الفلاحي، خصوصاً زراعة الحبوب البورية."
    },
    {
        "id": "b232",
        "msa_query": "هل تتفاوت كمية الأمطار في المغرب من سنة لأخرى؟",
        "darija_query": "واش كمية الشتا كتختلف من عام لعام؟",
        "gold_answer": "نعم، تتفاوت بشكل كبير من سنة لأخرى.",
        "source_chunk_id": "chunk_067",
        "source_chunk_text": "تتفاوت كمية التساقطات المطرية في المغرب بشكل كبير من سنة لأخرى، ويترقب الفلاحون بشكل خاص أمطار الخريف والشتاء لأنها أساسية لنجاح الموسم الفلاحي، خصوصاً زراعة الحبوب البورية."
    },
    {
        "id": "b233",
        "msa_query": "لأي نوع من الزراعة تُعتبر الأمطار أساسية بشكل خاص؟",
        "darija_query": "لأي نوع ديال الزراعة الشتا أساسية بزاف؟",
        "gold_answer": "زراعة الحبوب البورية.",
        "source_chunk_id": "chunk_067",
        "source_chunk_text": "تتفاوت كمية التساقطات المطرية في المغرب بشكل كبير من سنة لأخرى، ويترقب الفلاحون بشكل خاص أمطار الخريف والشتاء لأنها أساسية لنجاح الموسم الفلاحي، خصوصاً زراعة الحبوب البورية."
    },
    {
        "id": "b234",
        "msa_query": "من يترقب أمطار الخريف والشتاء بشكل خاص؟",
        "darija_query": "شكون لي كيترقب شتا الخريف بزاف؟",
        "gold_answer": "الفلاحون يترقبونها بشكل خاص.",
        "source_chunk_id": "chunk_067",
        "source_chunk_text": "تتفاوت كمية التساقطات المطرية في المغرب بشكل كبير من سنة لأخرى، ويترقب الفلاحون بشكل خاص أمطار الخريف والشتاء لأنها أساسية لنجاح الموسم الفلاحي، خصوصاً زراعة الحبوب البورية."
    },
    {
        "id": "b235",
        "msa_query": "ما أهمية موسم الأمطار للفلاحة؟",
        "darija_query": "شنو أهمية موسم الشتا للفلاحة؟",
        "gold_answer": "أساسية لنجاح الموسم الفلاحي.",
        "source_chunk_id": "chunk_067",
        "source_chunk_text": "تتفاوت كمية التساقطات المطرية في المغرب بشكل كبير من سنة لأخرى، ويترقب الفلاحون بشكل خاص أمطار الخريف والشتاء لأنها أساسية لنجاح الموسم الفلاحي، خصوصاً زراعة الحبوب البورية."
    },
    {
        "id": "b236",
        "msa_query": "ماذا تغطي شبكة الحافلات في المدن الكبرى؟",
        "darija_query": "أش كتغطي شبكة الباص فالمدن الكبار؟",
        "gold_answer": "تغطي معظم الأحياء.",
        "source_chunk_id": "chunk_068",
        "source_chunk_text": "تغطي شبكة الحافلات (الباص) في المدن المغربية الكبرى معظم الأحياء، وتُعتبر وسيلة نقل اقتصادية مقارنة بالطاكسي، رغم أنها قد تشهد ازدحاماً كبيراً خلال ساعات الذروة."
    },
    {
        "id": "b237",
        "msa_query": "لماذا تُعتبر الحافلة وسيلة نقل اقتصادية؟",
        "darija_query": "علاش الباص وسيلة نقل رخيصة؟",
        "gold_answer": "مقارنة بالطاكسي، تُعتبر اقتصادية.",
        "source_chunk_id": "chunk_068",
        "source_chunk_text": "تغطي شبكة الحافلات (الباص) في المدن المغربية الكبرى معظم الأحياء، وتُعتبر وسيلة نقل اقتصادية مقارنة بالطاكسي، رغم أنها قد تشهد ازدحاماً كبيراً خلال ساعات الذروة."
    },
    {
        "id": "b238",
        "msa_query": "متى تشهد الحافلات ازدحاماً كبيراً؟",
        "darija_query": "إمتى كيكون الباص مزدحم بزاف؟",
        "gold_answer": "خلال ساعات الذروة.",
        "source_chunk_id": "chunk_068",
        "source_chunk_text": "تغطي شبكة الحافلات (الباص) في المدن المغربية الكبرى معظم الأحياء، وتُعتبر وسيلة نقل اقتصادية مقارنة بالطاكسي، رغم أنها قد تشهد ازدحاماً كبيراً خلال ساعات الذروة."
    },
    {
        "id": "b239",
        "msa_query": "هل تغطي شبكة الحافلات جميع الأحياء؟",
        "darija_query": "واش الباص كيغطي جميع الأحياء؟",
        "gold_answer": "تغطي معظم الأحياء، وليس بالضرورة جميعها.",
        "source_chunk_id": "chunk_068",
        "source_chunk_text": "تغطي شبكة الحافلات (الباص) في المدن المغربية الكبرى معظم الأحياء، وتُعتبر وسيلة نقل اقتصادية مقارنة بالطاكسي، رغم أنها قد تشهد ازدحاماً كبيراً خلال ساعات الذروة."
    },
    {
        "id": "b240",
        "msa_query": "بماذا تُقارن الحافلة من حيث الثمن؟",
        "darija_query": "بأش كتقارن الباص من ناحية الثمن؟",
        "gold_answer": "تُقارن بالطاكسي، وتكون أرخص منه.",
        "source_chunk_id": "chunk_068",
        "source_chunk_text": "تغطي شبكة الحافلات (الباص) في المدن المغربية الكبرى معظم الأحياء، وتُعتبر وسيلة نقل اقتصادية مقارنة بالطاكسي، رغم أنها قد تشهد ازدحاماً كبيراً خلال ساعات الذروة."
    },
    {
        "id": "b241",
        "msa_query": "ماذا يحتاج الزبون لفتح حساب بنكي؟",
        "darija_query": "أش محتاج الزبون باش يحل حساب فالبنك؟",
        "gold_answer": "يحتاج بطاقة التعريف الوطنية وشهادة السكنى وإثبات الدخل.",
        "source_chunk_id": "chunk_069",
        "source_chunk_text": "لفتح حساب بنكي في المغرب، يحتاج الزبون عادة لتقديم بطاقة التعريف الوطنية وشهادة السكنى وإثبات الدخل، وتوفر أغلب البنوك حسابات مجانية أو بأثمنة رمزية للشباب والطلبة."
    },
    {
        "id": "b242",
        "msa_query": "لمن توفر البنوك حسابات مجانية أو بأثمنة رمزية؟",
        "darija_query": "لمن كتوفر البنوك حسابات مجانية؟",
        "gold_answer": "للشباب والطلبة.",
        "source_chunk_id": "chunk_069",
        "source_chunk_text": "لفتح حساب بنكي في المغرب، يحتاج الزبون عادة لتقديم بطاقة التعريف الوطنية وشهادة السكنى وإثبات الدخل، وتوفر أغلب البنوك حسابات مجانية أو بأثمنة رمزية للشباب والطلبة."
    },
    {
        "id": "b243",
        "msa_query": "هل تتطلب جميع البنوك نفس الوثائق لفتح حساب؟",
        "darija_query": "واش جميع البنوك كتطلب نفس الوثائق؟",
        "gold_answer": "عادة تتطلب البطاقة الوطنية وشهادة السكنى وإثبات الدخل.",
        "source_chunk_id": "chunk_069",
        "source_chunk_text": "لفتح حساب بنكي في المغرب، يحتاج الزبون عادة لتقديم بطاقة التعريف الوطنية وشهادة السكنى وإثبات الدخل، وتوفر أغلب البنوك حسابات مجانية أو بأثمنة رمزية للشباب والطلبة."
    },
    {
        "id": "b244",
        "msa_query": "ما هي إحدى الوثائق المطلوبة لفتح حساب بنكي؟",
        "darija_query": "شنو وحدة من الوثائق المطلوبة باش تحل حساب؟",
        "gold_answer": "بطاقة التعريف الوطنية.",
        "source_chunk_id": "chunk_069",
        "source_chunk_text": "لفتح حساب بنكي في المغرب، يحتاج الزبون عادة لتقديم بطاقة التعريف الوطنية وشهادة السكنى وإثبات الدخل، وتوفر أغلب البنوك حسابات مجانية أو بأثمنة رمزية للشباب والطلبة."
    },
    {
        "id": "b245",
        "msa_query": "هل يمكن للطلبة الحصول على حسابات بنكية رمزية الثمن؟",
        "darija_query": "واش الطلبة يقدرو ياخدو حساب رخيص؟",
        "gold_answer": "نعم، توفر أغلب البنوك حسابات بأثمنة رمزية للطلبة.",
        "source_chunk_id": "chunk_069",
        "source_chunk_text": "لفتح حساب بنكي في المغرب، يحتاج الزبون عادة لتقديم بطاقة التعريف الوطنية وشهادة السكنى وإثبات الدخل، وتوفر أغلب البنوك حسابات مجانية أو بأثمنة رمزية للشباب والطلبة."
    },
    {
        "id": "b246",
        "msa_query": "ما هي العملة الرسمية للمملكة المغربية؟",
        "darija_query": "شنو هي العملة الرسمية ديال المغرب؟",
        "gold_answer": "الدرهم المغربي.",
        "source_chunk_id": "chunk_070",
        "source_chunk_text": "الدرهم المغربي هو العملة الرسمية للمملكة، وهو غير قابل للتحويل بشكل كامل خارج المغرب، ما يعني أن صرفه يخضع لقيود معينة عند السفر أو التعامل مع العملات الأجنبية."
    },
    {
        "id": "b247",
        "msa_query": "هل الدرهم قابل للتحويل بشكل كامل خارج المغرب؟",
        "darija_query": "واش الدرهم كيتحول بالكامل برا المغرب؟",
        "gold_answer": "لا، هو غير قابل للتحويل بشكل كامل خارج المغرب.",
        "source_chunk_id": "chunk_070",
        "source_chunk_text": "الدرهم المغربي هو العملة الرسمية للمملكة، وهو غير قابل للتحويل بشكل كامل خارج المغرب، ما يعني أن صرفه يخضع لقيود معينة عند السفر أو التعامل مع العملات الأجنبية."
    },
    {
        "id": "b248",
        "msa_query": "ماذا يعني عدم قابلية التحويل الكامل للدرهم؟",
        "darija_query": "أش معناها ما الدرهم غير قابل للتحويل بالكامل؟",
        "gold_answer": "يعني أن صرفه يخضع لقيود عند السفر أو التعامل مع العملات الأجنبية.",
        "source_chunk_id": "chunk_070",
        "source_chunk_text": "الدرهم المغربي هو العملة الرسمية للمملكة، وهو غير قابل للتحويل بشكل كامل خارج المغرب، ما يعني أن صرفه يخضع لقيود معينة عند السفر أو التعامل مع العملات الأجنبية."
    },
    {
        "id": "b249",
        "msa_query": "متى تظهر قيود صرف الدرهم بشكل واضح؟",
        "darija_query": "إمتى كتبان القيود ديال صرف الدرهم؟",
        "gold_answer": "عند السفر أو التعامل مع العملات الأجنبية.",
        "source_chunk_id": "chunk_070",
        "source_chunk_text": "الدرهم المغربي هو العملة الرسمية للمملكة، وهو غير قابل للتحويل بشكل كامل خارج المغرب، ما يعني أن صرفه يخضع لقيود معينة عند السفر أو التعامل مع العملات الأجنبية."
    },
    {
        "id": "b250",
        "msa_query": "هل الدرهم عملة قابلة للتحويل بالكامل؟",
        "darija_query": "واش الدرهم عملة قابلة للتحويل بالكامل؟",
        "gold_answer": "لا، غير قابل للتحويل بشكل كامل.",
        "source_chunk_id": "chunk_070",
        "source_chunk_text": "الدرهم المغربي هو العملة الرسمية للمملكة، وهو غير قابل للتحويل بشكل كامل خارج المغرب، ما يعني أن صرفه يخضع لقيود معينة عند السفر أو التعامل مع العملات الأجنبية."
    },
    {
        "id": "b251",
        "msa_query": "هل التأمين على السيارات إلزامي في المغرب؟",
        "darija_query": "واش التأمين ديال الطوموبيل إجباري فالمغرب؟",
        "gold_answer": "نعم، يُعتبر التأمين على السيارات إلزامياً.",
        "source_chunk_id": "chunk_071",
        "source_chunk_text": "يُعتبر التأمين على السيارات إلزامياً في المغرب، إذ لا يمكن لأي مركبة الاستعمال على الطريق العمومي دون شهادة تأمين سارية المفعول، وتختلف الأثمنة حسب نوع السيارة وسجل السائق."
    },
    {
        "id": "b252",
        "msa_query": "ماذا يحتاج السائق ليستعمل سيارته على الطريق العمومي؟",
        "darija_query": "أش محتاج السايق باش يخدم الطريق العمومي؟",
        "gold_answer": "يحتاج شهادة تأمين سارية المفعول.",
        "source_chunk_id": "chunk_071",
        "source_chunk_text": "يُعتبر التأمين على السيارات إلزامياً في المغرب، إذ لا يمكن لأي مركبة الاستعمال على الطريق العمومي دون شهادة تأمين سارية المفعول، وتختلف الأثمنة حسب نوع السيارة وسجل السائق."
    },
    {
        "id": "b253",
        "msa_query": "ما الذي يحدد ثمن التأمين على السيارة؟",
        "darija_query": "أش لي كيحدد ثمن تأمين الطوموبيل؟",
        "gold_answer": "تختلف الأثمنة حسب نوع السيارة وسجل السائق.",
        "source_chunk_id": "chunk_071",
        "source_chunk_text": "يُعتبر التأمين على السيارات إلزامياً في المغرب، إذ لا يمكن لأي مركبة الاستعمال على الطريق العمومي دون شهادة تأمين سارية المفعول، وتختلف الأثمنة حسب نوع السيارة وسجل السائق."
    },
    {
        "id": "b254",
        "msa_query": "هل يمكن استعمال سيارة دون تأمين على الطريق العمومي؟",
        "darija_query": "واش ممكن نخدم الطوموبيل بلا تأمين؟",
        "gold_answer": "لا، لا يمكن استعمالها دون شهادة تأمين سارية المفعول.",
        "source_chunk_id": "chunk_071",
        "source_chunk_text": "يُعتبر التأمين على السيارات إلزامياً في المغرب، إذ لا يمكن لأي مركبة الاستعمال على الطريق العمومي دون شهادة تأمين سارية المفعول، وتختلف الأثمنة حسب نوع السيارة وسجل السائق."
    },
    {
        "id": "b255",
        "msa_query": "ما نوع التأمين المذكور في النص؟",
        "darija_query": "أش نوع التأمين لي تهضر عليه النص؟",
        "gold_answer": "التأمين على السيارات.",
        "source_chunk_id": "chunk_071",
        "source_chunk_text": "يُعتبر التأمين على السيارات إلزامياً في المغرب، إذ لا يمكن لأي مركبة الاستعمال على الطريق العمومي دون شهادة تأمين سارية المفعول، وتختلف الأثمنة حسب نوع السيارة وسجل السائق."
    },
    {
        "id": "b256",
        "msa_query": "ما الذي تسلمه مصالح الحالة المدنية؟",
        "darija_query": "أش كتسلم مصالح الحالة المدنية؟",
        "gold_answer": "تسلم شهادات الازدياد والوفاة والزواج.",
        "source_chunk_id": "chunk_072",
        "source_chunk_text": "تتوفر الجماعات الترابية والملحقات الإدارية على مصالح الحالة المدنية التي تسلم شهادات الازدياد والوفاة والزواج، وتُعتبر هذه الوثائق ضرورية لإنجاز أغلب الإجراءات الإدارية الأخرى."
    },
    {
        "id": "b257",
        "msa_query": "أين تتوفر مصالح الحالة المدنية؟",
        "darija_query": "فين كتوجد مصالح الحالة المدنية؟",
        "gold_answer": "في الجماعات الترابية والملحقات الإدارية.",
        "source_chunk_id": "chunk_072",
        "source_chunk_text": "تتوفر الجماعات الترابية والملحقات الإدارية على مصالح الحالة المدنية التي تسلم شهادات الازدياد والوفاة والزواج، وتُعتبر هذه الوثائق ضرورية لإنجاز أغلب الإجراءات الإدارية الأخرى."
    },
    {
        "id": "b258",
        "msa_query": "لماذا تُعتبر شهادات الحالة المدنية ضرورية؟",
        "darija_query": "علاش شهادات الحالة المدنية ضرورية؟",
        "gold_answer": "ضرورية لإنجاز أغلب الإجراءات الإدارية الأخرى.",
        "source_chunk_id": "chunk_072",
        "source_chunk_text": "تتوفر الجماعات الترابية والملحقات الإدارية على مصالح الحالة المدنية التي تسلم شهادات الازدياد والوفاة والزواج، وتُعتبر هذه الوثائق ضرورية لإنجاز أغلب الإجراءات الإدارية الأخرى."
    },
    {
        "id": "b259",
        "msa_query": "اذكر مثالاً عن شهادة تسلمها الحالة المدنية؟",
        "darija_query": "عطيني مثال على شهادة كتسلمها الحالة المدنية؟",
        "gold_answer": "شهادة الازدياد مثال على ذلك.",
        "source_chunk_id": "chunk_072",
        "source_chunk_text": "تتوفر الجماعات الترابية والملحقات الإدارية على مصالح الحالة المدنية التي تسلم شهادات الازدياد والوفاة والزواج، وتُعتبر هذه الوثائق ضرورية لإنجاز أغلب الإجراءات الإدارية الأخرى."
    },
    {
        "id": "b260",
        "msa_query": "هل تُصدر شهادة الزواج من مصالح الحالة المدنية؟",
        "darija_query": "واش شهادة الزواج كتصدر من الحالة المدنية؟",
        "gold_answer": "نعم، تُصدر شهادات الازدياد والوفاة والزواج.",
        "source_chunk_id": "chunk_072",
        "source_chunk_text": "تتوفر الجماعات الترابية والملحقات الإدارية على مصالح الحالة المدنية التي تسلم شهادات الازدياد والوفاة والزواج، وتُعتبر هذه الوثائق ضرورية لإنجاز أغلب الإجراءات الإدارية الأخرى."
    },
    {
        "id": "b261",
        "msa_query": "ماذا أصبح بإمكان المستهلكين فعله عبر تطبيقات الهاتف؟",
        "darija_query": "أش أصبح يقدر يدير المستهلك من التطبيق؟",
        "gold_answer": "طلب المنتجات والوجبات الجاهزة مع توصيلها للمنزل.",
        "source_chunk_id": "chunk_073",
        "source_chunk_text": "شهد قطاع التجارة الإلكترونية والتوصيل بالمغرب نمواً ملحوظاً في السنوات الأخيرة، حيث أصبح بإمكان المستهلكين طلب المنتجات والوجبات الجاهزة عبر تطبيقات الهاتف مع توصيلها مباشرة للمنزل."
    },
    {
        "id": "b262",
        "msa_query": "هل شهد قطاع التجارة الإلكترونية نمواً بالمغرب؟",
        "darija_query": "واش التجارة الإلكترونية زادات فالمغرب؟",
        "gold_answer": "نعم، شهدت نمواً ملحوظاً في السنوات الأخيرة.",
        "source_chunk_id": "chunk_073",
        "source_chunk_text": "شهد قطاع التجارة الإلكترونية والتوصيل بالمغرب نمواً ملحوظاً في السنوات الأخيرة، حيث أصبح بإمكان المستهلكين طلب المنتجات والوجبات الجاهزة عبر تطبيقات الهاتف مع توصيلها مباشرة للمنزل."
    },
    {
        "id": "b263",
        "msa_query": "كيف يتم توصيل الطلبات للمستهلكين؟",
        "darija_query": "كيفاش كيتوصل الطلب للمستهلك؟",
        "gold_answer": "يتم توصيلها مباشرة للمنزل.",
        "source_chunk_id": "chunk_073",
        "source_chunk_text": "شهد قطاع التجارة الإلكترونية والتوصيل بالمغرب نمواً ملحوظاً في السنوات الأخيرة، حيث أصبح بإمكان المستهلكين طلب المنتجات والوجبات الجاهزة عبر تطبيقات الهاتف مع توصيلها مباشرة للمنزل."
    },
    {
        "id": "b264",
        "msa_query": "عبر ماذا يمكن طلب الوجبات الجاهزة؟",
        "darija_query": "من أين ممكن نطلب الماكلة الجاهزة؟",
        "gold_answer": "عبر تطبيقات الهاتف.",
        "source_chunk_id": "chunk_073",
        "source_chunk_text": "شهد قطاع التجارة الإلكترونية والتوصيل بالمغرب نمواً ملحوظاً في السنوات الأخيرة، حيث أصبح بإمكان المستهلكين طلب المنتجات والوجبات الجاهزة عبر تطبيقات الهاتف مع توصيلها مباشرة للمنزل."
    },
    {
        "id": "b265",
        "msa_query": "ما القطاع الذي شهد نمواً ملحوظاً حسب النص؟",
        "darija_query": "شنو القطاع لي زاد بزاف حسب النص؟",
        "gold_answer": "قطاع التجارة الإلكترونية والتوصيل.",
        "source_chunk_id": "chunk_073",
        "source_chunk_text": "شهد قطاع التجارة الإلكترونية والتوصيل بالمغرب نمواً ملحوظاً في السنوات الأخيرة، حيث أصبح بإمكان المستهلكين طلب المنتجات والوجبات الجاهزة عبر تطبيقات الهاتف مع توصيلها مباشرة للمنزل."
    },
    {
        "id": "b266",
        "msa_query": "أين تبقى المساومة جزءاً أساسياً من تجربة الشراء؟",
        "darija_query": "فين الساوماة كتبقى جزء أساسي من الشرا؟",
        "gold_answer": "في الأسواق التقليدية والحوانيت الصغيرة.",
        "source_chunk_id": "chunk_074",
        "source_chunk_text": "على عكس المتاجر الكبرى ذات الأثمنة الثابتة، تبقى المساومة جزءاً أساسياً من تجربة الشراء في الأسواق التقليدية والحوانيت الصغيرة، حيث يحاول الزبون الحصول على أفضل ثمن ممكن."
    },
    {
        "id": "b267",
        "msa_query": "ماذا يحاول الزبون فعله عند المساومة؟",
        "darija_query": "أش كيحاول الزبون يدير فالساوماة؟",
        "gold_answer": "يحاول الحصول على أفضل ثمن ممكن.",
        "source_chunk_id": "chunk_074",
        "source_chunk_text": "على عكس المتاجر الكبرى ذات الأثمنة الثابتة، تبقى المساومة جزءاً أساسياً من تجربة الشراء في الأسواق التقليدية والحوانيت الصغيرة، حيث يحاول الزبون الحصول على أفضل ثمن ممكن."
    },
    {
        "id": "b268",
        "msa_query": "هل المتاجر الكبرى تعتمد أثمنة ثابتة؟",
        "darija_query": "واش المتاجر الكبار عندها أثمنة ثابتة؟",
        "gold_answer": "نعم، على عكس الأسواق التقليدية، أثمنتها ثابتة.",
        "source_chunk_id": "chunk_074",
        "source_chunk_text": "على عكس المتاجر الكبرى ذات الأثمنة الثابتة، تبقى المساومة جزءاً أساسياً من تجربة الشراء في الأسواق التقليدية والحوانيت الصغيرة، حيث يحاول الزبون الحصول على أفضل ثمن ممكن."
    },
    {
        "id": "b269",
        "msa_query": "في أي نوع من المتاجر تكون المساومة غائبة؟",
        "darija_query": "فأي نوع ديال المتاجر ماكايناش الساوماة؟",
        "gold_answer": "في المتاجر الكبرى ذات الأثمنة الثابتة.",
        "source_chunk_id": "chunk_074",
        "source_chunk_text": "على عكس المتاجر الكبرى ذات الأثمنة الثابتة، تبقى المساومة جزءاً أساسياً من تجربة الشراء في الأسواق التقليدية والحوانيت الصغيرة، حيث يحاول الزبون الحصول على أفضل ثمن ممكن."
    },
    {
        "id": "b270",
        "msa_query": "من يحاول الحصول على أفضل ثمن في السوق التقليدي؟",
        "darija_query": "شكون كيحاول ياخد أحسن ثمن فالسوق؟",
        "gold_answer": "الزبون يحاول ذلك.",
        "source_chunk_id": "chunk_074",
        "source_chunk_text": "على عكس المتاجر الكبرى ذات الأثمنة الثابتة، تبقى المساومة جزءاً أساسياً من تجربة الشراء في الأسواق التقليدية والحوانيت الصغيرة، حيث يحاول الزبون الحصول على أفضل ثمن ممكن."
    },
    {
        "id": "b271",
        "msa_query": "متى يقدم كثير من المصلين الصدقة؟",
        "darija_query": "إمتى كيعطيو المصلين الصدقة؟",
        "gold_answer": "يوم الجمعة عند باب المسجد.",
        "source_chunk_id": "chunk_075",
        "source_chunk_text": "يحرص كثير من المصلين على تقديم الصدقة يوم الجمعة عند باب المسجد، وتُعتبر هذه العادة جزءاً من التضامن الاجتماعي المرتبط بالممارسة الدينية اليومية للمغاربة."
    },
    {
        "id": "b272",
        "msa_query": "ما ارتباط عادة تقديم الصدقة؟",
        "darija_query": "بأش مرتبطة عادة الصدقة؟",
        "gold_answer": "مرتبطة بالتضامن الاجتماعي والممارسة الدينية اليومية.",
        "source_chunk_id": "chunk_075",
        "source_chunk_text": "يحرص كثير من المصلين على تقديم الصدقة يوم الجمعة عند باب المسجد، وتُعتبر هذه العادة جزءاً من التضامن الاجتماعي المرتبط بالممارسة الدينية اليومية للمغاربة."
    },
    {
        "id": "b273",
        "msa_query": "أين تُقدم الصدقة عادة يوم الجمعة؟",
        "darija_query": "فين كتعطى الصدقة نهار الجمعة؟",
        "gold_answer": "عند باب المسجد.",
        "source_chunk_id": "chunk_075",
        "source_chunk_text": "يحرص كثير من المصلين على تقديم الصدقة يوم الجمعة عند باب المسجد، وتُعتبر هذه العادة جزءاً من التضامن الاجتماعي المرتبط بالممارسة الدينية اليومية للمغاربة."
    },
    {
        "id": "b274",
        "msa_query": "هل تقديم الصدقة يوم الجمعة عادة منتشرة؟",
        "darija_query": "واش إعطاء الصدقة نهار الجمعة عادة منتشرة؟",
        "gold_answer": "نعم، يحرص عليها كثير من المصلين.",
        "source_chunk_id": "chunk_075",
        "source_chunk_text": "يحرص كثير من المصلين على تقديم الصدقة يوم الجمعة عند باب المسجد، وتُعتبر هذه العادة جزءاً من التضامن الاجتماعي المرتبط بالممارسة الدينية اليومية للمغاربة."
    },
    {
        "id": "b275",
        "msa_query": "ما نوع التضامن المرتبط بتقديم الصدقة؟",
        "darija_query": "شنو نوع التضامن المرتبط بالصدقة؟",
        "gold_answer": "التضامن الاجتماعي.",
        "source_chunk_id": "chunk_075",
        "source_chunk_text": "يحرص كثير من المصلين على تقديم الصدقة يوم الجمعة عند باب المسجد، وتُعتبر هذه العادة جزءاً من التضامن الاجتماعي المرتبط بالممارسة الدينية اليومية للمغاربة."
    },
    {
        "id": "b276",
        "msa_query": "متى تشهد الطرق إقبالاً كبيراً؟",
        "darija_query": "إمتى كتشهد الطرق إقبال كبير؟",
        "gold_answer": "في الأيام التي تسبق الأعياد الدينية.",
        "source_chunk_id": "chunk_076",
        "source_chunk_text": "تشهد الطرق ومحطات النقل بالمغرب إقبالاً كبيراً في الأيام التي تسبق الأعياد الدينية، حيث يتنقل كثير من المغاربة لزيارة عائلاتهم في مدنهم الأصلية، ما يتسبب أحياناً في اكتظاظ شديد."
    },
    {
        "id": "b277",
        "msa_query": "لماذا يتنقل كثير من المغاربة قبل الأعياد؟",
        "darija_query": "علاش المغاربة كيتنقلو قبل الأعياد؟",
        "gold_answer": "لزيارة عائلاتهم في مدنهم الأصلية.",
        "source_chunk_id": "chunk_076",
        "source_chunk_text": "تشهد الطرق ومحطات النقل بالمغرب إقبالاً كبيراً في الأيام التي تسبق الأعياد الدينية، حيث يتنقل كثير من المغاربة لزيارة عائلاتهم في مدنهم الأصلية، ما يتسبب أحياناً في اكتظاظ شديد."
    },
    {
        "id": "b278",
        "msa_query": "ماذا يتسبب فيه الإقبال الكبير على الطرق أحياناً؟",
        "darija_query": "أش كيسبب الإقبال الكبير على الطرق؟",
        "gold_answer": "يتسبب أحياناً في اكتظاظ شديد.",
        "source_chunk_id": "chunk_076",
        "source_chunk_text": "تشهد الطرق ومحطات النقل بالمغرب إقبالاً كبيراً في الأيام التي تسبق الأعياد الدينية، حيث يتنقل كثير من المغاربة لزيارة عائلاتهم في مدنهم الأصلية، ما يتسبب أحياناً في اكتظاظ شديد."
    },
    {
        "id": "b279",
        "msa_query": "إلى أين يتنقل المغاربة قبل الأعياد؟",
        "darija_query": "لفين كيتنقلو المغاربة قبل الأعياد؟",
        "gold_answer": "إلى مدنهم الأصلية لزيارة عائلاتهم.",
        "source_chunk_id": "chunk_076",
        "source_chunk_text": "تشهد الطرق ومحطات النقل بالمغرب إقبالاً كبيراً في الأيام التي تسبق الأعياد الدينية، حيث يتنقل كثير من المغاربة لزيارة عائلاتهم في مدنهم الأصلية، ما يتسبب أحياناً في اكتظاظ شديد."
    },
    {
        "id": "b280",
        "msa_query": "ما الذي يشهد إقبالاً كبيراً قبل الأعياد؟",
        "darija_query": "أش لي كيشهد إقبال كبير قبل الأعياد؟",
        "gold_answer": "الطرق ومحطات النقل.",
        "source_chunk_id": "chunk_076",
        "source_chunk_text": "تشهد الطرق ومحطات النقل بالمغرب إقبالاً كبيراً في الأيام التي تسبق الأعياد الدينية، حيث يتنقل كثير من المغاربة لزيارة عائلاتهم في مدنهم الأصلية، ما يتسبب أحياناً في اكتظاظ شديد."
    },
    {
        "id": "b281",
        "msa_query": "عبر أي جهة يتم تسجيل الأطفال في التعليم الابتدائي؟",
        "darija_query": "من أي جهة كيتسجل الطفل فالابتدائي؟",
        "gold_answer": "عبر الأكاديميات الجهوية للتربية والتكوين.",
        "source_chunk_id": "chunk_077",
        "source_chunk_text": "يتم تسجيل الأطفال في التعليم الابتدائي العمومي عبر الأكاديميات الجهوية للتربية والتكوين، ويُشترط عادة بلوغ الطفل سن السادسة قبل بداية الموسم الدراسي، مع تقديم شهادة الازدياد."
    },
    {
        "id": "b282",
        "msa_query": "ما هي السن المشترطة لتسجيل الطفل؟",
        "darija_query": "شحال السن المشروطة باش يتسجل الطفل؟",
        "gold_answer": "يُشترط بلوغ الطفل سن السادسة.",
        "source_chunk_id": "chunk_077",
        "source_chunk_text": "يتم تسجيل الأطفال في التعليم الابتدائي العمومي عبر الأكاديميات الجهوية للتربية والتكوين، ويُشترط عادة بلوغ الطفل سن السادسة قبل بداية الموسم الدراسي، مع تقديم شهادة الازدياد."
    },
    {
        "id": "b283",
        "msa_query": "ما الوثيقة المطلوبة لتسجيل الطفل في الابتدائي؟",
        "darija_query": "أش الوثيقة المطلوبة باش يتسجل الطفل؟",
        "gold_answer": "شهادة الازدياد.",
        "source_chunk_id": "chunk_077",
        "source_chunk_text": "يتم تسجيل الأطفال في التعليم الابتدائي العمومي عبر الأكاديميات الجهوية للتربية والتكوين، ويُشترط عادة بلوغ الطفل سن السادسة قبل بداية الموسم الدراسي، مع تقديم شهادة الازدياد."
    },
    {
        "id": "b284",
        "msa_query": "متى يجب بلوغ الطفل سن السادسة؟",
        "darija_query": "إمتى خاص الطفل يكون عندو ستة سنين؟",
        "gold_answer": "قبل بداية الموسم الدراسي.",
        "source_chunk_id": "chunk_077",
        "source_chunk_text": "يتم تسجيل الأطفال في التعليم الابتدائي العمومي عبر الأكاديميات الجهوية للتربية والتكوين، ويُشترط عادة بلوغ الطفل سن السادسة قبل بداية الموسم الدراسي، مع تقديم شهادة الازدياد."
    },
    {
        "id": "b285",
        "msa_query": "هل التسجيل في التعليم الابتدائي العمومي إلزامي عبر جهة محددة؟",
        "darija_query": "واش التسجيل فالابتدائي خاصو يمر من جهة معينة؟",
        "gold_answer": "نعم، يتم عبر الأكاديميات الجهوية للتربية والتكوين.",
        "source_chunk_id": "chunk_077",
        "source_chunk_text": "يتم تسجيل الأطفال في التعليم الابتدائي العمومي عبر الأكاديميات الجهوية للتربية والتكوين، ويُشترط عادة بلوغ الطفل سن السادسة قبل بداية الموسم الدراسي، مع تقديم شهادة الازدياد."
    },
    {
        "id": "b286",
        "msa_query": "هل التعليم العالي العمومي مجاني في المغرب؟",
        "darija_query": "واش التعليم العالي العمومي مجاني فالمغرب؟",
        "gold_answer": "نعم، يبقى مجانياً في أغلب الجامعات والكليات العمومية.",
        "source_chunk_id": "chunk_078",
        "source_chunk_text": "يبقى التعليم العالي العمومي في المغرب مجانياً في أغلب الجامعات والكليات، في حين تفرض المدارس والجامعات الخاصة رسوماً دراسية سنوية تختلف بشكل كبير حسب المؤسسة والتخصص."
    },
    {
        "id": "b287",
        "msa_query": "ماذا تفرض المدارس والجامعات الخاصة؟",
        "darija_query": "أش كتفرض المدارس والجامعات الخاصة؟",
        "gold_answer": "تفرض رسوماً دراسية سنوية.",
        "source_chunk_id": "chunk_078",
        "source_chunk_text": "يبقى التعليم العالي العمومي في المغرب مجانياً في أغلب الجامعات والكليات، في حين تفرض المدارس والجامعات الخاصة رسوماً دراسية سنوية تختلف بشكل كبير حسب المؤسسة والتخصص."
    },
    {
        "id": "b288",
        "msa_query": "هل تختلف رسوم التعليم الخاص حسب المؤسسة؟",
        "darija_query": "واش الرسوم ديال التعليم الخاص كتختلف حسب المؤسسة؟",
        "gold_answer": "نعم، تختلف بشكل كبير حسب المؤسسة والتخصص.",
        "source_chunk_id": "chunk_078",
        "source_chunk_text": "يبقى التعليم العالي العمومي في المغرب مجانياً في أغلب الجامعات والكليات، في حين تفرض المدارس والجامعات الخاصة رسوماً دراسية سنوية تختلف بشكل كبير حسب المؤسسة والتخصص."
    },
    {
        "id": "b289",
        "msa_query": "ما الفرق بين التعليم العالي العمومي والخاص من حيث الثمن؟",
        "darija_query": "شنو الفرق بين التعليم العمومي والخاص من ناحية الثمن؟",
        "gold_answer": "العمومي مجاني في أغلبه، بينما الخاص يفرض رسوماً دراسية.",
        "source_chunk_id": "chunk_078",
        "source_chunk_text": "يبقى التعليم العالي العمومي في المغرب مجانياً في أغلب الجامعات والكليات، في حين تفرض المدارس والجامعات الخاصة رسوماً دراسية سنوية تختلف بشكل كبير حسب المؤسسة والتخصص."
    },
    {
        "id": "b290",
        "msa_query": "على ماذا تعتمد رسوم الجامعات الخاصة؟",
        "darija_query": "على أش كتعتمد الرسوم ديال الجامعات الخاصة؟",
        "gold_answer": "تعتمد على المؤسسة والتخصص.",
        "source_chunk_id": "chunk_078",
        "source_chunk_text": "يبقى التعليم العالي العمومي في المغرب مجانياً في أغلب الجامعات والكليات، في حين تفرض المدارس والجامعات الخاصة رسوماً دراسية سنوية تختلف بشكل كبير حسب المؤسسة والتخصص."
    },
    {
        "id": "b291",
        "msa_query": "لماذا يُعتبر تقديم أتاي من أهم مظاهر الضيافة؟",
        "darija_query": "علاش تقديم أتاي من أهم مظاهر الضيافة؟",
        "gold_answer": "لأنه من أهم مظاهر الضيافة المغربية.",
        "source_chunk_id": "chunk_079",
        "source_chunk_text": "يعتبر تقديم أتاي للضيوف من أهم مظاهر الضيافة المغربية، حيث يُحرص على تحضيره بطريقة تقليدية أمام الضيف، ويُعاد تقديمه عدة مرات كعلامة على الترحاب والكرم."
    },
    {
        "id": "b292",
        "msa_query": "كيف يُحضَّر أتاي للضيوف؟",
        "darija_query": "كيفاش كيتحضر أتاي للضياف؟",
        "gold_answer": "يُحضَّر بطريقة تقليدية أمام الضيف.",
        "source_chunk_id": "chunk_079",
        "source_chunk_text": "يعتبر تقديم أتاي للضيوف من أهم مظاهر الضيافة المغربية، حيث يُحرص على تحضيره بطريقة تقليدية أمام الضيف، ويُعاد تقديمه عدة مرات كعلامة على الترحاب والكرم."
    },
    {
        "id": "b293",
        "msa_query": "كم مرة يُعاد تقديم أتاي للضيف؟",
        "darija_query": "شحال مرة كيتعاود تقديم أتاي للضيف؟",
        "gold_answer": "يُعاد تقديمه عدة مرات.",
        "source_chunk_id": "chunk_079",
        "source_chunk_text": "يعتبر تقديم أتاي للضيوف من أهم مظاهر الضيافة المغربية، حيث يُحرص على تحضيره بطريقة تقليدية أمام الضيف، ويُعاد تقديمه عدة مرات كعلامة على الترحاب والكرم."
    },
    {
        "id": "b294",
        "msa_query": "ماذا يرمز تكرار تقديم أتاي؟",
        "darija_query": "أش كيرمز تكرار أتاي؟",
        "gold_answer": "يرمز إلى الترحاب والكرم.",
        "source_chunk_id": "chunk_079",
        "source_chunk_text": "يعتبر تقديم أتاي للضيوف من أهم مظاهر الضيافة المغربية، حيث يُحرص على تحضيره بطريقة تقليدية أمام الضيف، ويُعاد تقديمه عدة مرات كعلامة على الترحاب والكرم."
    },
    {
        "id": "b295",
        "msa_query": "أين يُحضَّر أتاي عادة؟",
        "darija_query": "فين كيتحضر أتاي عادة؟",
        "gold_answer": "يُحضَّر أمام الضيف.",
        "source_chunk_id": "chunk_079",
        "source_chunk_text": "يعتبر تقديم أتاي للضيوف من أهم مظاهر الضيافة المغربية، حيث يُحرص على تحضيره بطريقة تقليدية أمام الضيف، ويُعاد تقديمه عدة مرات كعلامة على الترحاب والكرم."
    },
    {
        "id": "b296",
        "msa_query": "متى يرتدي المغاربة الجلباب أو القفطان التقليدي؟",
        "darija_query": "إمتى كيلبسو المغاربة الجلابة ولا القفطان؟",
        "gold_answer": "في المناسبات الدينية والاجتماعية الكبرى.",
        "source_chunk_id": "chunk_080",
        "source_chunk_text": "يرتدي كثير من المغاربة الجلباب أو القفطان التقليدي في المناسبات الدينية والاجتماعية الكبرى كالأعياد والأعراس، في حين يقتصر اللباس التقليدي في الحياة اليومية على فئة أقل."
    },
    {
        "id": "b297",
        "msa_query": "اذكر مثالاً عن مناسبة يُرتدى فيها اللباس التقليدي؟",
        "darija_query": "عطيني مثال على مناسبة كيتلبس فيها اللباس التقليدي؟",
        "gold_answer": "الأعياد والأعراس من الأمثلة.",
        "source_chunk_id": "chunk_080",
        "source_chunk_text": "يرتدي كثير من المغاربة الجلباب أو القفطان التقليدي في المناسبات الدينية والاجتماعية الكبرى كالأعياد والأعراس، في حين يقتصر اللباس التقليدي في الحياة اليومية على فئة أقل."
    },
    {
        "id": "b298",
        "msa_query": "هل ينتشر اللباس التقليدي في الحياة اليومية؟",
        "darija_query": "واش اللباس التقليدي منتشر فالحياة اليومية؟",
        "gold_answer": "يقتصر في الحياة اليومية على فئة أقل.",
        "source_chunk_id": "chunk_080",
        "source_chunk_text": "يرتدي كثير من المغاربة الجلباب أو القفطان التقليدي في المناسبات الدينية والاجتماعية الكبرى كالأعياد والأعراس، في حين يقتصر اللباس التقليدي في الحياة اليومية على فئة أقل."
    },
    {
        "id": "b299",
        "msa_query": "ما هما اللباسان التقليديان المذكوران في النص؟",
        "darija_query": "شنو هوما اللباسين التقليديين لي تهضر عليهم النص؟",
        "gold_answer": "الجلباب والقفطان.",
        "source_chunk_id": "chunk_080",
        "source_chunk_text": "يرتدي كثير من المغاربة الجلباب أو القفطان التقليدي في المناسبات الدينية والاجتماعية الكبرى كالأعياد والأعراس، في حين يقتصر اللباس التقليدي في الحياة اليومية على فئة أقل."
    },
    {
        "id": "b300",
        "msa_query": "في أي نوع من المناسبات يُرتدى اللباس التقليدي بكثرة؟",
        "darija_query": "فأي نوع ديال المناسبات كيتلبس اللباس التقليدي بزاف؟",
        "gold_answer": "في المناسبات الدينية والاجتماعية الكبرى.",
        "source_chunk_id": "chunk_080",
        "source_chunk_text": "يرتدي كثير من المغاربة الجلباب أو القفطان التقليدي في المناسبات الدينية والاجتماعية الكبرى كالأعياد والأعراس، في حين يقتصر اللباس التقليدي في الحياة اليومية على فئة أقل."
    }
]


### Load data into a DataFrame

In [4]:
import pandas as pd

df = pd.DataFrame(SAMPLE_DATA)
print(f"Loaded {len(df)} items.")
df.head()

Loaded 4 items.


,id,msa_query,darija_query,gold_answer,source_chunk_id,source_chunk_text
0,q1,ما هي عاصمة المغرب؟,شنو هي العاصمة ديال المغرب؟,الرباط هي عاصمة المغرب.,chunk_001,الرباط هي العاصمة الإدارية للمملكة المغربية وت...
1,q2,ما هي عاصمة المغرب؟,ما هي عاصمة المغرب؟,الرباط.,chunk_999,الرباط هي العاصمة الإدارية للمملكة المغربية.
2,q3,كم عدد سكان المغرب؟,شحال ديال الساكنة كاينة فالمغرب؟,يبلغ عدد سكان المغرب حوالي 37 مليون نسمة.,chunk_002,بلغ عدد سكان المملكة المغربية حوالي 37 مليون ن...
3,q4,متى تأسست جامعة القرويين؟,إمتى تأسست جامعة القرويين؟,تأسست جامعة القرويين سنة 1963 بعد استقلال المغرب.,chunk_003,تأسست جامعة القرويين سنة 859 ميلادية في مدينة ...


### Check 1: Schema / completeness validation (Pandera)

In [6]:
!pip install -q pandera

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.0/502.0 kB 8.8 MB/s eta 0:00:00


In [7]:
import pandera.pandas as pa
from pandera import Column, Check, DataFrameSchema

schema = DataFrameSchema({
    "id": Column(str, unique=True, nullable=False),
    "msa_query": Column(str, Check.str_length(min_value=3), nullable=False),
    "darija_query": Column(str, Check.str_length(min_value=3), nullable=False),
    "gold_answer": Column(str, Check.str_length(min_value=1), nullable=False),
    "source_chunk_id": Column(str, nullable=False),
    "source_chunk_text": Column(str, nullable=False),
})

schema_errors = []
try:
    schema.validate(df, lazy=True)
    print("✅ Schema check passed: all required fields present and well-formed.")
except pa.errors.SchemaErrors as e:
    schema_errors = e.failure_cases
    print("⚠️  Schema issues found:")
    print(schema_errors[["column", "check", "failure_case", "index"]])

# Referential integrity: does source_chunk_id exist in the known corpus?
bad_refs = df[~df["source_chunk_id"].isin(CONFIG["known_chunk_ids"])]
if len(bad_refs):
    print(f"\n⚠️  {len(bad_refs)} item(s) reference a source_chunk_id not found in the corpus:")
    print(bad_refs[["id", "source_chunk_id"]])
else:
    print("✅ All source_chunk_id references resolve to known corpus chunks.")

/usr/local/lib/python3.13/dist-packages/pandera/_pandas_deprecated.py:144: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


✅ Schema check passed: all required fields present and well-formed.

⚠️  1 item(s) reference a source_chunk_id not found in the corpus:
   id source_chunk_id
1  q2       chunk_999


### Load embedding model (used by Checks 2, 3-support, and 5)

In [8]:
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer(CONFIG["embedding_model"])

def cosine_sim(a: str, b: str) -> float:
    emb = embedder.encode([a, b], convert_to_tensor=True, normalize_embeddings=True)
    return float(util.cos_sim(emb[0], emb[1]))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Check 2: MSA/Darija pair alignment + trivial-pair detection

In [9]:
alignment_rows = []
for _, row in df.iterrows():
    sim = cosine_sim(row["msa_query"], row["darija_query"])
    flag = ""
    if sim < CONFIG["alignment_similarity_threshold"]:
        flag = "LOW_ALIGNMENT (pair may not mean the same thing)"
    elif sim > CONFIG["trivial_pair_threshold"]:
        flag = "TOO_SIMILAR (darija version may just be MSA with tiny edits)"
    alignment_rows.append({"id": row["id"], "similarity": round(sim, 3), "flag": flag})

alignment_df = pd.DataFrame(alignment_rows)
print(alignment_df)
flagged_alignment = alignment_df[alignment_df["flag"] != ""]
print(f"\n{len(flagged_alignment)} item(s) flagged for manual review (alignment).")

   id  similarity                                               flag
0  q1       0.790                                                   
1  q2       1.000  TOO_SIMILAR (darija version may just be MSA wi...
2  q3       0.621                                                   
3  q4       0.999  TOO_SIMILAR (darija version may just be MSA wi...

2 item(s) flagged for manual review (alignment).


### Check 3: Dialect authenticity (lexicon scan)

In [10]:
def darija_marker_count(text: str) -> int:
    return sum(1 for marker in CONFIG["darija_markers"] if marker in text)

df["darija_marker_hits"] = df["darija_query"].apply(darija_marker_count)
weak_dialect = df[df["darija_marker_hits"] == 0]
print("Darija marker hits per item:")
print(df[["id", "darija_query", "darija_marker_hits"]])
if len(weak_dialect):
    print(f"\n⚠️  {len(weak_dialect)} item(s) contain no recognized Darija markers — "
          f"review manually, may just be lightly-edited MSA:")
    print(weak_dialect[["id", "darija_query"]])

Darija marker hits per item:
   id                      darija_query  darija_marker_hits
0  q1       شنو هي العاصمة ديال المغرب؟                   2
1  q2               ما هي عاصمة المغرب؟                   0
2  q3  شحال ديال الساكنة كاينة فالمغرب؟                   2
3  q4        إمتى تأسست جامعة القرويين؟                   0

⚠️  2 item(s) contain no recognized Darija markers — review manually, may just be lightly-edited MSA:
   id                darija_query
1  q2         ما هي عاصمة المغرب؟
3  q4  إمتى تأسست جامعة القرويين؟


### Check 4: Answer-grounding via LLM-as-judge

In [ ]:
# NOTE: This cell calls the Groq API (free tier) as the LLM-as-judge for
# answer-grounding. Get a free API key at https://console.groq.com/keys
#
# In Colab, store it securely instead of hardcoding it:
#   from google.colab import userdata
#   GROQ_API_KEY = userdata.get('GROQ_API_KEY')
# (Add it once via the key icon in the left sidebar -> Secrets)

# !pip install -q groq

import os
from groq import Groq

# Option A (Colab secrets, recommended):
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    # Option B: fall back to an environment variable or paste directly (not recommended)
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

client = Groq(api_key=GROQ_API_KEY)

def call_llm_judge(answer: str, source_text: str, model: str = "llama-3.3-70b-versatile") -> str:
    """
    Asks Groq's LLM whether `answer` is fully supported by `source_text`.
    Returns "YES" or "NO". Falls back to an embedding-similarity heuristic
    if no API key is set, so the notebook still runs end-to-end.
    """
    if not GROQ_API_KEY:
        sim = cosine_sim(answer, source_text)
        return "YES" if sim > 0.5 else "NO"

    prompt = (
        f'Source text: "{source_text}"\n'
        f'Claim: "{answer}"\n\n'
        'Question: Is the claim fully and accurately supported by the source text?\n'
        'Consider factual details like dates, numbers, and names carefully.\n'
        'Think briefly, then answer with exactly one word: YES or NO.'
    )

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=400,
        )
        verdict = response.choices[0].message.content.strip().upper()
        return "YES" if "YES" in verdict else "NO"
    except Exception as e:
        print(f"  (Groq API call failed for grounding check: {e} — falling back to embedding similarity)")
        sim = cosine_sim(answer, source_text)
        return "YES" if sim > 0.5 else "NO"

grounding_rows = []
for _, row in df.iterrows():
    verdict = call_llm_judge(row["gold_answer"], row["source_chunk_text"])
    grounding_rows.append({"id": row["id"], "grounded": verdict})

grounding_df = pd.DataFrame(grounding_rows)
print(grounding_df)
ungrounded = grounding_df[grounding_df["grounded"] == "NO"]
if len(ungrounded):
    print(f"\n⚠️  {len(ungrounded)} item(s) flagged as possibly ungrounded/hallucinated answers:")
    print(ungrounded)


### Check 5: Duplicate / near-duplicate detection

In [12]:
import itertools

dup_rows = []
ids = df["id"].tolist()
queries = df["msa_query"].tolist()
for (i1, q1), (i2, q2) in itertools.combinations(zip(ids, queries), 2):
    sim = cosine_sim(q1, q2)
    if sim > CONFIG["near_duplicate_threshold"]:
        dup_rows.append({"id_1": i1, "id_2": i2, "similarity": round(sim, 3)})

if dup_rows:
    dup_df = pd.DataFrame(dup_rows)
    print(f"⚠️  {len(dup_df)} near-duplicate pair(s) found:")
    print(dup_df)
else:
    print("✅ No near-duplicate MSA queries found.")

⚠️  1 near-duplicate pair(s) found:
  id_1 id_2  similarity
0   q1   q2         1.0


### Check 6: Coverage check (source chunk usage distribution)

In [13]:
coverage = df["source_chunk_id"].value_counts()
print("Source chunk usage distribution:")
print(coverage)
if coverage.max() > max(3, len(df) * 0.3):
    print(f"\n⚠️  Chunk '{coverage.idxmax()}' is used by {coverage.max()} items — "
          f"benchmark may be over-concentrated on this passage.")
else:
    print("✅ Source chunk usage looks reasonably spread out.")

Source chunk usage distribution:
source_chunk_id
chunk_001    1
chunk_999    1
chunk_002    1
chunk_003    1
Name: count, dtype: int64
✅ Source chunk usage looks reasonably spread out.


### Final report: consolidated list of flagged items for human review

In [14]:
review_ids = set()
review_ids.update(schema_errors["index"].tolist() if len(schema_errors) else [])
review_ids.update(bad_refs["id"].tolist())
review_ids.update(flagged_alignment["id"].tolist())
review_ids.update(weak_dialect["id"].tolist())
review_ids.update(ungrounded["id"].tolist())
for r in dup_rows:
    review_ids.update([r["id_1"], r["id_2"]])

print("=" * 60)
print(f"VALIDATION SUMMARY — {len(df)} items checked")
print("=" * 60)
print(f"Items flagged for manual review by the native Darija speaker: {len(review_ids)}")
print(sorted(review_ids))
print("\nAll other items passed automated checks and can skip manual review,")
print("unless included in your random spot-check sample.")

VALIDATION SUMMARY — 4 items checked
Items flagged for manual review by the native Darija speaker: 3
['q1', 'q2', 'q4']

All other items passed automated checks and can skip manual review,
unless included in your random spot-check sample.
